# Convert CSVs format to DBSQL

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd

# Define portable project paths for GitHub
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_BASE_DIR = PROJECT_ROOT / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)

# Define the folder path that contains the CSV files
folder_path = DATA_DIR

# Define input CSV files
csv_files = {
    "campus_load": folder_path / "campus_load.csv"
}

try:
    # Convert each CSV file into a separate SQLite database file
    for table_name, csv_path in csv_files.items():

        # Define the output database path
        db_path = folder_path / f"{table_name}.db"

        # Read the CSV file into a pandas DataFrame
        df = pd.read_csv(csv_path)

        # Clean column names by removing leading and trailing spaces
        df.columns = df.columns.str.strip()

        # Create a connection to the SQLite database
        conn = sqlite3.connect(db_path)

        try:
            # Save the DataFrame as a table inside the SQLite database
            df.to_sql(table_name, conn, if_exists="replace", index=False)

            print(f"Database created successfully: {db_path}")

        finally:
            # Close the database connection
            conn.close()

except FileNotFoundError as e:
    print("One of the CSV files was not found.")
    print(e)

except Exception as e:
    print("An error occurred while creating the database files.")
    print(e)

# CLEAN_THE_DATA_SET

In [ ]:
import sqlite3
from pathlib import Path

# Define portable project paths for GitHub
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_BASE_DIR = PROJECT_ROOT / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)

# Define the folder path that contains the database files
folder_path = DATA_DIR

# Define database files and their original table names
databases = {
    "campus_load.db": "campus_load"
}

# Define the name of the timestamp column
timestamp_column = "data_timestamp"

# Define the name of the new modified table
modified_table_name = "MODIFIED_DATASET"

for db_file_name, original_table_name in databases.items():
    # Define the full database path
    db_path = folder_path / db_file_name

    # Connect to the SQLite database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        # Drop the modified table if it already exists
        cursor.execute(f'DROP TABLE IF EXISTS "{modified_table_name}"')

        # Create a new modified table with all original columns
        # The timestamp is split as plain text, so no value is rounded or changed
        query = f"""
        CREATE TABLE "{modified_table_name}" AS
        SELECT
            *,
            CASE
                WHEN instr("{timestamp_column}", ' ') > 0
                THEN substr("{timestamp_column}", 1, instr("{timestamp_column}", ' ') - 1)
                ELSE "{timestamp_column}"
            END AS DATE_OF_THE_YEAR,

            CASE
                WHEN instr("{timestamp_column}", ' ') > 0
                THEN substr("{timestamp_column}", instr("{timestamp_column}", ' ') + 1)
                ELSE NULL
            END AS HOURLY_TIME

        FROM "{original_table_name}";
        """

        # Execute the query
        cursor.execute(query)

        # Save changes
        conn.commit()

        print(f"Modified table created successfully in: {db_path}")
        print(f"Table name: {modified_table_name}")

    except Exception as e:
        print(f"An error occurred while processing: {db_path}")
        print(e)

    finally:
        # Close the database connection
        conn.close()

# Pre_Processing

In [ ]:
"""
Q1-Level EDA & Preprocessing for Load Datasets
================================================
Reads MODIFIED_DATASET from SQLite .db files and performs:

  1. Full data audit
  2. Robust timestamp parsing for mixed timestamp formats
  3. Descriptive statistics
  4. Distribution analysis
  5. Outlier detection
  6. Time-gap analysis
  7. Missing interval estimation
  8. New quality-control columns written back to SQLite:
       - time_delta_seconds
       - time_delta_minutes
       - is_irregular_interval
       - is_gap
       - timestamp_parse_ok
       - is_duplicate_timestamp
       - is_negative_value
       - is_iqr_outlier
       - is_mad_outlier
       - is_bad_sample
  9. Publication-quality plots and CSV reports

Important fixes compared with the previous version:
  - Mixed timestamp formats are handled correctly.
  - Duplicate timestamps are counted only among valid parsed timestamps.
  - NaT values are not treated as duplicate timestamps.
  - Time deltas are computed only between valid timestamps.
  - SQLite receives Python None instead of pandas NaN for missing values.
  - Outlier labels in plots use full-dataset counts, not sampled counts.

All comments are in English.
Raw values are not rounded or modified.
"""

from __future__ import annotations

import os
import sqlite3
import warnings
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)


# ── Optional heavy imports ────────────────────────────────────────────────────
try:
    from scipy import stats as sp_stats
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

try:
    from statsmodels.graphics.tsaplots import plot_acf
    HAS_SM = True
except ImportError:
    HAS_SM = False


# Portable project paths for GitHub
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_BASE_DIR = PROJECT_ROOT / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)

# ═════════════════════════════════════════════════════════════════════════════
# USER CONFIGURATION — edit only this block
# ═════════════════════════════════════════════════════════════════════════════

DB_PATHS = [
    DATA_DIR / "campus_load.db",
]

TABLE_NAME     = "MODIFIED_DATASET"
TIMESTAMP_COL  = "data_timestamp"
VALUE_COL      = "value"

OUTPUT_DIR     = OUTPUT_BASE_DIR / "EDA_OUTPUTS"

# Main expected gap definition:
# is_gap = 1 when delta > expected_step * GAP_FACTOR
GAP_FACTOR = 3.0

# More sensitive interval irregularity definition:
# is_irregular_interval = 1 when delta > expected_step * IRREGULAR_FACTOR
# This detects smaller missing intervals too.
IRREGULAR_FACTOR = 1.5

# Chunk size for reading large tables
CHUNK_SIZE = 300_000

# ACF lags
ACF_MAX_LAGS = 1500

# Robust z-score threshold
ROBUST_Z_THRESH = 3.5

# Maximum number of examples saved for bad timestamp / gap reports
MAX_EXAMPLE_ROWS = 20_000

# ═════════════════════════════════════════════════════════════════════════════


# ── Plot aesthetics ───────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", context="talk", font_scale=1.05)

plt.rcParams.update(
    {
        "figure.dpi": 130,
        "savefig.dpi": 200,
        "axes.spines.right": False,
        "axes.spines.top": False,
        "figure.constrained_layout.use": True,
    }
)

PALETTE = {
    "main": "#2196F3",
    "outlier": "#E53935",
    "gap": "#FF6F00",
    "good": "#43A047",
    "bar": "#5C6BC0",
    "neutral": "#78909C",
}


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — General helpers
# ─────────────────────────────────────────────────────────────────────────────

def qident(identifier: str) -> str:
    """Safely quote a SQLite identifier such as a table or column name."""
    return '"' + identifier.replace('"', '""') + '"'


def tag(db_path: str) -> str:
    """Return database file name without extension."""
    return Path(db_path).stem


def save_fig(path: Path) -> None:
    """Save the current matplotlib figure and close all open figures."""
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, bbox_inches="tight")
    plt.close("all")


def sqlite_safe_value(value):
    """Convert pandas / numpy missing values to SQLite-safe Python values."""
    if pd.isna(value):
        return None

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    if isinstance(value, np.bool_):
        return int(value)

    return value


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — SQLite I/O helpers
# ─────────────────────────────────────────────────────────────────────────────

def read_table_chunked(conn: sqlite3.Connection, table: str) -> pd.DataFrame:
    """
    Read a large SQLite table in chunks.
    Returns a concatenated DataFrame with an extra '_rowid_' column.
    """
    table_q = qident(table)

    total_rows = pd.read_sql_query(
        f"SELECT COUNT(*) AS n FROM {table_q};",
        conn
    ).iloc[0, 0]

    print(f"    Total rows: {total_rows:,} — reading in chunks of {CHUNK_SIZE:,}")

    chunks = []
    offset = 0

    while offset < total_rows:
        chunk = pd.read_sql_query(
            f"""
            SELECT rowid AS _rowid_, *
            FROM {table_q}
            ORDER BY rowid
            LIMIT {CHUNK_SIZE}
            OFFSET {offset};
            """,
            conn,
        )
        chunks.append(chunk)

        offset += CHUNK_SIZE
        print(f"    Read {min(offset, total_rows):,} / {total_rows:,} rows", end="\r")

    print()

    if not chunks:
        return pd.DataFrame()

    return pd.concat(chunks, ignore_index=True)


def ensure_columns(conn: sqlite3.Connection, table: str, cols: Dict[str, str]) -> None:
    """Add new columns to SQLite table if they do not already exist."""
    table_q = qident(table)

    existing = pd.read_sql_query(
        f"PRAGMA table_info({table_q});",
        conn
    )["name"].tolist()

    cur = conn.cursor()

    for col, coltype in cols.items():
        if col not in existing:
            cur.execute(
                f"ALTER TABLE {table_q} ADD COLUMN {qident(col)} {coltype};"
            )

    conn.commit()


def batch_update_rowid(
    conn: sqlite3.Connection,
    table: str,
    df: pd.DataFrame,
    col_map: Dict[str, str],
    batch: int = 50_000,
) -> None:
    """
    Write computed columns back to SQLite using rowid.

    col_map format:
        {
            "sqlite_column_name": "dataframe_column_name"
        }
    """
    table_q = qident(table)

    set_clause = ", ".join(f"{qident(sqlite_col)} = ?" for sqlite_col in col_map)
    sql = f"UPDATE {table_q} SET {set_clause} WHERE rowid = ?;"

    sqlite_cols = list(col_map.keys())
    df_cols = list(col_map.values())

    cur = conn.cursor()
    total = len(df)

    for start in range(0, total, batch):
        end = min(start + batch, total)
        sub = df.iloc[start:end]

        data = []

        for row in sub[df_cols + ["_rowid_"]].itertuples(index=False, name=None):
            values = [sqlite_safe_value(v) for v in row[:-1]]
            rowid = int(row[-1])
            data.append(tuple(values + [rowid]))

        cur.executemany(sql, data)
        conn.commit()

        print(f"    Written {end:,} / {total:,} rows to SQLite", end="\r")

    print()


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — Statistical helpers
# ─────────────────────────────────────────────────────────────────────────────

def parse_timestamps(ts_series: pd.Series) -> pd.Series:
    """
    Robustly parse timestamp strings without modifying the original values.

    Handles:
        YYYY-MM-DD HH:MM:SS
        YYYY-MM-DD HH:MM:SS.sss
        timezone-aware timestamps such as:
            YYYY-MM-DD HH:MM:SS+01:00
            YYYY-MM-DD HH:MM:SSZ

    Important:
        All parsed timestamps are normalized to timezone-naive UTC.
        This avoids the pandas error:
        "can't compare offset-naive and offset-aware datetimes"
    """
    s = ts_series.astype("string").str.strip()

    parsed = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")

    valid_text_mask = s.notna() & s.ne("")

    if not valid_text_mask.any():
        return parsed

    # First attempt: pandas >= 2.0 supports format="mixed".
    try:
        tmp = pd.to_datetime(
            s.loc[valid_text_mask],
            errors="coerce",
            format="mixed",
            utc=True,
            cache=True,
        )

    except TypeError:
        # Older pandas versions do not support format="mixed".
        tmp = pd.to_datetime(
            s.loc[valid_text_mask],
            errors="coerce",
            utc=True,
            cache=True,
        )

    # Convert timezone-aware UTC timestamps to timezone-naive datetime64[ns].
    # This makes all timestamps comparable and sortable.
    if hasattr(tmp, "dt"):
        tmp = tmp.dt.tz_convert(None)
    else:
        tmp = tmp.tz_convert(None)

    parsed.loc[valid_text_mask] = tmp

    return parsed


def mode_step(deltas: pd.Series) -> float:
    """
    Estimate the dominant sampling step in seconds.
    Positive deltas are rounded to 10 ms buckets to handle small jitter.
    """
    d = deltas.dropna()
    d = d[d > 0]

    if d.empty:
        return np.nan

    buckets = (d * 100).round().astype("Int64")
    mode_val = buckets.mode()

    if mode_val.empty:
        return float(d.median())

    return float(mode_val.iloc[0]) / 100.0


def iqr_bounds(x: pd.Series) -> Tuple[float, float, float, float]:
    """Return Q1, Q3, lower fence, and upper fence using 1.5 × IQR."""
    q1 = float(x.quantile(0.25))
    q3 = float(x.quantile(0.75))
    iqr = q3 - q1

    return q1, q3, q1 - 1.5 * iqr, q3 + 1.5 * iqr


def mad_zscore(x: pd.Series) -> pd.Series:
    """
    Compute MAD-based robust z-score.
    Returns a Series aligned to x.
    """
    med = x.median()
    mad = (x - med).abs().median()

    if pd.isna(mad) or mad == 0:
        return pd.Series(np.zeros(len(x)), index=x.index)

    return 0.6745 * (x - med) / mad


def normality_test(x: pd.Series, sample_size: int = 5000) -> Tuple[str, Optional[float]]:
    """
    Run D'Agostino-Pearson normality test on a sample.
    This is only a diagnostic, not a modeling decision.
    """
    xd = x.dropna()

    if not HAS_SCIPY or xd.size < 20:
        return "not_run", None

    if xd.size > sample_size:
        xd = xd.sample(sample_size, random_state=42)

    try:
        _, p_value = sp_stats.normaltest(xd.values)
        return "D'Agostino-Pearson", float(p_value)

    except Exception:
        return "error", None


def describe_extended(x: pd.Series) -> Dict:
    """Comprehensive descriptive statistics for a numeric Series."""
    xd = x.dropna()

    if xd.empty:
        return {}

    test_name, p_value = normality_test(x)

    return {
        "count": int(xd.size),
        "null_count": int(x.isna().sum()),
        "mean": float(xd.mean()),
        "std": float(xd.std()),
        "min": float(xd.min()),
        "p01": float(xd.quantile(0.01)),
        "p05": float(xd.quantile(0.05)),
        "p25": float(xd.quantile(0.25)),
        "median": float(xd.median()),
        "p75": float(xd.quantile(0.75)),
        "p95": float(xd.quantile(0.95)),
        "p99": float(xd.quantile(0.99)),
        "max": float(xd.max()),
        "range": float(xd.max() - xd.min()),
        "skewness": float(xd.skew()),
        "kurtosis": float(xd.kurtosis()),
        "normality_test": test_name,
        "normality_pvalue": p_value,
        "cv_percent": float(xd.std() / xd.mean() * 100) if xd.mean() != 0 else np.nan,
    }


def estimate_missing_intervals(deltas: pd.Series, expected_step: float) -> int:
    """
    Estimate missing intervals from time deltas.

    Example:
        expected_step = 60 seconds
        delta = 300 seconds
        estimated missing intervals = 300 / 60 - 1 = 4
    """
    if pd.isna(expected_step) or expected_step <= 0:
        return 0

    d = deltas.dropna()
    d = d[d > expected_step * IRREGULAR_FACTOR]

    if d.empty:
        return 0

    estimated = np.maximum(np.round(d / expected_step) - 1, 0)

    return int(estimated.sum())


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — Plotting helpers
# ─────────────────────────────────────────────────────────────────────────────

def _downsample(df: pd.DataFrame, max_pts: int, seed: int = 42) -> pd.DataFrame:
    """Downsample for plotting only. The raw data is not modified."""
    if len(df) > max_pts:
        return df.sample(max_pts, random_state=seed).sort_values("_ts_parsed")

    return df


def _valid_time_value(df: pd.DataFrame) -> pd.DataFrame:
    """Return rows with valid timestamp and valid numeric value."""
    return df.dropna(subset=["_ts_parsed", VALUE_COL]).copy()


def _profile_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return data suitable for temporal profile plots.
    Negative values and null values are removed from these profile summaries.
    """
    tmp = _valid_time_value(df)

    if "is_negative_value" in tmp.columns:
        tmp = tmp[tmp["is_negative_value"] == 0]

    return tmp


def plot_value_over_time(df: pd.DataFrame, tag_name: str, out: Path) -> None:
    plot_df = _downsample(_valid_time_value(df), 250_000)

    if plot_df.empty:
        return

    fig, ax = plt.subplots(figsize=(16, 5))
    ax.plot(
        plot_df["_ts_parsed"],
        plot_df[VALUE_COL],
        linewidth=0.6,
        alpha=0.85,
        color=PALETTE["main"],
    )

    ax.set_title(f"{tag_name} | Load value over time", fontweight="bold")
    ax.set_xlabel("Time")
    ax.set_ylabel(VALUE_COL)

    save_fig(out / "01_value_over_time.png")


def plot_scatter_time(df: pd.DataFrame, tag_name: str, out: Path) -> None:
    plot_df = _downsample(_valid_time_value(df), 150_000)

    if plot_df.empty:
        return

    fig, ax = plt.subplots(figsize=(16, 5))
    ax.scatter(
        plot_df["_ts_parsed"],
        plot_df[VALUE_COL],
        s=4,
        alpha=0.3,
        color=PALETTE["main"],
        edgecolors="none",
    )

    ax.set_title(f"{tag_name} | Scatter: value vs time", fontweight="bold")
    ax.set_xlabel("Time")
    ax.set_ylabel(VALUE_COL)

    save_fig(out / "02_scatter_value_time.png")


def plot_hist_kde(x: pd.Series, tag_name: str, out: Path) -> None:
    xd = x.dropna()

    if xd.empty:
        return

    if len(xd) > 800_000:
        xd = xd.sample(800_000, random_state=42)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    sns.histplot(
        xd,
        bins=150,
        kde=True,
        stat="density",
        color=PALETTE["main"],
        ax=axes[0],
    )

    axes[0].set_title("Distribution — histogram + KDE")
    axes[0].set_xlabel(VALUE_COL)

    sorted_x = np.sort(xd.values)
    ecdf = np.arange(1, len(sorted_x) + 1) / len(sorted_x)

    axes[1].plot(sorted_x, ecdf, color=PALETTE["bar"], linewidth=1.2)
    axes[1].set_title("Empirical CDF")
    axes[1].set_xlabel(VALUE_COL)
    axes[1].set_ylabel("Cumulative probability")

    fig.suptitle(f"{tag_name} | Value distribution", fontweight="bold")

    save_fig(out / "03_hist_kde_ecdf.png")


def plot_box_violin(x: pd.Series, tag_name: str, out: Path) -> None:
    xd = x.dropna()

    if xd.empty:
        return

    if len(xd) > 300_000:
        xd = xd.sample(300_000, random_state=42)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.boxplot(
        x=xd,
        ax=axes[0],
        color=PALETTE["main"],
        flierprops=dict(marker=".", alpha=0.3, markersize=3),
    )

    axes[0].set_title("Boxplot")
    axes[0].set_xlabel(VALUE_COL)

    sns.violinplot(
        x=xd,
        ax=axes[1],
        color=PALETTE["good"],
        inner="quartile",
        cut=0,
    )

    axes[1].set_title("Violin plot")
    axes[1].set_xlabel(VALUE_COL)

    fig.suptitle(f"{tag_name} | Box & Violin", fontweight="bold")

    save_fig(out / "04_box_violin.png")


def plot_outliers(
    df: pd.DataFrame,
    tag_name: str,
    out: Path,
    low: float,
    high: float,
    total_outliers: int,
) -> None:
    plot_df = _downsample(_valid_time_value(df), 200_000)

    if plot_df.empty:
        return

    plot_df["outlier"] = (
        (plot_df[VALUE_COL] < low) |
        (plot_df[VALUE_COL] > high)
    )

    normal = plot_df[~plot_df["outlier"]]
    abnorm = plot_df[plot_df["outlier"]]

    fig, ax = plt.subplots(figsize=(16, 5))

    ax.scatter(
        normal["_ts_parsed"],
        normal[VALUE_COL],
        s=4,
        alpha=0.25,
        color=PALETTE["main"],
        edgecolors="none",
        label="Normal",
    )

    ax.scatter(
        abnorm["_ts_parsed"],
        abnorm[VALUE_COL],
        s=12,
        alpha=0.7,
        color=PALETTE["outlier"],
        edgecolors="none",
        label=f"IQR outlier in full data (n={total_outliers:,})",
    )

    ax.axhline(
        low,
        color=PALETTE["outlier"],
        linestyle="--",
        linewidth=1,
        alpha=0.8,
        label=f"Lower fence = {low:.4f}",
    )

    ax.axhline(
        high,
        color=PALETTE["outlier"],
        linestyle="--",
        linewidth=1,
        alpha=0.8,
        label=f"Upper fence = {high:.4f}",
    )

    ax.legend(loc="upper right", fontsize=10)
    ax.set_title(f"{tag_name} | IQR outliers highlighted", fontweight="bold")
    ax.set_xlabel("Time")
    ax.set_ylabel(VALUE_COL)

    save_fig(out / "05_outliers_iqr.png")


def plot_mad_outliers(df: pd.DataFrame, tag_name: str, out: Path) -> None:
    plot_df = _downsample(_valid_time_value(df), 200_000)

    if plot_df.empty:
        return

    if "robust_zscore_abs" not in plot_df.columns:
        plot_df["robust_zscore_abs"] = mad_zscore(plot_df[VALUE_COL]).abs()

    fig, ax = plt.subplots(figsize=(16, 5))

    sc = ax.scatter(
        plot_df["_ts_parsed"],
        plot_df[VALUE_COL],
        c=plot_df["robust_zscore_abs"],
        cmap="RdYlBu_r",
        vmin=0,
        vmax=ROBUST_Z_THRESH * 1.5,
        s=4,
        alpha=0.4,
        edgecolors="none",
    )

    plt.colorbar(sc, ax=ax, label="|Robust z-score|")

    ax.set_title(
        f"{tag_name} | MAD robust z-score (threshold = {ROBUST_Z_THRESH})",
        fontweight="bold",
    )

    ax.set_xlabel("Time")
    ax.set_ylabel(VALUE_COL)

    save_fig(out / "06_mad_robust_zscore.png")


def plot_delta_distribution(
    deltas: pd.Series,
    tag_name: str,
    out: Path,
    exp_step: float,
) -> None:
    d = deltas.dropna()
    d = d[d > 0]

    if d.empty:
        return

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    axes[0].hist(
        d,
        bins=200,
        color=PALETTE["bar"],
        edgecolor="none",
        alpha=0.85,
    )

    axes[0].set_yscale("log")

    if pd.notna(exp_step):
        axes[0].axvline(
            exp_step,
            color=PALETTE["outlier"],
            linestyle="--",
            linewidth=1.5,
            label=f"Mode step = {exp_step:.2f}s",
        )
        axes[0].legend()

    axes[0].set_title("Time-delta distribution (log-y)")
    axes[0].set_xlabel("delta (seconds)")
    axes[0].set_ylabel("Count (log)")

    rounded = (d * 100).round().astype("Int64")
    top15 = rounded.value_counts().head(15)

    labels = [f"{v / 100:.2f}s" for v in top15.index.astype(float)]

    axes[1].barh(
        labels[::-1],
        top15.values[::-1],
        color=PALETTE["bar"],
        edgecolor="none",
    )

    axes[1].set_title("Top-15 time-delta values")
    axes[1].set_xlabel("Count")
    axes[1].set_ylabel("delta (seconds)")

    fig.suptitle(f"{tag_name} | Sampling interval analysis", fontweight="bold")

    save_fig(out / "07_time_delta_analysis.png")


def plot_missing_bar(null_report: Dict, tag_name: str, out: Path) -> None:
    keys = list(null_report.keys())
    values = list(null_report.values())

    fig, ax = plt.subplots(figsize=(12, 5))

    bars = ax.bar(
        keys,
        values,
        color=PALETTE["outlier"],
        edgecolor="none",
        alpha=0.85,
    )

    ax.bar_label(bars, fmt="%d", padding=3, fontsize=11)
    ax.set_title(f"{tag_name} | Missing / invalid value counts", fontweight="bold")
    ax.set_ylabel("Count")
    ax.set_xlabel("")

    plt.xticks(rotation=15, ha="right")

    save_fig(out / "08_missing_counts.png")


def plot_gap_scatter(df: pd.DataFrame, exp_step: float, tag_name: str, out: Path) -> None:
    if pd.isna(exp_step):
        return

    gap_threshold = exp_step * GAP_FACTOR

    gaps = df[(df["is_gap"] == 1) & df["_ts_parsed"].notna()].copy()

    if gaps.empty:
        return

    plot_gaps = _downsample(gaps, 60_000)

    fig, ax = plt.subplots(figsize=(16, 5))

    ax.scatter(
        plot_gaps["_ts_parsed"],
        plot_gaps["time_delta_seconds"],
        s=10,
        alpha=0.6,
        color=PALETTE["gap"],
        edgecolors="none",
    )

    ax.axhline(
        gap_threshold,
        linestyle="--",
        linewidth=1.5,
        color="black",
        alpha=0.7,
        label=f"Gap threshold = {gap_threshold:.1f}s",
    )

    ax.set_yscale("log")
    ax.set_title(
        f"{tag_name} | Gap locations (delta > {GAP_FACTOR}× expected step)",
        fontweight="bold",
    )

    ax.set_xlabel("Time")
    ax.set_ylabel("time_delta_seconds (log)")
    ax.legend()

    save_fig(out / "09_gap_locations.png")


def plot_min_max(df: pd.DataFrame, tag_name: str, out: Path) -> None:
    valid_df = _valid_time_value(df)

    if valid_df.empty:
        return

    plot_df = _downsample(valid_df, 250_000)

    idx_min = valid_df[VALUE_COL].idxmin()
    idx_max = valid_df[VALUE_COL].idxmax()

    pts = valid_df.loc[[idx_min, idx_max], ["_ts_parsed", VALUE_COL]].dropna()

    fig, ax = plt.subplots(figsize=(16, 5))

    ax.plot(
        plot_df["_ts_parsed"],
        plot_df[VALUE_COL],
        linewidth=0.6,
        alpha=0.8,
        color=PALETTE["main"],
    )

    ax.scatter(
        pts["_ts_parsed"],
        pts[VALUE_COL],
        s=100,
        zorder=5,
        color=PALETTE["outlier"],
        label="Min / Max",
    )

    for _, r in pts.iterrows():
        ax.annotate(
            f"{r[VALUE_COL]:.4f}",
            xy=(r["_ts_parsed"], r[VALUE_COL]),
            xytext=(10, 6),
            textcoords="offset points",
            fontsize=11,
            color=PALETTE["outlier"],
        )

    ax.legend()
    ax.set_title(f"{tag_name} | Global minimum and maximum highlighted", fontweight="bold")
    ax.set_xlabel("Time")
    ax.set_ylabel(VALUE_COL)

    save_fig(out / "10_min_max_highlight.png")


def plot_monthly_profile(df: pd.DataFrame, tag_name: str, out: Path) -> None:
    tmp = _profile_df(df)

    if tmp.empty:
        return

    tmp["month"] = tmp["_ts_parsed"].dt.month

    monthly = tmp.groupby("month")[VALUE_COL].agg(["mean", "std"]).reset_index()

    month_labels = [
        "Jan", "Feb", "Mar", "Apr", "May", "Jun",
        "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
    ]

    fig, ax = plt.subplots(figsize=(12, 5))

    ax.bar(
        monthly["month"],
        monthly["mean"],
        yerr=monthly["std"],
        capsize=4,
        color=PALETTE["bar"],
        edgecolor="none",
        alpha=0.85,
        error_kw={"elinewidth": 1.2, "ecolor": PALETTE["neutral"]},
    )

    ax.set_xticks(monthly["month"])
    ax.set_xticklabels([month_labels[m - 1] for m in monthly["month"]])

    ax.set_title(f"{tag_name} | Monthly load profile (mean ± std)", fontweight="bold")
    ax.set_xlabel("Month")
    ax.set_ylabel(f"Mean {VALUE_COL}")

    save_fig(out / "11_monthly_profile.png")


def plot_hourly_profile(df: pd.DataFrame, tag_name: str, out: Path) -> None:
    tmp = _profile_df(df)

    if tmp.empty:
        return

    tmp["hour"] = tmp["_ts_parsed"].dt.hour

    hourly = tmp.groupby("hour")[VALUE_COL].agg(["mean", "std"]).reset_index()

    fig, ax = plt.subplots(figsize=(13, 5))

    ax.fill_between(
        hourly["hour"],
        hourly["mean"] - hourly["std"],
        hourly["mean"] + hourly["std"],
        alpha=0.25,
        color=PALETTE["main"],
        label="±1 std",
    )

    ax.plot(
        hourly["hour"],
        hourly["mean"],
        color=PALETTE["main"],
        linewidth=2.5,
        marker="o",
        markersize=5,
        label="Mean",
    )

    ax.set_xticks(range(0, 24))
    ax.set_title(f"{tag_name} | Hourly load profile (mean ± std)", fontweight="bold")
    ax.set_xlabel("Hour of day")
    ax.set_ylabel(f"Mean {VALUE_COL}")
    ax.legend()

    save_fig(out / "12_hourly_profile.png")


def plot_heatmap_hour_dow(df: pd.DataFrame, tag_name: str, out: Path) -> None:
    tmp = _profile_df(df)

    if tmp.empty:
        return

    tmp["hour"] = tmp["_ts_parsed"].dt.hour
    tmp["dow"] = tmp["_ts_parsed"].dt.dayofweek

    pivot = tmp.pivot_table(
        values=VALUE_COL,
        index="hour",
        columns="dow",
        aggfunc="mean",
    )

    pivot = pivot.reindex(columns=range(7))
    pivot.columns = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

    fig, ax = plt.subplots(figsize=(12, 9))

    sns.heatmap(
        pivot,
        cmap="YlOrRd",
        ax=ax,
        cbar_kws={"label": f"Mean {VALUE_COL}"},
        linewidths=0.3,
        linecolor="white",
    )

    ax.set_title(f"{tag_name} | Mean load — hour vs day of week", fontweight="bold")
    ax.set_xlabel("Day of week")
    ax.set_ylabel("Hour of day")

    save_fig(out / "13_heatmap_hour_dow.png")


def plot_acf_proper(df: pd.DataFrame, exp_step: float, tag_name: str, out: Path) -> None:
    if not HAS_SM:
        return

    tmp = _profile_df(df).sort_values("_ts_parsed")

    if tmp.empty:
        return

    x = tmp[VALUE_COL].dropna()

    if len(x) < 10:
        return

    # Use a contiguous slice, not a random sample.
    if len(x) > 50_000:
        x = x.iloc[:50_000]

    if pd.notna(exp_step) and exp_step > 0:
        one_day_lags = int(round(86400 / exp_step))
        lags = min(one_day_lags * 3, ACF_MAX_LAGS, len(x) - 1)
        approx_hours = lags * exp_step / 3600
        title = f"{tag_name} | ACF of {VALUE_COL} (first 50k points, {lags} lags ≈ {approx_hours:.1f}h)"
    else:
        lags = min(ACF_MAX_LAGS, len(x) - 1)
        title = f"{tag_name} | ACF of {VALUE_COL} ({lags} lags)"

    fig, ax = plt.subplots(figsize=(14, 5))

    plot_acf(
        x,
        lags=lags,
        ax=ax,
        alpha=0.05,
        color=PALETTE["main"],
        vlines_kwargs={"colors": PALETTE["main"], "linewidth": 0.8},
    )

    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Lag")
    ax.set_ylabel("Autocorrelation")

    save_fig(out / "14_acf.png")


def plot_value_by_year(df: pd.DataFrame, tag_name: str, out: Path) -> None:
    tmp = _profile_df(df)

    if tmp.empty:
        return

    tmp["year"] = tmp["_ts_parsed"].dt.year

    years = sorted(tmp["year"].dropna().unique())

    if not years:
        return

    fig, ax = plt.subplots(figsize=(12, 5))

    palette_y = sns.color_palette("tab10", len(years))

    for i, yr in enumerate(years):
        sub = tmp[tmp["year"] == yr][VALUE_COL].dropna()

        if sub.empty:
            continue

        ax.boxplot(
            sub.values,
            positions=[i],
            widths=0.5,
            patch_artist=True,
            boxprops=dict(facecolor=palette_y[i], alpha=0.7),
            flierprops=dict(marker=".", markersize=2, alpha=0.2),
            medianprops=dict(color="black", linewidth=2),
        )

    ax.set_xticks(range(len(years)))
    ax.set_xticklabels([str(y) for y in years])
    ax.set_title(f"{tag_name} | Load distribution by year", fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel(VALUE_COL)

    save_fig(out / "15_boxplot_by_year.png")


def plot_corr_heatmap(df: pd.DataFrame, tag_name: str, out: Path) -> None:
    num_df = df.select_dtypes(include=[np.number]).copy()

    drop_cols = [c for c in ["_rowid_"] if c in num_df.columns]
    num_df = num_df.drop(columns=drop_cols, errors="ignore")

    if num_df.shape[1] < 2:
        return

    if len(num_df) > 200_000:
        num_df = num_df.sample(200_000, random_state=42)

    corr = num_df.corr()

    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

    fig, ax = plt.subplots(figsize=(12, 9))

    sns.heatmap(
        corr,
        mask=mask,
        cmap="coolwarm",
        center=0,
        square=True,
        linewidths=0.4,
        ax=ax,
        annot=(corr.shape[0] <= 12),
        fmt=".2f",
        cbar_kws={"shrink": 0.75},
    )

    ax.set_title(f"{tag_name} | Correlation heatmap (numeric columns)", fontweight="bold")

    save_fig(out / "16_corr_heatmap.png")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 — Main EDA pipeline per database
# ─────────────────────────────────────────────────────────────────────────────

def run_eda(db_path: str) -> None:
    db_tag = tag(db_path)
    out_dir = OUTPUT_DIR / db_tag
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'=' * 70}")
    print(f"  Processing: {db_path}")
    print(f"{'=' * 70}")

    conn = sqlite3.connect(db_path)

    try:
        # ── 5.1 Read data ─────────────────────────────────────────────────────
        print("\n[1/8] Reading table from SQLite ...")

        df = read_table_chunked(conn, TABLE_NAME)

        if df.empty:
            raise ValueError(f"Table '{TABLE_NAME}' is empty.")

        for col in [TIMESTAMP_COL, VALUE_COL]:
            if col not in df.columns:
                raise ValueError(
                    f"Column '{col}' not found in '{TABLE_NAME}'. "
                    f"Available columns: {df.columns.tolist()}"
                )

        # ── 5.2 Parse and type conversion ─────────────────────────────────────
        print("[2/8] Parsing timestamps and converting numeric values ...")

        df[TIMESTAMP_COL] = df[TIMESTAMP_COL].astype("string").str.strip()
        df[VALUE_COL] = pd.to_numeric(df[VALUE_COL], errors="coerce")

        df["_ts_parsed"] = parse_timestamps(df[TIMESTAMP_COL])

        # Sort valid timestamps chronologically and push invalid timestamps to the end.
        df.sort_values(
            "_ts_parsed",
            inplace=True,
            ignore_index=True,
            na_position="last",
        )

        # ── 5.3 Timestamp quality flags ───────────────────────────────────────
        print("[3/8] Computing timestamp quality flags ...")

        timestamp_text_exists = df[TIMESTAMP_COL].notna() & df[TIMESTAMP_COL].ne("")
        timestamp_null_count = int(df[TIMESTAMP_COL].isna().sum())
        timestamp_empty_count = int(df[TIMESTAMP_COL].fillna("").eq("").sum())
        timestamp_unparseable_count = int((timestamp_text_exists & df["_ts_parsed"].isna()).sum())

        df["timestamp_parse_ok"] = df["_ts_parsed"].notna().astype("int64")

        valid_time_mask = df["_ts_parsed"].notna()

        df["is_duplicate_timestamp"] = 0

        if valid_time_mask.any():
            valid_ts = df.loc[valid_time_mask, "_ts_parsed"]
            dup_mask_valid = valid_ts.duplicated(keep=False)
            df.loc[valid_ts.index[dup_mask_valid], "is_duplicate_timestamp"] = 1

        duplicate_timestamps = int(df["is_duplicate_timestamp"].sum())

        # ── 5.4 Time deltas and gap flags ─────────────────────────────────────
        print("[4/8] Computing time deltas, irregular intervals, and gaps ...")

        df["time_delta_seconds"] = np.nan
        df["time_delta_minutes"] = np.nan
        df["previous_timestamp"] = pd.NaT

        if valid_time_mask.any():
            valid_idx = df.index[valid_time_mask]

            df.loc[valid_idx, "time_delta_seconds"] = (
                df.loc[valid_idx, "_ts_parsed"]
                .diff()
                .dt.total_seconds()
            )

            df.loc[valid_idx, "time_delta_minutes"] = (
                df.loc[valid_idx, "time_delta_seconds"] / 60.0
            )

            df.loc[valid_idx, "previous_timestamp"] = (
                df.loc[valid_idx, "_ts_parsed"].shift(1)
            )

        exp_step = mode_step(df["time_delta_seconds"])

        if pd.notna(exp_step) and exp_step > 0:
            irregular_threshold = exp_step * IRREGULAR_FACTOR
            gap_threshold = exp_step * GAP_FACTOR

            df["is_irregular_interval"] = (
                df["time_delta_seconds"] > irregular_threshold
            ).fillna(False).astype("int64")

            df["is_gap"] = (
                df["time_delta_seconds"] > gap_threshold
            ).fillna(False).astype("int64")

        else:
            irregular_threshold = np.nan
            gap_threshold = np.nan

            df["is_irregular_interval"] = 0
            df["is_gap"] = 0

        n_irregular_intervals = int(df["is_irregular_interval"].sum())
        n_gaps = int(df["is_gap"].sum())

        estimated_missing_intervals_all = estimate_missing_intervals(
            df["time_delta_seconds"],
            exp_step,
        )

        if pd.notna(exp_step) and exp_step > 0:
            gap_deltas = df.loc[df["is_gap"] == 1, "time_delta_seconds"].dropna()
            estimated_missing_intervals_in_large_gaps = int(
                np.maximum(np.round(gap_deltas / exp_step) - 1, 0).sum()
            )
        else:
            estimated_missing_intervals_in_large_gaps = 0

        # ── 5.5 Value quality flags and outliers ──────────────────────────────
        print("[5/8] Computing value quality flags and outliers ...")

        df["is_value_null"] = df[VALUE_COL].isna().astype("int64")
        df["is_negative_value"] = (df[VALUE_COL] < 0).fillna(False).astype("int64")

        value_null_count = int(df["is_value_null"].sum())
        negative_values = int(df["is_negative_value"].sum())
        zero_values = int((df[VALUE_COL] == 0).fillna(False).sum())

        v = df[VALUE_COL]
        v_non_null = v.dropna()

        stats_dict = describe_extended(v)

        if v_non_null.empty:
            q1 = q3 = iqr_low = iqr_high = np.nan
            n_iqr_outliers = 0
            n_mad_outliers = 0
            df["is_iqr_outlier"] = 0
            df["robust_zscore_abs"] = np.nan
            df["is_mad_outlier"] = 0

        else:
            q1, q3, iqr_low, iqr_high = iqr_bounds(v_non_null)

            df["is_iqr_outlier"] = (
                (df[VALUE_COL] < iqr_low) |
                (df[VALUE_COL] > iqr_high)
            ).fillna(False).astype("int64")

            n_iqr_outliers = int(df["is_iqr_outlier"].sum())

            rz = mad_zscore(v)
            df["robust_zscore_abs"] = rz.abs()

            df["is_mad_outlier"] = (
                df["robust_zscore_abs"] > ROBUST_Z_THRESH
            ).fillna(False).astype("int64")

            n_mad_outliers = int(df["is_mad_outlier"].sum())

        # Conservative bad sample definition:
        # Do not mark high IQR/MAD outliers as bad automatically.
        # High peaks can be real load behavior.
        df["is_bad_sample"] = (
            (df["timestamp_parse_ok"] == 0) |
            (df["is_value_null"] == 1) |
            (df["is_negative_value"] == 1)
        ).astype("int64")

        bad_samples = int(df["is_bad_sample"].sum())

        # ── 5.6 Audit report ─────────────────────────────────────────────────
        print("[6/8] Saving CSV reports ...")

        ts_min = df["_ts_parsed"].min()
        ts_max = df["_ts_parsed"].max()

        time_delta_null_count = int(df["time_delta_seconds"].isna().sum())

        expected_time_delta_null_count = (
            int(valid_time_mask.any()) +
            int((~valid_time_mask).sum())
        )

        null_report = {
            f"{TIMESTAMP_COL}_null": timestamp_null_count,
            f"{TIMESTAMP_COL}_empty": timestamp_empty_count,
            f"{TIMESTAMP_COL}_unparseable": timestamp_unparseable_count,
            f"{VALUE_COL}_null": value_null_count,
            "time_delta_seconds_null": time_delta_null_count,
        }

        audit = {
            "dataset": db_tag,
            "table": TABLE_NAME,
            "total_rows": len(df),
            "valid_timestamp_rows": int(valid_time_mask.sum()),
            "invalid_timestamp_rows": int((~valid_time_mask).sum()),
            "time_range_start": str(ts_min),
            "time_range_end": str(ts_max),
            "expected_step_seconds": exp_step,
            "expected_step_minutes": exp_step / 60 if pd.notna(exp_step) else np.nan,
            "irregular_factor": IRREGULAR_FACTOR,
            "irregular_threshold_seconds": irregular_threshold,
            "gap_factor": GAP_FACTOR,
            "gap_threshold_seconds": gap_threshold,
            "n_irregular_intervals": n_irregular_intervals,
            "n_gaps_detected": n_gaps,
            "estimated_missing_intervals_all_irregular": estimated_missing_intervals_all,
            "estimated_missing_intervals_in_large_gaps": estimated_missing_intervals_in_large_gaps,
            "duplicate_timestamps_valid_only": duplicate_timestamps,
            "negative_values": negative_values,
            "zero_values": zero_values,
            "bad_samples_conservative": bad_samples,
            "time_delta_null_count": time_delta_null_count,
            "expected_time_delta_null_count": expected_time_delta_null_count,
            **null_report,
        }

        pd.Series(audit).to_frame("value").to_csv(
            out_dir / "audit_report.csv",
            header=True,
        )

        pd.Series(stats_dict).to_frame("value").to_csv(
            out_dir / "descriptive_statistics.csv",
            header=True,
        )

        outlier_report = {
            "iqr_Q1": q1,
            "iqr_Q3": q3,
            "iqr_lower_fence": iqr_low,
            "iqr_upper_fence": iqr_high,
            "n_iqr_outliers": n_iqr_outliers,
            "pct_iqr_outliers": n_iqr_outliers / max(len(df), 1) * 100,
            "robust_z_threshold": ROBUST_Z_THRESH,
            "n_robust_outliers": n_mad_outliers,
            "pct_robust_outliers": n_mad_outliers / max(len(df), 1) * 100,
            "note": "IQR/MAD outliers are statistical outliers, not automatically bad samples.",
        }

        pd.Series(outlier_report).to_frame("value").to_csv(
            out_dir / "outlier_report.csv",
            header=True,
        )

        # Save unparseable timestamp examples.
        bad_ts_examples = df.loc[
            timestamp_text_exists & df["_ts_parsed"].isna(),
            ["_rowid_", TIMESTAMP_COL, VALUE_COL],
        ].head(MAX_EXAMPLE_ROWS)

        bad_ts_examples.to_csv(
            out_dir / "unparseable_timestamp_examples.csv",
            index=False,
        )

        # Save duplicate timestamp examples.
        duplicate_examples = df.loc[
            df["is_duplicate_timestamp"] == 1,
            ["_rowid_", TIMESTAMP_COL, "_ts_parsed", VALUE_COL],
        ].head(MAX_EXAMPLE_ROWS)

        duplicate_examples.to_csv(
            out_dir / "duplicate_timestamp_examples.csv",
            index=False,
        )

        # Save gap examples.
        gap_examples = df.loc[
            df["is_gap"] == 1,
            [
                "_rowid_",
                TIMESTAMP_COL,
                "previous_timestamp",
                "_ts_parsed",
                "time_delta_seconds",
                "time_delta_minutes",
                VALUE_COL,
            ],
        ].head(MAX_EXAMPLE_ROWS)

        gap_examples.to_csv(
            out_dir / "gap_examples.csv",
            index=False,
        )

        # Save bad sample examples.
        bad_sample_examples = df.loc[
            df["is_bad_sample"] == 1,
            [
                "_rowid_",
                TIMESTAMP_COL,
                "_ts_parsed",
                VALUE_COL,
                "timestamp_parse_ok",
                "is_value_null",
                "is_negative_value",
                "is_bad_sample",
            ],
        ].head(MAX_EXAMPLE_ROWS)

        bad_sample_examples.to_csv(
            out_dir / "bad_sample_examples.csv",
            index=False,
        )

        # Delta distribution table.
        d_pos = df["time_delta_seconds"].dropna()
        d_pos = d_pos[d_pos > 0]

        if not d_pos.empty:
            delta_tbl = (
                (d_pos * 100).round().astype("Int64")
                .value_counts()
                .rename_axis("delta_cs")
                .reset_index(name="count")
            )

            delta_tbl["delta_seconds"] = delta_tbl["delta_cs"] / 100.0
            delta_tbl["delta_minutes"] = delta_tbl["delta_seconds"] / 60.0

            delta_tbl.sort_values("count", ascending=False).to_csv(
                out_dir / "time_delta_distribution.csv",
                index=False,
            )

        # Print summary.
        print(f"\n  ── Summary for {db_tag} ──")
        print(f"  Rows                         : {len(df):,}")
        print(f"  Time range                   : {ts_min} → {ts_max}")

        if pd.notna(exp_step):
            print(f"  Expected step                : {exp_step:.2f}s ({exp_step / 60:.2f} min)")
        else:
            print("  Expected step                : unknown")

        print(f"  Timestamp null               : {timestamp_null_count:,}")
        print(f"  Timestamp empty              : {timestamp_empty_count:,}")
        print(f"  Timestamp unparseable        : {timestamp_unparseable_count:,}")
        print(f"  Value null                   : {value_null_count:,}")
        print(f"  Duplicate timestamps         : {duplicate_timestamps:,} valid parsed timestamps only")
        print(f"  Irregular intervals          : {n_irregular_intervals:,}")
        print(f"  Large gaps                   : {n_gaps:,}")
        print(f"  Est. missing intervals all   : {estimated_missing_intervals_all:,}")
        print(f"  Est. missing intervals gaps  : {estimated_missing_intervals_in_large_gaps:,}")
        print(f"  Negative values              : {negative_values:,}")
        print(f"  Bad samples conservative     : {bad_samples:,}")
        print(f"  time_delta_seconds NULL      : {time_delta_null_count:,}")
        print(f"  Expected time_delta NULL     : {expected_time_delta_null_count:,}")

        if stats_dict:
            print(f"  Min / Max                    : {stats_dict.get('min'):.6f} / {stats_dict.get('max'):.6f}")
            print(f"  Mean ± Std                   : {stats_dict.get('mean'):.6f} ± {stats_dict.get('std'):.6f}")
            print(f"  Skewness                     : {stats_dict.get('skewness'):.4f}")
            print(f"  Kurtosis                     : {stats_dict.get('kurtosis'):.4f}")

        print(f"  IQR outliers                 : {n_iqr_outliers:,} ({outlier_report['pct_iqr_outliers']:.2f}%)")
        print(f"  MAD outliers                 : {n_mad_outliers:,} ({outlier_report['pct_robust_outliers']:.2f}%)")
        print()

        # ── 5.7 Write corrected columns back to SQLite ────────────────────────
        print("[7/8] Writing corrected QC columns to SQLite ...")

        ensure_columns(
            conn,
            TABLE_NAME,
            {
                "time_delta_seconds": "REAL",
                "time_delta_minutes": "REAL",
                "is_irregular_interval": "INTEGER",
                "is_gap": "INTEGER",
                "timestamp_parse_ok": "INTEGER",
                "is_duplicate_timestamp": "INTEGER",
                "is_negative_value": "INTEGER",
                "is_iqr_outlier": "INTEGER",
                "is_mad_outlier": "INTEGER",
                "is_bad_sample": "INTEGER",
            },
        )

        batch_update_rowid(
            conn,
            TABLE_NAME,
            df,
            col_map={
                "time_delta_seconds": "time_delta_seconds",
                "time_delta_minutes": "time_delta_minutes",
                "is_irregular_interval": "is_irregular_interval",
                "is_gap": "is_gap",
                "timestamp_parse_ok": "timestamp_parse_ok",
                "is_duplicate_timestamp": "is_duplicate_timestamp",
                "is_negative_value": "is_negative_value",
                "is_iqr_outlier": "is_iqr_outlier",
                "is_mad_outlier": "is_mad_outlier",
                "is_bad_sample": "is_bad_sample",
            },
        )

        print()

        # ── 5.8 Generate plots ────────────────────────────────────────────────
        print("[8/8] Generating plots ...")

        plot_value_over_time(df, db_tag, out_dir)
        print("       01_value_over_time.png              ✓")

        plot_scatter_time(df, db_tag, out_dir)
        print("       02_scatter_value_time.png           ✓")

        plot_hist_kde(v, db_tag, out_dir)
        print("       03_hist_kde_ecdf.png                ✓")

        plot_box_violin(v, db_tag, out_dir)
        print("       04_box_violin.png                   ✓")

        if pd.notna(iqr_low) and pd.notna(iqr_high):
            plot_outliers(df, db_tag, out_dir, iqr_low, iqr_high, n_iqr_outliers)
            print("       05_outliers_iqr.png                 ✓")

        plot_mad_outliers(df, db_tag, out_dir)
        print("       06_mad_robust_zscore.png            ✓")

        plot_delta_distribution(df["time_delta_seconds"], db_tag, out_dir, exp_step)
        print("       07_time_delta_analysis.png          ✓")

        plot_missing_bar(null_report, db_tag, out_dir)
        print("       08_missing_counts.png               ✓")

        plot_gap_scatter(df, exp_step, db_tag, out_dir)
        print("       09_gap_locations.png                ✓")

        plot_min_max(df, db_tag, out_dir)
        print("       10_min_max_highlight.png            ✓")

        plot_monthly_profile(df, db_tag, out_dir)
        print("       11_monthly_profile.png              ✓")

        plot_hourly_profile(df, db_tag, out_dir)
        print("       12_hourly_profile.png               ✓")

        plot_heatmap_hour_dow(df, db_tag, out_dir)
        print("       13_heatmap_hour_dow.png             ✓")

        plot_acf_proper(df, exp_step, db_tag, out_dir)
        print("       14_acf.png                          ✓")

        plot_value_by_year(df, db_tag, out_dir)
        print("       15_boxplot_by_year.png              ✓")

        plot_corr_heatmap(df, db_tag, out_dir)
        print("       16_corr_heatmap.png                 ✓")

        print(f"\n  All outputs saved → {out_dir}\n")

    finally:
        conn.close()


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 — Entry point
# ─────────────────────────────────────────────────────────────────────────────

def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    for db in DB_PATHS:
        if not os.path.isfile(db):
            print(f"\n[SKIP] File not found: {db}")
            continue

        try:
            run_eda(db)

        except Exception as exc:
            print(f"\n[ERROR] {db}: {exc}")
            raise

    print("\n" + "=" * 70)
    print("  EDA complete for all databases.")
    print("=" * 70)


if __name__ == "__main__":
    main()

In [ ]:
"""
STEP 2 — Data Cleaning & 15-Minute Resampling
==============================================
Reads MODIFIED_DATASET, already audited in Step 1, from each SQLite .db file.

Actions:
  1. Load MODIFIED_DATASET and parse timestamps robustly
  2. Remove bad samples:
       - invalid / unparseable timestamps
       - null values
       - negative load values
  3. Resolve duplicate timestamps by taking the mean value per timestamp
  4. Create CLEANED_DATASET table inside the same .db file
  5. Resample to 15-minute intervals with:
       - mean value per interval
       - sample count per interval
       - expected sample count per interval
       - empty / partial / complete interval flags
       - reliability flag
       - real large-gap flag based on the original cleaned 1-minute sequence
  6. Add time feature columns to DATASET_15MIN:
       hour, minute, minute_of_day, day_of_week, month, year,
       day_of_year, quarter, week_of_year, is_weekend, season
  7. Create DATASET_15MIN table inside the same .db file
  8. Generate publication-quality plots
  9. Save CSV reports:
       - cleaning_report.csv
       - resample_report.csv
       - dataset_15min_sample.csv

Important corrections:
  - is_gap in DATASET_15MIN is computed from real gaps in the original cleaned data,
    not from the regular 15-minute grid.
  - Reliable interval threshold is set to 12 samples out of 15, i.e. about 80%.
  - Final 15-minute profiles and load duration curve use reliable intervals only.
  - Empty, partial, complete, and reliable coverage flags are saved.

All comments are in English.
Raw values are not rounded or modified.
"""

from __future__ import annotations

import os
import sqlite3
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)


# Portable project paths for GitHub
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_BASE_DIR = PROJECT_ROOT / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)

# ═════════════════════════════════════════════════════════════════════════════
# USER CONFIGURATION
# ═════════════════════════════════════════════════════════════════════════════

DB_PATHS = [
    DATA_DIR / "campus_load.db",
]

TABLE_SOURCE   = "MODIFIED_DATASET"
TABLE_CLEANED  = "CLEANED_DATASET"
TABLE_15MIN    = "DATASET_15MIN"

TIMESTAMP_COL  = "data_timestamp"
VALUE_COL      = "value"

OUTPUT_DIR     = OUTPUT_BASE_DIR / "STEP2_OUTPUTS"

CHUNK_SIZE     = 300_000

RESAMPLE_FREQ  = "15min"

# For 1-minute data, each 15-minute interval should contain 15 samples.
# 12 means at least 80% coverage is required for a reliable interval.
MIN_SAMPLES_PER_INTERVAL = 12

# A large gap in the cleaned original time series.
# Example: if two consecutive valid raw samples are more than 30 minutes apart,
# the later 15-minute interval is marked as having a large gap before it.
GAP_THRESHOLD_MINUTES = 30.0

# ═════════════════════════════════════════════════════════════════════════════


# ── Plot aesthetics ───────────────────────────────────────────────────────────

sns.set_theme(style="whitegrid", context="talk", font_scale=1.05)

plt.rcParams.update(
    {
        "figure.dpi": 130,
        "savefig.dpi": 200,
        "axes.spines.right": False,
        "axes.spines.top": False,
        "figure.constrained_layout.use": True,
    }
)

PALETTE = {
    "main": "#2196F3",
    "cleaned": "#43A047",
    "outlier": "#E53935",
    "gap": "#FF6F00",
    "bar": "#5C6BC0",
    "neutral": "#78909C",
    "warn": "#FFC107",
}


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — General helpers
# ─────────────────────────────────────────────────────────────────────────────

def qident(s: str) -> str:
    """Safely quote a SQLite identifier."""
    return '"' + s.replace('"', '""') + '"'


def tag(db_path: str) -> str:
    """Return database file name without extension."""
    return Path(db_path).stem


def save_fig(path: Path) -> None:
    """Save current matplotlib figure and close all figures."""
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, bbox_inches="tight")
    plt.close("all")


def create_index(conn: sqlite3.Connection, table: str, column: str, index_name: str) -> None:
    """Create a SQLite index if it does not already exist."""
    cur = conn.cursor()
    cur.execute(
        f"CREATE INDEX IF NOT EXISTS {qident(index_name)} "
        f"ON {qident(table)} ({qident(column)});"
    )
    conn.commit()


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — SQLite I/O helpers
# ─────────────────────────────────────────────────────────────────────────────

def read_table_chunked(conn: sqlite3.Connection, table: str) -> pd.DataFrame:
    """Read a SQLite table in chunks and return a single DataFrame."""
    table_q = qident(table)

    n = pd.read_sql_query(
        f"SELECT COUNT(*) AS n FROM {table_q};",
        conn
    ).iloc[0, 0]

    print(f"    Rows in {table}: {n:,}")

    chunks = []
    offset = 0

    while offset < n:
        chunk = pd.read_sql_query(
            f"""
            SELECT rowid AS _rowid_, *
            FROM {table_q}
            ORDER BY rowid
            LIMIT {CHUNK_SIZE}
            OFFSET {offset};
            """,
            conn,
        )

        chunks.append(chunk)
        offset += CHUNK_SIZE

        print(f"    Read {min(offset, n):,} / {n:,}", end="\r")

    print()

    if not chunks:
        return pd.DataFrame()

    return pd.concat(chunks, ignore_index=True)


def drop_and_create_table_from_df(
    conn: sqlite3.Connection,
    df: pd.DataFrame,
    table: str,
    if_exists: str = "replace",
) -> None:
    """
    Write a DataFrame to SQLite.

    The table is replaced by default.
    Datetime columns should already be converted to strings before this function.
    """
    df_to_write = df.copy()

    # Convert nullable integer-like columns safely.
    for col in df_to_write.columns:
        if pd.api.types.is_integer_dtype(df_to_write[col]):
            df_to_write[col] = df_to_write[col].astype(object).where(
                df_to_write[col].notna(),
                other=None,
            )

    df_to_write.to_sql(table, conn, if_exists=if_exists, index=False)
    conn.commit()

    print(f"    Saved {len(df_to_write):,} rows → SQLite table '{table}'")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — Timestamp parsing
# ─────────────────────────────────────────────────────────────────────────────

def parse_timestamps(ts_series: pd.Series) -> pd.Series:
    """
    Robust timestamp parser.

    Handles:
      - YYYY-MM-DD HH:MM:SS
      - YYYY-MM-DD HH:MM:SS.sss
      - timezone-aware timestamps

    All parsed timestamps are normalized to timezone-naive UTC-compatible
    datetime64[ns] so they can be sorted and compared safely.
    """
    s = ts_series.astype("string").str.strip()

    parsed = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")

    mask = s.notna() & s.ne("")

    if not mask.any():
        return parsed

    try:
        tmp = pd.to_datetime(
            s.loc[mask],
            errors="coerce",
            format="mixed",
            utc=True,
            cache=True,
        )
    except TypeError:
        tmp = pd.to_datetime(
            s.loc[mask],
            errors="coerce",
            utc=True,
            cache=True,
        )

    if hasattr(tmp, "dt"):
        tmp = tmp.dt.tz_convert(None)
    else:
        tmp = tmp.tz_convert(None)

    parsed.loc[mask] = tmp

    return parsed


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — Feature helpers
# ─────────────────────────────────────────────────────────────────────────────

def get_season(month: pd.Series) -> pd.Series:
    """
    Northern hemisphere seasons:
      0 = Winter: Dec, Jan, Feb
      1 = Spring: Mar, Apr, May
      2 = Summer: Jun, Jul, Aug
      3 = Autumn: Sep, Oct, Nov
    """
    conditions = [
        month.isin([12, 1, 2]),
        month.isin([3, 4, 5]),
        month.isin([6, 7, 8]),
        month.isin([9, 10, 11]),
    ]

    choices = [0, 1, 2, 3]

    return pd.Series(
        np.select(conditions, choices, default=-1),
        index=month.index,
    )


def add_time_features(df: pd.DataFrame, timestamp_col: str) -> pd.DataFrame:
    """Add standard calendar features from a timestamp column."""
    out = df.copy()

    ts = pd.to_datetime(out[timestamp_col], errors="coerce")

    out["hour"] = ts.dt.hour.astype("int64")
    out["minute"] = ts.dt.minute.astype("int64")
    out["minute_of_day"] = (ts.dt.hour * 60 + ts.dt.minute).astype("int64")
    out["day_of_week"] = ts.dt.dayofweek.astype("int64")
    out["month"] = ts.dt.month.astype("int64")
    out["year"] = ts.dt.year.astype("int64")
    out["day_of_year"] = ts.dt.dayofyear.astype("int64")
    out["quarter"] = ts.dt.quarter.astype("int64")
    out["week_of_year"] = ts.dt.isocalendar().week.astype("int64")
    out["is_weekend"] = (ts.dt.dayofweek >= 5).astype("int64")
    out["season"] = get_season(out["month"]).astype("int64")

    return out


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 — Plotting functions
# ─────────────────────────────────────────────────────────────────────────────

def _ds(df: pd.DataFrame, max_pts: int, ts_col: str) -> pd.DataFrame:
    """Downsample for plotting only."""
    if df.empty:
        return df

    if len(df) > max_pts:
        return df.sample(max_pts, random_state=42).sort_values(ts_col)

    return df.sort_values(ts_col)


def plot_raw_vs_cleaned(
    df_raw: pd.DataFrame,
    df_clean: pd.DataFrame,
    tag_name: str,
    out: Path,
) -> None:
    """Compare raw and cleaned time series."""
    raw_plot = _ds(
        df_raw.dropna(subset=["_ts_parsed", VALUE_COL]),
        200_000,
        "_ts_parsed",
    )

    clean_plot = _ds(
        df_clean.dropna(subset=["_ts_parsed", VALUE_COL]),
        200_000,
        "_ts_parsed",
    )

    if raw_plot.empty or clean_plot.empty:
        return

    fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=False)

    axes[0].plot(
        raw_plot["_ts_parsed"],
        raw_plot[VALUE_COL],
        linewidth=0.5,
        alpha=0.75,
        color=PALETTE["main"],
        label="Raw",
    )
    axes[0].set_title("Raw data before cleaning")
    axes[0].set_ylabel(VALUE_COL)
    axes[0].legend(loc="upper right", fontsize=10)

    axes[1].plot(
        clean_plot["_ts_parsed"],
        clean_plot[VALUE_COL],
        linewidth=0.5,
        alpha=0.75,
        color=PALETTE["cleaned"],
        label="Cleaned",
    )
    axes[1].set_title("Cleaned data after removing invalid timestamps, nulls, negatives, and duplicates")
    axes[1].set_ylabel(VALUE_COL)
    axes[1].set_xlabel("Time")
    axes[1].legend(loc="upper right", fontsize=10)

    fig.suptitle(f"{tag_name} | Raw vs Cleaned data", fontweight="bold")

    save_fig(out / "01_raw_vs_cleaned.png")


def plot_resampled_over_time(df15: pd.DataFrame, tag_name: str, out: Path) -> None:
    """Plot reliable 15-minute resampled load."""
    plot_df = _ds(
        df15.dropna(subset=["timestamp_15min", VALUE_COL]),
        200_000,
        "timestamp_15min",
    )

    if plot_df.empty:
        return

    fig, ax = plt.subplots(figsize=(16, 5))

    ax.plot(
        plot_df["timestamp_15min"],
        plot_df[VALUE_COL],
        linewidth=0.6,
        alpha=0.85,
        color=PALETTE["cleaned"],
    )

    ax.set_title(f"{tag_name} | 15-minute resampled load, reliable intervals only", fontweight="bold")
    ax.set_xlabel("Time")
    ax.set_ylabel("Mean load, 15-min")

    save_fig(out / "02_resampled_15min_reliable.png")


def plot_sample_count_hist(df15: pd.DataFrame, tag_name: str, out: Path) -> None:
    """Histogram of sample counts per 15-minute interval."""
    counts = df15["sample_count"].dropna()

    if counts.empty:
        return

    max_count = int(max(counts.max(), MIN_SAMPLES_PER_INTERVAL))

    fig, ax = plt.subplots(figsize=(12, 5))

    ax.hist(
        counts,
        bins=range(0, max_count + 2),
        color=PALETTE["bar"],
        edgecolor="white",
        alpha=0.9,
    )

    ax.axvline(
        MIN_SAMPLES_PER_INTERVAL,
        color=PALETTE["outlier"],
        linestyle="--",
        linewidth=2,
        label=f"Min reliable = {MIN_SAMPLES_PER_INTERVAL}",
    )

    ax.set_title(f"{tag_name} | Sample count per 15-min interval", fontweight="bold")
    ax.set_xlabel("Number of 1-min samples in interval")
    ax.set_ylabel("Number of intervals")
    ax.legend()

    save_fig(out / "03_sample_count_hist.png")


def plot_coverage_bar(df15: pd.DataFrame, tag_name: str, out: Path) -> None:
    """Show reliable vs unreliable interval counts per year."""
    tmp = df15.dropna(subset=["timestamp_15min"]).copy()

    if tmp.empty:
        return

    tmp["year"] = pd.to_datetime(tmp["timestamp_15min"]).dt.year
    tmp["reliable"] = tmp["is_reliable"].map({1: "Reliable", 0: "Unreliable"})

    group = (
        tmp.groupby(["year", "reliable"])
        .size()
        .reset_index(name="count")
    )

    years = sorted(group["year"].unique())

    x = np.arange(len(years))
    width = 0.35

    reliable_vals = [
        group.loc[
            (group["year"] == y) &
            (group["reliable"] == "Reliable"),
            "count",
        ].sum()
        for y in years
    ]

    unreliable_vals = [
        group.loc[
            (group["year"] == y) &
            (group["reliable"] == "Unreliable"),
            "count",
        ].sum()
        for y in years
    ]

    fig, ax = plt.subplots(figsize=(13, 5))

    bars1 = ax.bar(
        x - width / 2,
        reliable_vals,
        width,
        label="Reliable",
        color=PALETTE["cleaned"],
        alpha=0.85,
    )

    bars2 = ax.bar(
        x + width / 2,
        unreliable_vals,
        width,
        label="Unreliable",
        color=PALETTE["warn"],
        alpha=0.85,
    )

    ax.bar_label(bars1, fmt="%d", padding=2, fontsize=9)
    ax.bar_label(bars2, fmt="%d", padding=2, fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels([str(y) for y in years])
    ax.set_title(f"{tag_name} | Reliable vs unreliable 15-min intervals per year", fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel("Number of intervals")
    ax.legend()

    save_fig(out / "04_coverage_bar_by_year.png")


def plot_missing_heatmap(df15: pd.DataFrame, tag_name: str, out: Path) -> None:
    """Heatmap of empty intervals by hour and day of week."""
    tmp = df15.copy()

    if tmp.empty:
        return

    tmp["ts"] = pd.to_datetime(tmp["timestamp_15min"], errors="coerce")
    tmp = tmp.dropna(subset=["ts"])

    if tmp.empty:
        return

    tmp["hour"] = tmp["ts"].dt.hour
    tmp["dow"] = tmp["ts"].dt.dayofweek
    tmp["missing"] = (tmp["sample_count"] == 0).astype(int)

    pivot = tmp.pivot_table(
        values="missing",
        index="hour",
        columns="dow",
        aggfunc="mean",
    )

    pivot = pivot.reindex(index=range(24), columns=range(7))
    pivot.columns = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

    fig, ax = plt.subplots(figsize=(12, 9))

    sns.heatmap(
        pivot * 100,
        cmap="Reds",
        ax=ax,
        cbar_kws={"label": "% of intervals with 0 samples"},
        linewidths=0.3,
        linecolor="white",
        vmin=0,
        vmax=100,
    )

    ax.set_title(f"{tag_name} | Empty 15-min intervals — hour vs day of week", fontweight="bold")
    ax.set_xlabel("Day of week")
    ax.set_ylabel("Hour of day")

    save_fig(out / "05_missing_heatmap_hour_dow.png")


def plot_gap_heatmap(df15: pd.DataFrame, tag_name: str, out: Path) -> None:
    """Heatmap of large-gap intervals by hour and day of week."""
    tmp = df15.copy()

    if tmp.empty or "is_gap" not in tmp.columns:
        return

    tmp["ts"] = pd.to_datetime(tmp["timestamp_15min"], errors="coerce")
    tmp = tmp.dropna(subset=["ts"])

    if tmp.empty:
        return

    tmp["hour"] = tmp["ts"].dt.hour
    tmp["dow"] = tmp["ts"].dt.dayofweek

    pivot = tmp.pivot_table(
        values="is_gap",
        index="hour",
        columns="dow",
        aggfunc="mean",
    )

    pivot = pivot.reindex(index=range(24), columns=range(7))
    pivot.columns = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

    fig, ax = plt.subplots(figsize=(12, 9))

    sns.heatmap(
        pivot * 100,
        cmap="Oranges",
        ax=ax,
        cbar_kws={"label": "% of intervals after large gap"},
        linewidths=0.3,
        linecolor="white",
        vmin=0,
    )

    ax.set_title(f"{tag_name} | Large-gap intervals — hour vs day of week", fontweight="bold")
    ax.set_xlabel("Day of week")
    ax.set_ylabel("Hour of day")

    save_fig(out / "06_gap_heatmap_hour_dow.png")


def plot_cleaned_monthly(df_clean: pd.DataFrame, tag_name: str, out: Path) -> None:
    """Monthly profile from cleaned 1-minute data."""
    tmp = df_clean.dropna(subset=["_ts_parsed", VALUE_COL]).copy()

    if tmp.empty:
        return

    tmp["month"] = tmp["_ts_parsed"].dt.month

    monthly = tmp.groupby("month")[VALUE_COL].agg(["mean", "std"]).reset_index()

    month_labels = [
        "Jan", "Feb", "Mar", "Apr", "May", "Jun",
        "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
    ]

    fig, ax = plt.subplots(figsize=(12, 5))

    ax.bar(
        monthly["month"],
        monthly["mean"],
        yerr=monthly["std"],
        capsize=4,
        color=PALETTE["cleaned"],
        edgecolor="none",
        alpha=0.85,
        error_kw={"elinewidth": 1.2, "ecolor": PALETTE["neutral"]},
    )

    ax.set_xticks(monthly["month"])
    ax.set_xticklabels([month_labels[m - 1] for m in monthly["month"]])

    ax.set_title(f"{tag_name} | Monthly load profile after cleaning", fontweight="bold")
    ax.set_xlabel("Month")
    ax.set_ylabel("Mean load")

    save_fig(out / "07_cleaned_monthly_profile.png")


def plot_cleaned_hourly(df_clean: pd.DataFrame, tag_name: str, out: Path) -> None:
    """Hourly profile from cleaned 1-minute data."""
    tmp = df_clean.dropna(subset=["_ts_parsed", VALUE_COL]).copy()

    if tmp.empty:
        return

    tmp["hour"] = tmp["_ts_parsed"].dt.hour

    hourly = tmp.groupby("hour")[VALUE_COL].agg(["mean", "std"]).reset_index()

    fig, ax = plt.subplots(figsize=(13, 5))

    ax.fill_between(
        hourly["hour"],
        hourly["mean"] - hourly["std"],
        hourly["mean"] + hourly["std"],
        alpha=0.25,
        color=PALETTE["cleaned"],
        label="±1 std",
    )

    ax.plot(
        hourly["hour"],
        hourly["mean"],
        color=PALETTE["cleaned"],
        linewidth=2.5,
        marker="o",
        markersize=5,
        label="Mean",
    )

    ax.set_xticks(range(0, 24))
    ax.set_title(f"{tag_name} | Hourly load profile after cleaning", fontweight="bold")
    ax.set_xlabel("Hour of day")
    ax.set_ylabel("Mean load")
    ax.legend()

    save_fig(out / "08_cleaned_hourly_profile.png")


def plot_cleaned_heatmap(df_clean: pd.DataFrame, tag_name: str, out: Path) -> None:
    """Hour vs day-of-week heatmap from cleaned 1-minute data."""
    tmp = df_clean.dropna(subset=["_ts_parsed", VALUE_COL]).copy()

    if tmp.empty:
        return

    tmp["hour"] = tmp["_ts_parsed"].dt.hour
    tmp["dow"] = tmp["_ts_parsed"].dt.dayofweek

    pivot = tmp.pivot_table(
        values=VALUE_COL,
        index="hour",
        columns="dow",
        aggfunc="mean",
    )

    pivot = pivot.reindex(index=range(24), columns=range(7))
    pivot.columns = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

    fig, ax = plt.subplots(figsize=(12, 9))

    sns.heatmap(
        pivot,
        cmap="YlOrRd",
        ax=ax,
        cbar_kws={"label": f"Mean {VALUE_COL}"},
        linewidths=0.3,
        linecolor="white",
    )

    ax.set_title(f"{tag_name} | Mean load after cleaning — hour vs day of week", fontweight="bold")
    ax.set_xlabel("Day of week")
    ax.set_ylabel("Hour of day")

    save_fig(out / "09_cleaned_heatmap_hour_dow.png")


def plot_yearly_boxplot_cleaned(df_clean: pd.DataFrame, tag_name: str, out: Path) -> None:
    """Yearly boxplot from cleaned data."""
    tmp = df_clean.dropna(subset=["_ts_parsed", VALUE_COL]).copy()

    if tmp.empty:
        return

    tmp["year"] = tmp["_ts_parsed"].dt.year

    years = sorted(tmp["year"].dropna().unique())

    if not years:
        return

    palette_y = sns.color_palette("tab10", len(years))

    fig, ax = plt.subplots(figsize=(12, 5))

    for i, yr in enumerate(years):
        sub = tmp[tmp["year"] == yr][VALUE_COL].dropna()

        if sub.empty:
            continue

        ax.boxplot(
            sub.values,
            positions=[i],
            widths=0.5,
            patch_artist=True,
            boxprops=dict(facecolor=palette_y[i], alpha=0.7),
            flierprops=dict(marker=".", markersize=2, alpha=0.2),
            medianprops=dict(color="black", linewidth=2),
        )

    ax.set_xticks(range(len(years)))
    ax.set_xticklabels([str(y) for y in years])
    ax.set_title(f"{tag_name} | Load distribution by year, cleaned data", fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel(VALUE_COL)

    save_fig(out / "10_cleaned_yearly_boxplot.png")


def plot_load_duration_curve(df15: pd.DataFrame, tag_name: str, out: Path) -> None:
    """Load duration curve from reliable 15-minute data."""
    vals = df15[VALUE_COL].dropna().sort_values(ascending=False).values

    if len(vals) == 0:
        return

    pct = np.linspace(0, 100, len(vals))

    fig, ax = plt.subplots(figsize=(13, 5))

    ax.plot(
        pct,
        vals,
        color=PALETTE["bar"],
        linewidth=1.5,
    )

    ax.axhline(
        np.mean(vals),
        color=PALETTE["outlier"],
        linestyle="--",
        linewidth=1.5,
        label=f"Mean = {np.mean(vals):.2f}",
    )

    ax.axhline(
        np.median(vals),
        color=PALETTE["cleaned"],
        linestyle="--",
        linewidth=1.5,
        label=f"Median = {np.median(vals):.2f}",
    )

    ax.set_title(f"{tag_name} | Load duration curve, reliable 15-min intervals", fontweight="bold")
    ax.set_xlabel("% of time load is exceeded")
    ax.set_ylabel("Load value")
    ax.legend()

    save_fig(out / "11_load_duration_curve_reliable.png")


def plot_15min_monthly(df15: pd.DataFrame, tag_name: str, out: Path) -> None:
    """Monthly profile from reliable 15-minute data."""
    tmp = df15.dropna(subset=["timestamp_15min", VALUE_COL]).copy()

    if tmp.empty:
        return

    tmp["month"] = pd.to_datetime(tmp["timestamp_15min"]).dt.month

    monthly = tmp.groupby("month")[VALUE_COL].agg(["mean", "std"]).reset_index()

    month_labels = [
        "Jan", "Feb", "Mar", "Apr", "May", "Jun",
        "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
    ]

    fig, ax = plt.subplots(figsize=(12, 5))

    ax.bar(
        monthly["month"],
        monthly["mean"],
        yerr=monthly["std"],
        capsize=4,
        color=PALETTE["bar"],
        edgecolor="none",
        alpha=0.85,
        error_kw={"elinewidth": 1.2, "ecolor": PALETTE["neutral"]},
    )

    ax.set_xticks(monthly["month"])
    ax.set_xticklabels([month_labels[m - 1] for m in monthly["month"]])

    ax.set_title(f"{tag_name} | Monthly profile, reliable 15-min data", fontweight="bold")
    ax.set_xlabel("Month")
    ax.set_ylabel("Mean 15-min load")

    save_fig(out / "12_15min_monthly_profile_reliable.png")


def plot_15min_hourly(df15: pd.DataFrame, tag_name: str, out: Path) -> None:
    """Hourly profile from reliable 15-minute data."""
    tmp = df15.dropna(subset=["timestamp_15min", VALUE_COL]).copy()

    if tmp.empty:
        return

    tmp["hour"] = pd.to_datetime(tmp["timestamp_15min"]).dt.hour

    hourly = tmp.groupby("hour")[VALUE_COL].agg(["mean", "std"]).reset_index()

    fig, ax = plt.subplots(figsize=(13, 5))

    ax.fill_between(
        hourly["hour"],
        hourly["mean"] - hourly["std"],
        hourly["mean"] + hourly["std"],
        alpha=0.25,
        color=PALETTE["bar"],
        label="±1 std",
    )

    ax.plot(
        hourly["hour"],
        hourly["mean"],
        color=PALETTE["bar"],
        linewidth=2.5,
        marker="o",
        markersize=5,
        label="Mean",
    )

    ax.set_xticks(range(0, 24))
    ax.set_title(f"{tag_name} | Hourly profile, reliable 15-min data", fontweight="bold")
    ax.set_xlabel("Hour of day")
    ax.set_ylabel("Mean 15-min load")
    ax.legend()

    save_fig(out / "13_15min_hourly_profile_reliable.png")


def plot_reliable_vs_all(df15: pd.DataFrame, tag_name: str, out: Path) -> None:
    """Compare reliable vs all 15-minute intervals."""
    reliable = df15[
        df15["is_reliable"] == 1
    ].dropna(subset=["timestamp_15min", VALUE_COL])

    all_df = df15.dropna(subset=["timestamp_15min", VALUE_COL])

    if all_df.empty:
        return

    r_plot = _ds(reliable, 150_000, "timestamp_15min")
    a_plot = _ds(all_df, 150_000, "timestamp_15min")

    fig, ax = plt.subplots(figsize=(16, 5))

    ax.scatter(
        a_plot["timestamp_15min"],
        a_plot[VALUE_COL],
        s=3,
        alpha=0.3,
        color=PALETTE["warn"],
        edgecolors="none",
        label="All intervals",
    )

    if not r_plot.empty:
        ax.scatter(
            r_plot["timestamp_15min"],
            r_plot[VALUE_COL],
            s=3,
            alpha=0.5,
            color=PALETTE["cleaned"],
            edgecolors="none",
            label="Reliable intervals",
        )

    ax.set_title(f"{tag_name} | 15-min load: reliable vs all intervals", fontweight="bold")
    ax.set_xlabel("Time")
    ax.set_ylabel("Mean load")
    ax.legend(loc="upper right", fontsize=10, markerscale=4)

    save_fig(out / "14_reliable_vs_all_scatter.png")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 — Main pipeline per database
# ─────────────────────────────────────────────────────────────────────────────

def run_step2(db_path: str) -> None:
    db_tag = tag(db_path)
    out_dir = OUTPUT_DIR / db_tag
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'=' * 70}")
    print(f"  STEP 2 — {db_path}")
    print(f"{'=' * 70}")

    conn = sqlite3.connect(db_path)

    try:
        # ── 6.1 Read raw data ────────────────────────────────────────────────
        print("\n[1/6] Reading MODIFIED_DATASET ...")

        df_raw = read_table_chunked(conn, TABLE_SOURCE)

        if df_raw.empty:
            raise ValueError("Source table is empty.")

        for col in [TIMESTAMP_COL, VALUE_COL]:
            if col not in df_raw.columns:
                raise ValueError(f"Column '{col}' not found in source table.")

        df_raw[TIMESTAMP_COL] = df_raw[TIMESTAMP_COL].astype("string").str.strip()
        df_raw[VALUE_COL] = pd.to_numeric(df_raw[VALUE_COL], errors="coerce")
        df_raw["_ts_parsed"] = parse_timestamps(df_raw[TIMESTAMP_COL])

        rows_raw = len(df_raw)

        print(f"    Loaded {rows_raw:,} raw rows")

        # ── 6.2 Cleaning ─────────────────────────────────────────────────────
        print("[2/6] Cleaning data ...")

        mask_valid_ts = df_raw["_ts_parsed"].notna()
        removed_bad_ts = int((~mask_valid_ts).sum())

        mask_valid_val = df_raw[VALUE_COL].notna()
        removed_null_val = int((mask_valid_ts & ~mask_valid_val).sum())

        mask_non_negative = df_raw[VALUE_COL] >= 0
        removed_negative = int(
            (mask_valid_ts & mask_valid_val & ~mask_non_negative).sum()
        )

        df_clean_raw = df_raw[
            mask_valid_ts &
            mask_valid_val &
            mask_non_negative
        ].copy()

        before_dup = len(df_clean_raw)

        df_clean_raw = (
            df_clean_raw
            .groupby("_ts_parsed", as_index=False)
            .agg(
                **{
                    VALUE_COL: (VALUE_COL, "mean"),
                    TIMESTAMP_COL: (TIMESTAMP_COL, "first"),
                }
            )
        )

        removed_duplicates = before_dup - len(df_clean_raw)

        df_clean_raw = df_clean_raw.sort_values("_ts_parsed", ignore_index=True)

        # Compute real gaps in cleaned original data before resampling.
        df_clean_raw["delta_prev_minutes_raw"] = (
            df_clean_raw["_ts_parsed"]
            .diff()
            .dt.total_seconds()
            / 60.0
        )

        df_clean_raw["has_large_gap_before"] = (
            df_clean_raw["delta_prev_minutes_raw"] > GAP_THRESHOLD_MINUTES
        ).fillna(False).astype("int64")

        rows_cleaned = len(df_clean_raw)
        total_removed = rows_raw - rows_cleaned
        pct_removed = total_removed / max(rows_raw, 1) * 100

        print(f"    Removed bad timestamps   : {removed_bad_ts:,}")
        print(f"    Removed null values      : {removed_null_val:,}")
        print(f"    Removed negative values  : {removed_negative:,}")
        print(f"    Removed duplicates       : {removed_duplicates:,}")
        print(f"    Total removed            : {total_removed:,} ({pct_removed:.4f}%)")
        print(f"    Rows after cleaning      : {rows_cleaned:,}")

        # Build CLEANED_DATASET.
        df_cleaned = df_clean_raw[
            [
                TIMESTAMP_COL,
                "_ts_parsed",
                VALUE_COL,
                "delta_prev_minutes_raw",
                "has_large_gap_before",
            ]
        ].copy()

        df_cleaned.rename(
            columns={
                "_ts_parsed": "timestamp_parsed",
            },
            inplace=True,
        )

        df_cleaned = add_time_features(df_cleaned, "timestamp_parsed")

        # Convert timestamp to string for SQLite.
        df_cleaned["timestamp_parsed"] = pd.to_datetime(
            df_cleaned["timestamp_parsed"]
        ).astype(str)

        # ── 6.3 Save CLEANED_DATASET ─────────────────────────────────────────
        print("[3/6] Saving CLEANED_DATASET to SQLite ...")

        drop_and_create_table_from_df(conn, df_cleaned, TABLE_CLEANED)

        create_index(
            conn,
            TABLE_CLEANED,
            "timestamp_parsed",
            f"idx_{TABLE_CLEANED}_timestamp_parsed",
        )

        # ── 6.4 15-minute resampling ─────────────────────────────────────────
        print("[4/6] Resampling to 15-minute intervals ...")

        # Build real gap information from cleaned original data.
        gap_events = df_clean_raw.loc[
            df_clean_raw["has_large_gap_before"] == 1,
            ["_ts_parsed", "delta_prev_minutes_raw"],
        ].copy()

        if not gap_events.empty:
            gap_events["timestamp_15min"] = gap_events["_ts_parsed"].dt.floor(RESAMPLE_FREQ)

            gap_by_interval = (
                gap_events
                .groupby("timestamp_15min", as_index=False)
                .agg(
                    gap_event_count=("delta_prev_minutes_raw", "size"),
                    max_gap_before_minutes=("delta_prev_minutes_raw", "max"),
                )
            )
        else:
            gap_by_interval = pd.DataFrame(
                columns=[
                    "timestamp_15min",
                    "gap_event_count",
                    "max_gap_before_minutes",
                ]
            )

        # Build 1-minute series.
        ts_index = pd.to_datetime(df_clean_raw["_ts_parsed"])
        v_series = df_clean_raw.set_index(ts_index)[VALUE_COL].sort_index()

        # Resample to a regular 1-minute grid first.
        v_1min = v_series.resample("1min").mean()

        # Resample the regular 1-minute grid to 15 minutes.
        resampler_15 = v_1min.resample(RESAMPLE_FREQ)

        mean_15 = resampler_15.mean()
        count_15 = resampler_15.count()

        df_15min = pd.DataFrame(
            {
                "timestamp_15min": mean_15.index,
                VALUE_COL: mean_15.values,
                "sample_count": count_15.values,
            }
        )

        expected_samples = int(
            pd.Timedelta(RESAMPLE_FREQ) / pd.Timedelta("1min")
        )

        df_15min["expected_samples"] = expected_samples

        df_15min["coverage_ratio"] = (
            df_15min["sample_count"] / expected_samples
        )

        df_15min["is_empty_interval"] = (
            df_15min["sample_count"] == 0
        ).astype("int64")

        df_15min["is_partial_interval"] = (
            (df_15min["sample_count"] > 0) &
            (df_15min["sample_count"] < expected_samples)
        ).astype("int64")

        df_15min["is_complete_interval"] = (
            df_15min["sample_count"] == expected_samples
        ).astype("int64")

        df_15min["is_reliable"] = (
            df_15min["sample_count"] >= MIN_SAMPLES_PER_INTERVAL
        ).astype("int64")

        # This is only the regular grid delta, not a real data-gap indicator.
        df_15min["grid_delta_prev_minutes"] = (
            df_15min["timestamp_15min"]
            .diff()
            .dt.total_seconds()
            / 60.0
        )

        # Attach real gap information from the cleaned raw time series.
        df_15min = df_15min.merge(
            gap_by_interval,
            on="timestamp_15min",
            how="left",
        )

        df_15min["gap_event_count"] = (
            df_15min["gap_event_count"]
            .fillna(0)
            .astype("int64")
        )

        df_15min["max_gap_before_minutes"] = (
            df_15min["max_gap_before_minutes"]
            .fillna(0.0)
            .astype("float64")
        )

        # Meaningful gap flag.
        df_15min["is_gap"] = (
            df_15min["gap_event_count"] > 0
        ).astype("int64")

        # Add time features.
        df_15min = add_time_features(df_15min, "timestamp_15min")

        # Convert timestamp to string for SQLite.
        df_15min["timestamp_15min"] = pd.to_datetime(
            df_15min["timestamp_15min"]
        ).astype(str)

        rows_15min = len(df_15min)
        reliable_15 = int(df_15min["is_reliable"].sum())
        unreliable_15 = int((df_15min["is_reliable"] == 0).sum())
        empty_15 = int(df_15min["is_empty_interval"].sum())
        partial_15 = int(df_15min["is_partial_interval"].sum())
        complete_15 = int(df_15min["is_complete_interval"].sum())
        n_gaps_15 = int(df_15min["is_gap"].sum())

        df_15min_reliable = df_15min[
            df_15min["is_reliable"] == 1
        ].copy()

        print(f"    15-min intervals total     : {rows_15min:,}")
        print(f"    Expected samples / interval: {expected_samples}")
        print(f"    Reliable intervals         : {reliable_15:,}")
        print(f"    Unreliable intervals       : {unreliable_15:,}")
        print(f"    Empty intervals            : {empty_15:,}")
        print(f"    Partial intervals          : {partial_15:,}")
        print(f"    Complete intervals         : {complete_15:,}")
        print(f"    Real gap intervals         : {n_gaps_15:,}")

        # ── 6.5 Save DATASET_15MIN ───────────────────────────────────────────
        print("[5/6] Saving DATASET_15MIN to SQLite ...")

        drop_and_create_table_from_df(conn, df_15min, TABLE_15MIN)

        create_index(
            conn,
            TABLE_15MIN,
            "timestamp_15min",
            f"idx_{TABLE_15MIN}_timestamp_15min",
        )

        # ── 6.6 Save CSV reports ─────────────────────────────────────────────
        cleaning_report = {
            "dataset": db_tag,
            "rows_raw": rows_raw,
            "removed_bad_timestamps": removed_bad_ts,
            "removed_null_values": removed_null_val,
            "removed_negative_values": removed_negative,
            "removed_duplicates": removed_duplicates,
            "total_removed": total_removed,
            "pct_removed": pct_removed,
            "rows_after_cleaning": rows_cleaned,
            "large_gaps_in_cleaned_raw_sequence": int(df_clean_raw["has_large_gap_before"].sum()),
            "gap_threshold_minutes": GAP_THRESHOLD_MINUTES,
        }

        pd.Series(cleaning_report).to_frame("value").to_csv(
            out_dir / "cleaning_report.csv",
            header=True,
        )

        if len(df_15min_reliable) > 0:
            reliable_value_mean = float(df_15min_reliable[VALUE_COL].mean())
            reliable_value_std = float(df_15min_reliable[VALUE_COL].std())
            reliable_value_min = float(df_15min_reliable[VALUE_COL].min())
            reliable_value_max = float(df_15min_reliable[VALUE_COL].max())
        else:
            reliable_value_mean = np.nan
            reliable_value_std = np.nan
            reliable_value_min = np.nan
            reliable_value_max = np.nan

        resample_report = {
            "dataset": db_tag,
            "resample_frequency": RESAMPLE_FREQ,
            "expected_samples_per_interval": expected_samples,
            "min_samples_per_interval": MIN_SAMPLES_PER_INTERVAL,
            "reliability_rule": f"sample_count >= {MIN_SAMPLES_PER_INTERVAL}",
            "gap_threshold_minutes": GAP_THRESHOLD_MINUTES,
            "total_15min_intervals": rows_15min,
            "reliable_intervals": reliable_15,
            "unreliable_intervals": unreliable_15,
            "empty_intervals": empty_15,
            "partial_intervals": partial_15,
            "complete_intervals": complete_15,
            "pct_reliable": reliable_15 / max(rows_15min, 1) * 100,
            "pct_empty": empty_15 / max(rows_15min, 1) * 100,
            "pct_partial": partial_15 / max(rows_15min, 1) * 100,
            "pct_complete": complete_15 / max(rows_15min, 1) * 100,
            "real_gap_intervals_15min": n_gaps_15,
            "all_value_mean": float(df_15min[VALUE_COL].mean()),
            "all_value_std": float(df_15min[VALUE_COL].std()),
            "all_value_min": float(df_15min[VALUE_COL].min()),
            "all_value_max": float(df_15min[VALUE_COL].max()),
            "reliable_value_mean": reliable_value_mean,
            "reliable_value_std": reliable_value_std,
            "reliable_value_min": reliable_value_min,
            "reliable_value_max": reliable_value_max,
        }

        pd.Series(resample_report).to_frame("value").to_csv(
            out_dir / "resample_report.csv",
            header=True,
        )

        df_15min.head(500).to_csv(
            out_dir / "dataset_15min_sample.csv",
            index=False,
        )

        df_15min_reliable.head(500).to_csv(
            out_dir / "dataset_15min_reliable_sample.csv",
            index=False,
        )

        print(f"\n  ── Step 2 summary for {db_tag} ──")
        print(f"  Raw rows              : {rows_raw:,}")
        print(f"  Cleaned rows          : {rows_cleaned:,} ({pct_removed:.4f}% removed)")
        print(f"  15-min intervals      : {rows_15min:,}")
        print(f"  Reliable intervals    : {reliable_15:,} ({reliable_15 / max(rows_15min, 1) * 100:.2f}%)")
        print(f"  Unreliable intervals  : {unreliable_15:,}")
        print(f"  Empty intervals       : {empty_15:,}")
        print(f"  Partial intervals     : {partial_15:,}")
        print(f"  Complete intervals    : {complete_15:,}")
        print(f"  Real gap intervals    : {n_gaps_15:,}")
        print()

        # ── 6.7 Generate plots ───────────────────────────────────────────────
        print("[6/6] Generating plots ...")

        df_clean_plot = df_cleaned.copy()
        df_clean_plot["_ts_parsed"] = pd.to_datetime(
            df_clean_plot["timestamp_parsed"],
            errors="coerce",
        )

        df_raw_plot = df_raw.copy()

        df_15min_plot = df_15min.copy()
        df_15min_plot["timestamp_15min"] = pd.to_datetime(
            df_15min_plot["timestamp_15min"],
            errors="coerce",
        )

        df_15min_reliable_plot = df_15min_plot[
            df_15min_plot["is_reliable"] == 1
        ].copy()

        plot_raw_vs_cleaned(df_raw_plot, df_clean_plot, db_tag, out_dir)
        print("       01_raw_vs_cleaned.png                    ✓")

        plot_resampled_over_time(df_15min_reliable_plot, db_tag, out_dir)
        print("       02_resampled_15min_reliable.png          ✓")

        plot_sample_count_hist(df_15min_plot, db_tag, out_dir)
        print("       03_sample_count_hist.png                 ✓")

        plot_coverage_bar(df_15min_plot, db_tag, out_dir)
        print("       04_coverage_bar_by_year.png              ✓")

        plot_missing_heatmap(df_15min_plot, db_tag, out_dir)
        print("       05_missing_heatmap_hour_dow.png          ✓")

        plot_gap_heatmap(df_15min_plot, db_tag, out_dir)
        print("       06_gap_heatmap_hour_dow.png              ✓")

        plot_cleaned_monthly(df_clean_plot, db_tag, out_dir)
        print("       07_cleaned_monthly_profile.png           ✓")

        plot_cleaned_hourly(df_clean_plot, db_tag, out_dir)
        print("       08_cleaned_hourly_profile.png            ✓")

        plot_cleaned_heatmap(df_clean_plot, db_tag, out_dir)
        print("       09_cleaned_heatmap_hour_dow.png          ✓")

        plot_yearly_boxplot_cleaned(df_clean_plot, db_tag, out_dir)
        print("       10_cleaned_yearly_boxplot.png            ✓")

        plot_load_duration_curve(df_15min_reliable_plot, db_tag, out_dir)
        print("       11_load_duration_curve_reliable.png      ✓")

        plot_15min_monthly(df_15min_reliable_plot, db_tag, out_dir)
        print("       12_15min_monthly_profile_reliable.png    ✓")

        plot_15min_hourly(df_15min_reliable_plot, db_tag, out_dir)
        print("       13_15min_hourly_profile_reliable.png     ✓")

        plot_reliable_vs_all(df_15min_plot, db_tag, out_dir)
        print("       14_reliable_vs_all_scatter.png           ✓")

        print(f"\n  All outputs saved → {out_dir}\n")

    finally:
        conn.close()


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7 — Entry point
# ─────────────────────────────────────────────────────────────────────────────

def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    for db in DB_PATHS:
        if not os.path.isfile(db):
            print(f"\n[SKIP] File not found: {db}")
            continue

        try:
            run_step2(db)

        except Exception as exc:
            print(f"\n[ERROR] {db}: {exc}")
            raise

    print("\n" + "=" * 70)
    print("  Step 2 complete for all databases.")
    print("=" * 70)


if __name__ == "__main__":
    main()

In [ ]:
!pip install xgboost lightgbm joblib

In [ ]:
"""
STEP 3 — Forecasting Models for 15-Minute Load Data
====================================================

Reads DATASET_15MIN from each SQLite .db file and performs:

  1. Load reliable 15-minute intervals
  2. Build supervised time-series features:
       - calendar features
       - cyclical time features
       - lag features
       - rolling statistics
  3. Time-based train/test split
  4. Train and compare multiple forecasting models:
       Baselines:
         - Persistence 15-min
         - Daily Seasonal Naive
         - Weekly Seasonal Naive

       Machine Learning:
         - Linear Regression
         - Ridge Regression
         - ElasticNet
         - KNN
         - Linear SVR
         - RBF SVR
         - Random Forest
         - Extra Trees
         - Gradient Boosting
         - HistGradientBoosting
         - Neural Network / MLP
         - XGBoost, if installed
         - LightGBM, if installed

  5. Save:
       - metrics CSV
       - predictions CSV
       - best model predictions
       - trained model files
       - plots
       - SQLite tables:
           FORECAST_METRICS
           FORECAST_BEST_PREDICTIONS

Important:
  - Only DATASET_15MIN is used.
  - Only reliable intervals are used as valid target values.
  - Unreliable intervals are treated as missing for feature construction.
  - The model predicts the current 15-minute load using only past load values
    and calendar features.
  - No future target leakage is used.

Optional packages:
  pip install xgboost lightgbm joblib

All comments are in English.
"""

from __future__ import annotations

import os
import sqlite3
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR, LinearSVR
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, r2_score

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)


# ── Optional imports ─────────────────────────────────────────────────────────

try:
    import joblib
    HAS_JOBLIB = True
except ImportError:
    HAS_JOBLIB = False

try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False

try:
    from lightgbm import LGBMRegressor
    HAS_LIGHTGBM = True
except ImportError:
    HAS_LIGHTGBM = False


# Portable project paths for GitHub
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_BASE_DIR = PROJECT_ROOT / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)

# ═════════════════════════════════════════════════════════════════════════════
# USER CONFIGURATION
# ═════════════════════════════════════════════════════════════════════════════

DB_PATHS = [
    DATA_DIR / "campus_load.db",
]

TABLE_15MIN = "DATASET_15MIN"

TIMESTAMP_COL = "timestamp_15min"
VALUE_COL = "value"

OUTPUT_DIR = OUTPUT_BASE_DIR / "STEP3_FORECASTING_OUTPUTS"

# The dataset is 15-minute data.
FREQ = "15min"

# Forecasting horizon:
# horizon = 1 means predict current interval using previous intervals.
FORECAST_HORIZON = 1

# Test split:
# If a dataset-specific date is available and valid, it is used.
# Otherwise, the last TEST_SIZE_RATIO portion is used as the test set.
TEST_SIZE_RATIO = 0.20

# Dataset-specific test start date.
TEST_START_BY_DATASET = {
    "campus_load": "2023-01-01",
}

# Minimum rows required.
MIN_TRAIN_ROWS = 1000
MIN_TEST_ROWS = 200

# Save trained model files.
SAVE_MODELS = True

# Some models can be very slow on large datasets.
# For those models, only the most recent N training rows are used.
# None means use all training rows.
MODEL_TRAIN_LIMITS = {
    "LinearRegression": None,
    "Ridge": None,
    "ElasticNet": None,
    "KNN": 60_000,
    "LinearSVR": 80_000,
    "SVR_RBF": 20_000,
    "RandomForest": 120_000,
    "ExtraTrees": 120_000,
    "GradientBoosting": 80_000,
    "HistGradientBoosting": None,
    "NeuralNetwork_MLP": 80_000,
    "XGBoost": 150_000,
    "LightGBM": None,
}

RANDOM_STATE = 42
N_JOBS = -1

# Lag features in 15-minute steps.
# 1   = 15 minutes
# 4   = 1 hour
# 96  = 1 day
# 672 = 1 week
LAGS = {
    "lag_1": 1,
    "lag_2": 2,
    "lag_4": 4,
    "lag_8": 8,
    "lag_12": 12,
    "lag_24": 24,
    "lag_48": 48,
    "lag_96": 96,
    "lag_192": 192,
    "lag_672": 672,
}

ROLLING_WINDOWS = {
    "roll_4": 4,
    "roll_12": 12,
    "roll_24": 24,
    "roll_96": 96,
    "roll_672": 672,
}

# Minimum ratio of non-null values required inside rolling windows.
ROLLING_MIN_RATIO = 0.70

# Plot settings.
PLOT_DPI = 180

# ═════════════════════════════════════════════════════════════════════════════


# ── Plot aesthetics ───────────────────────────────────────────────────────────

sns.set_theme(style="whitegrid", context="talk", font_scale=1.00)

plt.rcParams.update(
    {
        "figure.dpi": 130,
        "savefig.dpi": PLOT_DPI,
        "axes.spines.right": False,
        "axes.spines.top": False,
        "figure.constrained_layout.use": True,
    }
)

PALETTE = {
    "main": "#2196F3",
    "actual": "#212121",
    "pred": "#E53935",
    "bar": "#5C6BC0",
    "best": "#43A047",
    "warn": "#FFC107",
    "neutral": "#78909C",
}


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — General helpers
# ─────────────────────────────────────────────────────────────────────────────

def qident(s: str) -> str:
    """Safely quote a SQLite identifier."""
    return '"' + s.replace('"', '""') + '"'


def tag(db_path: str) -> str:
    """Return database file name without extension."""
    return Path(db_path).stem


def save_fig(path: Path) -> None:
    """Save current matplotlib figure and close all figures."""
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, bbox_inches="tight")
    plt.close("all")


def parse_timestamp_column(s: pd.Series) -> pd.Series:
    """Parse timestamp column robustly."""
    try:
        return pd.to_datetime(s, errors="coerce", format="mixed")
    except TypeError:
        return pd.to_datetime(s, errors="coerce")


def safe_to_sql(df: pd.DataFrame, conn: sqlite3.Connection, table: str) -> None:
    """Save a DataFrame to SQLite safely."""
    out = df.copy()

    for col in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out[col]):
            out[col] = out[col].astype(str)

    out.to_sql(table, conn, if_exists="replace", index=False)
    conn.commit()


def make_output_dirs(db_tag: str) -> Dict[str, Path]:
    """Create output folder structure."""
    base = OUTPUT_DIR / db_tag
    dirs = {
        "base": base,
        "plots": base / "plots",
        "models": base / "models",
        "predictions": base / "predictions",
        "reports": base / "reports",
    }

    for p in dirs.values():
        p.mkdir(parents=True, exist_ok=True)

    return dirs


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — Metrics
# ─────────────────────────────────────────────────────────────────────────────

def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Root mean squared error."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Mean absolute percentage error.

    Zero targets are ignored because division by zero is invalid.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mask = y_true != 0

    if mask.sum() == 0:
        return np.nan

    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def smape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Symmetric mean absolute percentage error."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    denom = np.abs(y_true) + np.abs(y_pred)
    mask = denom != 0

    if mask.sum() == 0:
        return np.nan

    return float(np.mean(2.0 * np.abs(y_pred[mask] - y_true[mask]) / denom[mask]) * 100)


def evaluate_predictions(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    model_name: str,
    dataset_name: str,
    split_name: str,
) -> Dict:
    """Compute evaluation metrics."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mask = np.isfinite(y_true) & np.isfinite(y_pred)

    y_true = y_true[mask]
    y_pred = y_pred[mask]

    if len(y_true) == 0:
        return {
            "dataset": dataset_name,
            "split": split_name,
            "model": model_name,
            "n": 0,
            "MAE": np.nan,
            "RMSE": np.nan,
            "MAPE_percent": np.nan,
            "SMAPE_percent": np.nan,
            "R2": np.nan,
        }

    return {
        "dataset": dataset_name,
        "split": split_name,
        "model": model_name,
        "n": int(len(y_true)),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": rmse(y_true, y_pred),
        "MAPE_percent": mape(y_true, y_pred),
        "SMAPE_percent": smape(y_true, y_pred),
        "R2": float(r2_score(y_true, y_pred)) if len(y_true) > 1 else np.nan,
    }


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — Data loading and feature engineering
# ─────────────────────────────────────────────────────────────────────────────

def load_15min_dataset(conn: sqlite3.Connection) -> pd.DataFrame:
    """Load DATASET_15MIN from SQLite."""
    table_q = qident(TABLE_15MIN)

    df = pd.read_sql_query(
        f"""
        SELECT *
        FROM {table_q}
        ORDER BY {qident(TIMESTAMP_COL)};
        """,
        conn,
    )

    if df.empty:
        raise ValueError(f"Table {TABLE_15MIN} is empty.")

    if TIMESTAMP_COL not in df.columns:
        raise ValueError(f"Column '{TIMESTAMP_COL}' not found in {TABLE_15MIN}.")

    if VALUE_COL not in df.columns:
        raise ValueError(f"Column '{VALUE_COL}' not found in {TABLE_15MIN}.")

    df[TIMESTAMP_COL] = parse_timestamp_column(df[TIMESTAMP_COL])
    df[VALUE_COL] = pd.to_numeric(df[VALUE_COL], errors="coerce")

    if "is_reliable" not in df.columns:
        raise ValueError("Column 'is_reliable' not found. Run Step 2 first.")

    df["is_reliable"] = pd.to_numeric(df["is_reliable"], errors="coerce").fillna(0).astype(int)

    df = df.dropna(subset=[TIMESTAMP_COL]).sort_values(TIMESTAMP_COL).reset_index(drop=True)

    return df


def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add calendar and cyclical time features."""
    out = df.copy()

    ts = out[TIMESTAMP_COL]

    out["hour"] = ts.dt.hour.astype(int)
    out["minute"] = ts.dt.minute.astype(int)
    out["minute_of_day"] = (ts.dt.hour * 60 + ts.dt.minute).astype(int)
    out["day_of_week"] = ts.dt.dayofweek.astype(int)
    out["day_of_month"] = ts.dt.day.astype(int)
    out["month"] = ts.dt.month.astype(int)
    out["quarter"] = ts.dt.quarter.astype(int)
    out["day_of_year"] = ts.dt.dayofyear.astype(int)
    out["week_of_year"] = ts.dt.isocalendar().week.astype(int)
    out["is_weekend"] = (ts.dt.dayofweek >= 5).astype(int)

    # Cyclical encodings.
    out["minute_of_day_sin"] = np.sin(2 * np.pi * out["minute_of_day"] / 1440.0)
    out["minute_of_day_cos"] = np.cos(2 * np.pi * out["minute_of_day"] / 1440.0)

    out["day_of_week_sin"] = np.sin(2 * np.pi * out["day_of_week"] / 7.0)
    out["day_of_week_cos"] = np.cos(2 * np.pi * out["day_of_week"] / 7.0)

    out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12.0)
    out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12.0)

    out["day_of_year_sin"] = np.sin(2 * np.pi * out["day_of_year"] / 365.25)
    out["day_of_year_cos"] = np.cos(2 * np.pi * out["day_of_year"] / 365.25)

    return out


def build_supervised_dataset(df: pd.DataFrame, dataset_name: str) -> Tuple[pd.DataFrame, List[str]]:
    """
    Build supervised learning dataset.

    Target:
      y_current = reliable load value at timestamp t

    Features:
      - calendar features at t
      - lagged values before t
      - rolling statistics before t

    Unreliable intervals are treated as missing values for target and lag creation.
    """
    df = df.copy()
    df = df.dropna(subset=[TIMESTAMP_COL]).sort_values(TIMESTAMP_COL)

    start = df[TIMESTAMP_COL].min().floor(FREQ)
    end = df[TIMESTAMP_COL].max().ceil(FREQ)

    full_index = pd.date_range(start=start, end=end, freq=FREQ)

    grid = pd.DataFrame({TIMESTAMP_COL: full_index})

    keep_cols = [TIMESTAMP_COL, VALUE_COL, "is_reliable"]

    optional_cols = [
        "sample_count",
        "coverage_ratio",
        "is_empty_interval",
        "is_partial_interval",
        "is_complete_interval",
        "is_gap",
        "max_gap_before_minutes",
    ]

    for c in optional_cols:
        if c in df.columns:
            keep_cols.append(c)

    df_grid = grid.merge(
        df[keep_cols],
        on=TIMESTAMP_COL,
        how="left",
    )

    # Valid observed value for modeling.
    df_grid["value_model"] = np.where(
        (df_grid["is_reliable"] == 1) & df_grid[VALUE_COL].notna(),
        df_grid[VALUE_COL],
        np.nan,
    )

    df_grid = add_calendar_features(df_grid)

    # Lag features.
    for lag_name, lag_n in LAGS.items():
        df_grid[f"value_{lag_name}"] = df_grid["value_model"].shift(lag_n)

    # Rolling features based only on past data.
    shifted = df_grid["value_model"].shift(1)

    for roll_name, window in ROLLING_WINDOWS.items():
        min_periods = max(1, int(window * ROLLING_MIN_RATIO))

        df_grid[f"{roll_name}_mean"] = shifted.rolling(
            window=window,
            min_periods=min_periods,
        ).mean()

        df_grid[f"{roll_name}_std"] = shifted.rolling(
            window=window,
            min_periods=min_periods,
        ).std()

        df_grid[f"{roll_name}_min"] = shifted.rolling(
            window=window,
            min_periods=min_periods,
        ).min()

        df_grid[f"{roll_name}_max"] = shifted.rolling(
            window=window,
            min_periods=min_periods,
        ).max()

    # Difference features.
    df_grid["diff_lag_1_4"] = df_grid["value_lag_1"] - df_grid["value_lag_4"]
    df_grid["diff_lag_1_96"] = df_grid["value_lag_1"] - df_grid["value_lag_96"]
    df_grid["diff_lag_96_672"] = df_grid["value_lag_96"] - df_grid["value_lag_672"]

    # Target.
    # For horizon=1, predict current timestamp using only lagged past values.
    df_grid["target"] = df_grid["value_model"]

    calendar_features = [
        "hour",
        "minute",
        "minute_of_day",
        "day_of_week",
        "day_of_month",
        "month",
        "quarter",
        "day_of_year",
        "week_of_year",
        "is_weekend",
        "minute_of_day_sin",
        "minute_of_day_cos",
        "day_of_week_sin",
        "day_of_week_cos",
        "month_sin",
        "month_cos",
        "day_of_year_sin",
        "day_of_year_cos",
    ]

    lag_features = [f"value_{name}" for name in LAGS.keys()]

    rolling_features = []
    for roll_name in ROLLING_WINDOWS.keys():
        rolling_features.extend(
            [
                f"{roll_name}_mean",
                f"{roll_name}_std",
                f"{roll_name}_min",
                f"{roll_name}_max",
            ]
        )

    diff_features = [
        "diff_lag_1_4",
        "diff_lag_1_96",
        "diff_lag_96_672",
    ]

    feature_cols = calendar_features + lag_features + rolling_features + diff_features

    supervised = df_grid.dropna(subset=feature_cols + ["target"]).copy()
    supervised = supervised.sort_values(TIMESTAMP_COL).reset_index(drop=True)

    if supervised.empty:
        raise ValueError(f"No supervised rows available for {dataset_name}.")

    return supervised, feature_cols


def split_train_test(
    df: pd.DataFrame,
    dataset_name: str,
) -> Tuple[pd.DataFrame, pd.DataFrame, str]:
    """Time-based train/test split."""
    df = df.sort_values(TIMESTAMP_COL).reset_index(drop=True)

    test_start_str = TEST_START_BY_DATASET.get(dataset_name)
    split_note = ""

    if test_start_str is not None:
        test_start = pd.to_datetime(test_start_str)

        train_df = df[df[TIMESTAMP_COL] < test_start].copy()
        test_df = df[df[TIMESTAMP_COL] >= test_start].copy()

        if len(train_df) >= MIN_TRAIN_ROWS and len(test_df) >= MIN_TEST_ROWS:
            split_note = f"date_split_at_{test_start_str}"
            return train_df, test_df, split_note

    # Fallback: last TEST_SIZE_RATIO as test.
    split_idx = int(len(df) * (1.0 - TEST_SIZE_RATIO))

    train_df = df.iloc[:split_idx].copy()
    test_df = df.iloc[split_idx:].copy()

    split_note = f"ratio_split_last_{TEST_SIZE_RATIO:.0%}"

    if len(train_df) < MIN_TRAIN_ROWS:
        raise ValueError(f"Not enough training rows: {len(train_df)}")

    if len(test_df) < MIN_TEST_ROWS:
        raise ValueError(f"Not enough test rows: {len(test_df)}")

    return train_df, test_df, split_note


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — Models
# ─────────────────────────────────────────────────────────────────────────────

def get_model_registry() -> Tuple[Dict[str, object], List[str]]:
    """Create model registry."""
    models: Dict[str, object] = {}
    skipped: List[str] = []

    models["LinearRegression"] = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", LinearRegression()),
        ]
    )

    models["Ridge"] = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=5.0, random_state=RANDOM_STATE)),
        ]
    )

    models["ElasticNet"] = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "model",
                ElasticNet(
                    alpha=0.001,
                    l1_ratio=0.20,
                    max_iter=5000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

    models["KNN"] = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", KNeighborsRegressor(n_neighbors=10, weights="distance")),
        ]
    )

    models["LinearSVR"] = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "model",
                LinearSVR(
                    C=1.0,
                    epsilon=0.1,
                    max_iter=5000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

    models["SVR_RBF"] = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "model",
                SVR(
                    kernel="rbf",
                    C=20.0,
                    gamma="scale",
                    epsilon=0.1,
                ),
            ),
        ]
    )

    models["RandomForest"] = RandomForestRegressor(
        n_estimators=250,
        max_depth=None,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
    )

    models["ExtraTrees"] = ExtraTreesRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
    )

    models["GradientBoosting"] = GradientBoostingRegressor(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        random_state=RANDOM_STATE,
    )

    models["HistGradientBoosting"] = HistGradientBoostingRegressor(
        max_iter=400,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=0.01,
        random_state=RANDOM_STATE,
    )

    models["NeuralNetwork_MLP"] = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "model",
                MLPRegressor(
                    hidden_layer_sizes=(128, 64, 32),
                    activation="relu",
                    solver="adam",
                    alpha=0.0001,
                    batch_size=512,
                    learning_rate_init=0.001,
                    max_iter=300,
                    early_stopping=True,
                    validation_fraction=0.15,
                    n_iter_no_change=20,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

    if HAS_XGBOOST:
        models["XGBoost"] = XGBRegressor(
            n_estimators=600,
            learning_rate=0.04,
            max_depth=6,
            min_child_weight=3,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
            tree_method="hist",
        )
    else:
        skipped.append("XGBoost skipped because xgboost is not installed.")

    if HAS_LIGHTGBM:
        models["LightGBM"] = LGBMRegressor(
            n_estimators=800,
            learning_rate=0.035,
            num_leaves=64,
            max_depth=-1,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.05,
            reg_lambda=0.10,
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
        )
    else:
        skipped.append("LightGBM skipped because lightgbm is not installed.")

    return models, skipped


def limit_training_data(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    model_name: str,
) -> Tuple[pd.DataFrame, pd.Series]:
    """Limit training rows for slow models using the most recent rows."""
    limit = MODEL_TRAIN_LIMITS.get(model_name)

    if limit is None:
        return X_train, y_train

    if len(X_train) <= limit:
        return X_train, y_train

    return X_train.iloc[-limit:].copy(), y_train.iloc[-limit:].copy()


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 — Plots
# ─────────────────────────────────────────────────────────────────────────────

def plot_train_test_split(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    dataset_name: str,
    out_dir: Path,
) -> None:
    """Plot train/test split."""
    fig, ax = plt.subplots(figsize=(16, 5))

    ax.plot(
        train_df[TIMESTAMP_COL],
        train_df["target"],
        linewidth=0.7,
        alpha=0.8,
        label="Train",
        color=PALETTE["main"],
    )

    ax.plot(
        test_df[TIMESTAMP_COL],
        test_df["target"],
        linewidth=0.7,
        alpha=0.8,
        label="Test",
        color=PALETTE["warn"],
    )

    ax.set_title(f"{dataset_name} | Train/Test split", fontweight="bold")
    ax.set_xlabel("Time")
    ax.set_ylabel("Load")
    ax.legend()

    save_fig(out_dir / "01_train_test_split.png")


def plot_metrics_comparison(metrics_df: pd.DataFrame, dataset_name: str, out_dir: Path) -> None:
    """Plot RMSE and MAE comparison."""
    if metrics_df.empty:
        return

    tmp = metrics_df.sort_values("RMSE").copy()

    fig, ax = plt.subplots(figsize=(14, 7))

    sns.barplot(
        data=tmp,
        y="model",
        x="RMSE",
        ax=ax,
        color=PALETTE["bar"],
    )

    ax.set_title(f"{dataset_name} | Model comparison by RMSE", fontweight="bold")
    ax.set_xlabel("RMSE")
    ax.set_ylabel("Model")

    save_fig(out_dir / "02_model_comparison_rmse.png")

    fig, ax = plt.subplots(figsize=(14, 7))

    sns.barplot(
        data=tmp.sort_values("MAE"),
        y="model",
        x="MAE",
        ax=ax,
        color=PALETTE["best"],
    )

    ax.set_title(f"{dataset_name} | Model comparison by MAE", fontweight="bold")
    ax.set_xlabel("MAE")
    ax.set_ylabel("Model")

    save_fig(out_dir / "03_model_comparison_mae.png")


def plot_actual_vs_prediction(
    pred_df: pd.DataFrame,
    dataset_name: str,
    model_name: str,
    out_dir: Path,
) -> None:
    """Plot actual vs predicted for best model."""
    if pred_df.empty:
        return

    fig, ax = plt.subplots(figsize=(16, 5))

    ax.plot(
        pred_df[TIMESTAMP_COL],
        pred_df["y_true"],
        linewidth=0.8,
        color=PALETTE["actual"],
        label="Actual",
    )

    ax.plot(
        pred_df[TIMESTAMP_COL],
        pred_df["y_pred"],
        linewidth=0.8,
        color=PALETTE["pred"],
        alpha=0.85,
        label=f"Predicted — {model_name}",
    )

    ax.set_title(f"{dataset_name} | Actual vs predicted — {model_name}", fontweight="bold")
    ax.set_xlabel("Time")
    ax.set_ylabel("Load")
    ax.legend()

    save_fig(out_dir / "04_best_model_actual_vs_predicted.png")


def plot_actual_vs_prediction_zoom(
    pred_df: pd.DataFrame,
    dataset_name: str,
    model_name: str,
    out_dir: Path,
    days: int = 14,
) -> None:
    """Plot zoomed actual vs predicted."""
    if pred_df.empty:
        return

    start = pred_df[TIMESTAMP_COL].min()
    end = start + pd.Timedelta(days=days)

    zoom = pred_df[
        (pred_df[TIMESTAMP_COL] >= start) &
        (pred_df[TIMESTAMP_COL] < end)
    ].copy()

    if zoom.empty:
        return

    fig, ax = plt.subplots(figsize=(16, 5))

    ax.plot(
        zoom[TIMESTAMP_COL],
        zoom["y_true"],
        linewidth=1.0,
        color=PALETTE["actual"],
        label="Actual",
    )

    ax.plot(
        zoom[TIMESTAMP_COL],
        zoom["y_pred"],
        linewidth=1.0,
        color=PALETTE["pred"],
        alpha=0.85,
        label=f"Predicted — {model_name}",
    )

    ax.set_title(f"{dataset_name} | Zoomed forecast, first {days} test days — {model_name}", fontweight="bold")
    ax.set_xlabel("Time")
    ax.set_ylabel("Load")
    ax.legend()

    save_fig(out_dir / "05_best_model_zoom_first_days.png")


def plot_predicted_vs_actual_scatter(
    pred_df: pd.DataFrame,
    dataset_name: str,
    model_name: str,
    out_dir: Path,
) -> None:
    """Scatter plot predicted vs actual."""
    if pred_df.empty:
        return

    sample = pred_df.copy()

    if len(sample) > 80_000:
        sample = sample.sample(80_000, random_state=RANDOM_STATE)

    fig, ax = plt.subplots(figsize=(7, 7))

    ax.scatter(
        sample["y_true"],
        sample["y_pred"],
        s=8,
        alpha=0.35,
        color=PALETTE["main"],
        edgecolors="none",
    )

    min_v = min(sample["y_true"].min(), sample["y_pred"].min())
    max_v = max(sample["y_true"].max(), sample["y_pred"].max())

    ax.plot([min_v, max_v], [min_v, max_v], "--", color="black", linewidth=1)

    ax.set_title(f"{dataset_name} | Predicted vs actual — {model_name}", fontweight="bold")
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")

    save_fig(out_dir / "06_predicted_vs_actual_scatter.png")


def plot_residuals(
    pred_df: pd.DataFrame,
    dataset_name: str,
    model_name: str,
    out_dir: Path,
) -> None:
    """Plot residual distribution and residual over time."""
    if pred_df.empty:
        return

    pred_df = pred_df.copy()
    pred_df["residual"] = pred_df["y_true"] - pred_df["y_pred"]

    fig, ax = plt.subplots(figsize=(12, 5))

    sns.histplot(
        pred_df["residual"].dropna(),
        bins=100,
        kde=True,
        ax=ax,
        color=PALETTE["bar"],
    )

    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_title(f"{dataset_name} | Residual distribution — {model_name}", fontweight="bold")
    ax.set_xlabel("Residual = actual - predicted")
    ax.set_ylabel("Count")

    save_fig(out_dir / "07_residual_distribution.png")

    sample = pred_df.copy()

    if len(sample) > 100_000:
        sample = sample.sample(100_000, random_state=RANDOM_STATE).sort_values(TIMESTAMP_COL)

    fig, ax = plt.subplots(figsize=(16, 5))

    ax.scatter(
        sample[TIMESTAMP_COL],
        sample["residual"],
        s=4,
        alpha=0.35,
        color=PALETTE["neutral"],
        edgecolors="none",
    )

    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set_title(f"{dataset_name} | Residuals over time — {model_name}", fontweight="bold")
    ax.set_xlabel("Time")
    ax.set_ylabel("Residual")

    save_fig(out_dir / "08_residuals_over_time.png")


def plot_error_by_hour(
    pred_df: pd.DataFrame,
    dataset_name: str,
    model_name: str,
    out_dir: Path,
) -> None:
    """Plot MAE by hour."""
    if pred_df.empty:
        return

    tmp = pred_df.copy()
    tmp["hour"] = tmp[TIMESTAMP_COL].dt.hour
    tmp["abs_error"] = np.abs(tmp["y_true"] - tmp["y_pred"])

    hourly = tmp.groupby("hour")["abs_error"].mean().reset_index()

    fig, ax = plt.subplots(figsize=(12, 5))

    ax.bar(
        hourly["hour"],
        hourly["abs_error"],
        color=PALETTE["bar"],
        alpha=0.85,
    )

    ax.set_xticks(range(24))
    ax.set_title(f"{dataset_name} | MAE by hour — {model_name}", fontweight="bold")
    ax.set_xlabel("Hour of day")
    ax.set_ylabel("MAE")

    save_fig(out_dir / "09_mae_by_hour.png")


def plot_feature_importance(
    fitted_model: object,
    feature_cols: List[str],
    dataset_name: str,
    model_name: str,
    out_dir: Path,
) -> None:
    """Plot feature importance or coefficients if available."""
    model = fitted_model

    if isinstance(fitted_model, Pipeline):
        model = fitted_model.named_steps.get("model", fitted_model)

    importance = None

    if hasattr(model, "feature_importances_"):
        importance = np.asarray(model.feature_importances_, dtype=float)

    elif hasattr(model, "coef_"):
        coef = np.asarray(model.coef_).ravel()
        importance = np.abs(coef)

    if importance is None:
        return

    if len(importance) != len(feature_cols):
        return

    imp_df = pd.DataFrame(
        {
            "feature": feature_cols,
            "importance": importance,
        }
    ).sort_values("importance", ascending=False).head(25)

    fig, ax = plt.subplots(figsize=(12, 8))

    sns.barplot(
        data=imp_df,
        y="feature",
        x="importance",
        ax=ax,
        color=PALETTE["best"],
    )

    ax.set_title(f"{dataset_name} | Top feature importance — {model_name}", fontweight="bold")
    ax.set_xlabel("Importance")
    ax.set_ylabel("Feature")

    save_fig(out_dir / "10_feature_importance_best_model.png")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 — Forecasting pipeline per database
# ─────────────────────────────────────────────────────────────────────────────

def run_forecasting_for_db(db_path: str) -> None:
    dataset_name = tag(db_path)
    dirs = make_output_dirs(dataset_name)

    print(f"\n{'=' * 80}")
    print(f"  STEP 3 — Forecasting for {db_path}")
    print(f"{'=' * 80}")

    conn = sqlite3.connect(db_path)

    try:
        # ── Load and feature engineering ─────────────────────────────────────
        print("\n[1/8] Loading DATASET_15MIN ...")
        df15 = load_15min_dataset(conn)

        print(f"    Rows loaded              : {len(df15):,}")
        print(f"    Reliable intervals       : {int((df15['is_reliable'] == 1).sum()):,}")
        print(f"    Time range               : {df15[TIMESTAMP_COL].min()} → {df15[TIMESTAMP_COL].max()}")

        print("\n[2/8] Building supervised forecasting features ...")
        supervised, feature_cols = build_supervised_dataset(df15, dataset_name)

        print(f"    Supervised rows          : {len(supervised):,}")
        print(f"    Number of features       : {len(feature_cols):,}")

        train_df, test_df, split_note = split_train_test(supervised, dataset_name)

        print(f"    Split method             : {split_note}")
        print(f"    Train rows               : {len(train_df):,}")
        print(f"    Test rows                : {len(test_df):,}")
        print(f"    Train range              : {train_df[TIMESTAMP_COL].min()} → {train_df[TIMESTAMP_COL].max()}")
        print(f"    Test range               : {test_df[TIMESTAMP_COL].min()} → {test_df[TIMESTAMP_COL].max()}")

        X_train = train_df[feature_cols].copy()
        y_train = train_df["target"].copy()

        X_test = test_df[feature_cols].copy()
        y_test = test_df["target"].copy()

        # Save feature sample.
        supervised.head(1000).to_csv(
            dirs["reports"] / "supervised_features_sample.csv",
            index=False,
        )

        pd.Series(feature_cols, name="feature").to_csv(
            dirs["reports"] / "feature_columns.csv",
            index=False,
        )

        # ── Baseline models ──────────────────────────────────────────────────
        print("\n[3/8] Evaluating baseline models ...")

        all_metrics: List[Dict] = []
        prediction_frames: List[pd.DataFrame] = []

        baseline_defs = {
            "Baseline_Persistence_15min": "value_lag_1",
            "Baseline_SeasonalNaive_Daily": "value_lag_96",
            "Baseline_SeasonalNaive_Weekly": "value_lag_672",
        }

        for model_name, feature_name in baseline_defs.items():
            if feature_name not in test_df.columns:
                continue

            y_pred = test_df[feature_name].values

            metric = evaluate_predictions(
                y_true=y_test.values,
                y_pred=y_pred,
                model_name=model_name,
                dataset_name=dataset_name,
                split_name="test",
            )

            all_metrics.append(metric)

            pred_df = pd.DataFrame(
                {
                    "dataset": dataset_name,
                    "model": model_name,
                    TIMESTAMP_COL: test_df[TIMESTAMP_COL].values,
                    "y_true": y_test.values,
                    "y_pred": y_pred,
                }
            )

            prediction_frames.append(pred_df)

            print(
                f"    {model_name:35s} "
                f"MAE={metric['MAE']:.4f}  RMSE={metric['RMSE']:.4f}  "
                f"SMAPE={metric['SMAPE_percent']:.2f}%"
            )

        # ── ML models ────────────────────────────────────────────────────────
        print("\n[4/8] Training machine learning models ...")

        models, skipped_models = get_model_registry()

        best_model_name = None
        best_model_obj = None
        best_rmse = np.inf

        for skipped in skipped_models:
            print(f"    [SKIP] {skipped}")

        with open(dirs["reports"] / "skipped_models.txt", "w", encoding="utf-8") as f:
            for skipped in skipped_models:
                f.write(skipped + "\n")

        for model_name, model in models.items():
            print(f"\n    Training {model_name} ...")

            X_fit, y_fit = limit_training_data(X_train, y_train, model_name)

            print(f"      Fit rows: {len(X_fit):,}")

            try:
                model.fit(X_fit, y_fit)

                y_pred = model.predict(X_test)

                metric = evaluate_predictions(
                    y_true=y_test.values,
                    y_pred=y_pred,
                    model_name=model_name,
                    dataset_name=dataset_name,
                    split_name="test",
                )

                all_metrics.append(metric)

                pred_df = pd.DataFrame(
                    {
                        "dataset": dataset_name,
                        "model": model_name,
                        TIMESTAMP_COL: test_df[TIMESTAMP_COL].values,
                        "y_true": y_test.values,
                        "y_pred": y_pred,
                    }
                )

                prediction_frames.append(pred_df)

                print(
                    f"      MAE={metric['MAE']:.4f}  "
                    f"RMSE={metric['RMSE']:.4f}  "
                    f"SMAPE={metric['SMAPE_percent']:.2f}%  "
                    f"R2={metric['R2']:.4f}"
                )

                if np.isfinite(metric["RMSE"]) and metric["RMSE"] < best_rmse:
                    best_rmse = metric["RMSE"]
                    best_model_name = model_name
                    best_model_obj = model

                if SAVE_MODELS and HAS_JOBLIB:
                    joblib.dump(
                        model,
                        dirs["models"] / f"{model_name}.joblib",
                    )

            except Exception as exc:
                print(f"      [ERROR] {model_name} failed: {exc}")

                failed_path = dirs["reports"] / "failed_models.txt"
                with open(failed_path, "a", encoding="utf-8") as f:
                    f.write(f"{model_name}: {exc}\n")

        # ── Save metrics and predictions ─────────────────────────────────────
        print("\n[5/8] Saving metrics and predictions ...")

        metrics_df = pd.DataFrame(all_metrics)

        if metrics_df.empty:
            raise ValueError("No model metrics were generated.")

        metrics_df = metrics_df.sort_values("RMSE").reset_index(drop=True)

        metrics_df.to_csv(
            dirs["reports"] / "forecast_metrics.csv",
            index=False,
        )

        safe_to_sql(metrics_df, conn, "FORECAST_METRICS")

        all_predictions = pd.concat(prediction_frames, ignore_index=True)

        all_predictions.to_csv(
            dirs["predictions"] / "all_model_predictions.csv",
            index=False,
        )

        best_row = metrics_df.iloc[0]
        best_name_by_metric = best_row["model"]

        best_pred_df = all_predictions[
            all_predictions["model"] == best_name_by_metric
        ].copy()

        best_pred_df[TIMESTAMP_COL] = parse_timestamp_column(best_pred_df[TIMESTAMP_COL])

        best_pred_df["error"] = best_pred_df["y_true"] - best_pred_df["y_pred"]
        best_pred_df["abs_error"] = best_pred_df["error"].abs()

        best_pred_df.to_csv(
            dirs["predictions"] / "best_model_predictions.csv",
            index=False,
        )

        safe_to_sql(best_pred_df, conn, "FORECAST_BEST_PREDICTIONS")

        print("\n    Model ranking by RMSE:")
        print(metrics_df[["model", "MAE", "RMSE", "SMAPE_percent", "R2"]].to_string(index=False))

        print(f"\n    Best model by RMSE: {best_name_by_metric}")

        # If best model is a baseline, best_model_obj may be None.
        if best_model_obj is None or best_model_name != best_name_by_metric:
            best_model_obj = models.get(best_name_by_metric)
            best_model_name = best_name_by_metric

        # ── Plots ────────────────────────────────────────────────────────────
        print("\n[6/8] Generating plots ...")

        plot_train_test_split(train_df, test_df, dataset_name, dirs["plots"])
        print("    01_train_test_split.png                         ✓")

        plot_metrics_comparison(metrics_df, dataset_name, dirs["plots"])
        print("    02_model_comparison_rmse.png / 03_mae.png        ✓")

        plot_actual_vs_prediction(best_pred_df, dataset_name, best_name_by_metric, dirs["plots"])
        print("    04_best_model_actual_vs_predicted.png            ✓")

        plot_actual_vs_prediction_zoom(best_pred_df, dataset_name, best_name_by_metric, dirs["plots"], days=14)
        print("    05_best_model_zoom_first_days.png                ✓")

        plot_predicted_vs_actual_scatter(best_pred_df, dataset_name, best_name_by_metric, dirs["plots"])
        print("    06_predicted_vs_actual_scatter.png               ✓")

        plot_residuals(best_pred_df, dataset_name, best_name_by_metric, dirs["plots"])
        print("    07_residual_distribution.png / 08_residuals.png  ✓")

        plot_error_by_hour(best_pred_df, dataset_name, best_name_by_metric, dirs["plots"])
        print("    09_mae_by_hour.png                               ✓")

        if best_model_obj is not None:
            plot_feature_importance(
                fitted_model=best_model_obj,
                feature_cols=feature_cols,
                dataset_name=dataset_name,
                model_name=best_name_by_metric,
                out_dir=dirs["plots"],
            )
            print("    10_feature_importance_best_model.png             ✓")

        # ── Summary report ──────────────────────────────────────────────────
        print("\n[7/8] Saving text summary ...")

        summary_path = dirs["reports"] / "forecasting_summary.txt"

        with open(summary_path, "w", encoding="utf-8") as f:
            f.write(f"Forecasting summary for {dataset_name}\n")
            f.write("=" * 70 + "\n\n")
            f.write(f"Source table: {TABLE_15MIN}\n")
            f.write(f"Rows loaded: {len(df15):,}\n")
            f.write(f"Supervised rows: {len(supervised):,}\n")
            f.write(f"Number of features: {len(feature_cols):,}\n")
            f.write(f"Split method: {split_note}\n")
            f.write(f"Train rows: {len(train_df):,}\n")
            f.write(f"Test rows: {len(test_df):,}\n")
            f.write(f"Train range: {train_df[TIMESTAMP_COL].min()} -> {train_df[TIMESTAMP_COL].max()}\n")
            f.write(f"Test range: {test_df[TIMESTAMP_COL].min()} -> {test_df[TIMESTAMP_COL].max()}\n\n")
            f.write("Best model by RMSE:\n")
            f.write(str(best_row.to_dict()) + "\n\n")
            f.write("All metrics:\n")
            f.write(metrics_df.to_string(index=False))
            f.write("\n\nSkipped models:\n")
            for skipped in skipped_models:
                f.write(skipped + "\n")

        print(f"    Summary saved → {summary_path}")

        # ── Final console summary ───────────────────────────────────────────
        print("\n[8/8] Finished.")

        print(f"\n  ── STEP 3 summary for {dataset_name} ──")
        print(f"  Supervised rows       : {len(supervised):,}")
        print(f"  Features              : {len(feature_cols):,}")
        print(f"  Train rows            : {len(train_df):,}")
        print(f"  Test rows             : {len(test_df):,}")
        print(f"  Best model            : {best_name_by_metric}")
        print(f"  Best RMSE             : {best_row['RMSE']:.4f}")
        print(f"  Best MAE              : {best_row['MAE']:.4f}")
        print(f"  Best SMAPE            : {best_row['SMAPE_percent']:.2f}%")
        print(f"  Outputs saved         : {dirs['base']}")
        print()

    finally:
        conn.close()


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7 — Entry point
# ─────────────────────────────────────────────────────────────────────────────

def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    for db_path in DB_PATHS:
        if not os.path.isfile(db_path):
            print(f"\n[SKIP] File not found: {db_path}")
            continue

        run_forecasting_for_db(db_path)

    print("\n" + "=" * 80)
    print("  STEP 3 complete for all databases.")
    print("=" * 80)


if __name__ == "__main__":
    main()

In [ ]:
import joblib
from pathlib import Path

# Portable project paths for GitHub
PROJECT_ROOT = Path.cwd()
OUTPUT_BASE_DIR = PROJECT_ROOT / "outputs"


MODEL_PATHS = [
    OUTPUT_BASE_DIR / "STEP3_FORECASTING_OUTPUTS" / "campus_load" / "models" / "NeuralNetwork_MLP.joblib",
]

for path in MODEL_PATHS:
    print("\n" + "=" * 80)
    print(path)

    pipe = joblib.load(path)
    mlp = pipe.named_steps["model"]

    print(f"Actual epochs run (n_iter_): {mlp.n_iter_}")
    print(f"Max epochs allowed: {mlp.max_iter}")
    print(f"Early stopping: {mlp.early_stopping}")
    print(f"Validation fraction: {mlp.validation_fraction}")
    print(f"Best validation score: {getattr(mlp, 'best_validation_score_', None)}")
    print(f"Final training loss: {mlp.loss_}")

    if hasattr(mlp, "loss_curve_"):
        print(f"Loss curve length: {len(mlp.loss_curve_)}")

    if hasattr(mlp, "validation_scores_"):
        print(f"Validation score length: {len(mlp.validation_scores_)}")

In [ ]:
import joblib
from pathlib import Path

# Portable project paths for GitHub
PROJECT_ROOT = Path.cwd()
OUTPUT_BASE_DIR = PROJECT_ROOT / "outputs"

import matplotlib.pyplot as plt

MODEL_PATH = OUTPUT_BASE_DIR / "STEP3_FORECASTING_OUTPUTS" / "campus_load" / "models" / "NeuralNetwork_MLP.joblib"

pipe = joblib.load(MODEL_PATH)
mlp = pipe.named_steps["model"]

plt.figure(figsize=(10, 5))
plt.plot(mlp.loss_curve_, label="Training loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("NeuralNetwork_MLP | Training loss curve")
plt.legend()
plt.grid(True)
plt.show()

if hasattr(mlp, "validation_scores_"):
    plt.figure(figsize=(10, 5))
    plt.plot(mlp.validation_scores_, label="Validation score")
    plt.xlabel("Epoch")
    plt.ylabel("Validation score")
    plt.title("NeuralNetwork_MLP | Validation score curve")
    plt.legend()
    plt.grid(True)
    plt.show()

print("Epochs run:", mlp.n_iter_)
print("Final loss:", mlp.loss_)
print("Best validation score:", getattr(mlp, "best_validation_score_", None))

In [ ]:
import sys
import subprocess

print("Python executable:")
print(sys.executable)

print("\nChecking installed packages...")
packages = ["lightgbm", "xgboost", "joblib"]

for pkg in packages:
    print(f"\n--- {pkg} ---")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "show", pkg],
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print(result.stdout)
    else:
        print(f"{pkg} is NOT installed in this Python environment.")

In [ ]:
import sys

!{sys.executable} -m pip install lightgbm

In [ ]:
pip install joblib xgboost lightgbm shap tensorflow

In [ ]:
"""
STEP 4 LITE — Multi-Horizon Walk-Forward Forecasting, Tuning, and Error Analysis
================================================================================

This is a lighter and practical version of Step 4.

It reads DATASET_15MIN from each SQLite .db file and performs:

  1. Multi-horizon forecasting:
       - 15-minute ahead
       - 1-hour ahead
       - 24-hour ahead

  2. Leakage-safe feature engineering:
       - Calendar features at target time
       - Lag features from forecast origin and earlier
       - Rolling features from forecast origin and earlier
       - No future target leakage

  3. Walk-forward validation:
       - campus_load: dataset-specific or automatic quarter-based folds

  4. Lightweight model comparison:
       Baselines:
         - Persistence
         - Daily seasonal naive
         - Weekly seasonal naive

       Machine learning:
         - Ridge
         - ElasticNet
         - LinearSVR
         - HistGradientBoosting
         - XGBoost, if installed
         - LightGBM, if installed

  5. Lightweight tuning:
       - LightGBM random search, 4 trials
       - XGBoost random search, 4 trials

  6. Error analysis:
       - Peak vs non-peak hours
       - Weekday vs weekend
       - High-load vs normal-load periods
       - MAE by hour

  7. Saves:
       - CSV reports
       - prediction files
       - plots
       - SQLite tables:
           STEP4_LITE_WALK_FORWARD_METRICS
           STEP4_LITE_FINAL_METRICS
           STEP4_LITE_SEGMENT_METRICS
           STEP4_LITE_BEST_PREDICTIONS
           STEP4_LITE_TUNING_RESULTS

Important:
  - DATASET_15MIN must already exist from Step 2.
  - Only reliable intervals are valid target values.
  - Unreliable intervals are treated as missing.
  - Deep Learning and SHAP are intentionally disabled in this lite version.

All comments are in English.
"""

from __future__ import annotations

import os
import json
import sqlite3
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import LinearSVR
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import ParameterSampler

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)


# ═════════════════════════════════════════════════════════════════════════════
# Optional imports
# ═════════════════════════════════════════════════════════════════════════════

try:
    import joblib
    HAS_JOBLIB = True
except ImportError:
    HAS_JOBLIB = False

try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False

try:
    from lightgbm import LGBMRegressor
    HAS_LIGHTGBM = True
except ImportError:
    HAS_LIGHTGBM = False


# Portable project paths for GitHub
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_BASE_DIR = PROJECT_ROOT / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)

# ═════════════════════════════════════════════════════════════════════════════
# USER CONFIGURATION
# ═════════════════════════════════════════════════════════════════════════════

DB_PATHS = [
    DATA_DIR / "campus_load.db",
]

TABLE_15MIN = "DATASET_15MIN"

TIMESTAMP_COL = "timestamp_15min"
VALUE_COL = "value"

OUTPUT_DIR = OUTPUT_BASE_DIR / "STEP4_LITE_OUTPUTS"

FREQ = "15min"
FREQ_MINUTES = 15

# Forecast horizons in 15-minute steps.
HORIZONS = {
    "h15min": 1,     # 15 minutes ahead
    "h1hour": 4,     # 1 hour ahead
    "h24hour": 96,   # 24 hours ahead
}

HORIZON_LABELS = {
    "h15min": "15-minute ahead",
    "h1hour": "1-hour ahead",
    "h24hour": "24-hour ahead",
}

# Lags relative to forecast origin.
# Forecast origin = target timestamp - horizon.
# lag 0 = latest known value at forecast origin.
LAGS_AFTER_ORIGIN = {
    "origin_lag_0": 0,
    "origin_lag_1": 1,
    "origin_lag_2": 2,
    "origin_lag_4": 4,
    "origin_lag_8": 8,
    "origin_lag_12": 12,
    "origin_lag_24": 24,
    "origin_lag_48": 48,
    "origin_lag_96": 96,
    "origin_lag_192": 192,
    "origin_lag_672": 672,
}

ROLLING_WINDOWS = {
    "roll_4": 4,
    "roll_12": 12,
    "roll_24": 24,
    "roll_96": 96,
    "roll_672": 672,
}

ROLLING_MIN_RATIO = 0.70

# Walk-forward folds for this dataset.
WALK_FORWARD_TEST_YEARS_BY_DATASET = {
    "campus_load": [2022, 2023],
}

MIN_TRAIN_ROWS = 1000
MIN_TEST_ROWS = 200

# Peak-hour definition.
PEAK_HOURS = list(range(8, 18))  # 08:00 to 17:59

# High-load threshold based on training target percentile.
HIGH_LOAD_PERCENTILE = 0.90

# Lightweight tuning.
ENABLE_TUNING = True
N_TUNING_TRIALS = 4
TUNING_TRAIN_LIMIT = 80_000
TUNING_VALID_RATIO = 0.20

SAVE_MODELS = True

# Training limits for speed.
MODEL_TRAIN_LIMITS = {
    "Ridge": None,
    "ElasticNet": None,
    "LinearSVR": 60_000,
    "HistGradientBoosting": None,
    "XGBoost": 80_000,
    "LightGBM": None,
    "XGBoost_Tuned": 80_000,
    "LightGBM_Tuned": None,
}

RANDOM_STATE = 42
N_JOBS = -1
PLOT_DPI = 180


# ═════════════════════════════════════════════════════════════════════════════
# Plot aesthetics
# ═════════════════════════════════════════════════════════════════════════════

sns.set_theme(style="whitegrid", context="talk", font_scale=1.00)

plt.rcParams.update(
    {
        "figure.dpi": 130,
        "savefig.dpi": PLOT_DPI,
        "axes.spines.right": False,
        "axes.spines.top": False,
        "figure.constrained_layout.use": True,
    }
)

PALETTE = {
    "main": "#2196F3",
    "actual": "#212121",
    "pred": "#E53935",
    "bar": "#5C6BC0",
    "best": "#43A047",
    "warn": "#FFC107",
    "neutral": "#78909C",
}


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 1 — General helpers
# ═════════════════════════════════════════════════════════════════════════════

def qident(s: str) -> str:
    """Safely quote a SQLite identifier."""
    return '"' + s.replace('"', '""') + '"'


def tag(db_path: str) -> str:
    """Return database file name without extension."""
    return Path(db_path).stem


def short_horizon_label(horizon_name: str) -> str:
    """Return readable horizon label."""
    return HORIZON_LABELS.get(horizon_name, horizon_name)


def save_fig(path: Path) -> None:
    """Save current matplotlib figure and close all figures."""
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, bbox_inches="tight")
    plt.close("all")


def parse_timestamp_column(s: pd.Series) -> pd.Series:
    """Parse timestamp column robustly."""
    try:
        return pd.to_datetime(s, errors="coerce", format="mixed")
    except TypeError:
        return pd.to_datetime(s, errors="coerce")


def safe_to_sql(df: pd.DataFrame, conn: sqlite3.Connection, table: str) -> None:
    """Save a DataFrame to SQLite safely."""
    out = df.copy()

    for col in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out[col]):
            out[col] = out[col].astype(str)

    out.to_sql(table, conn, if_exists="replace", index=False)
    conn.commit()


def make_output_dirs(dataset_name: str) -> Dict[str, Path]:
    """Create output folder structure."""
    base = OUTPUT_DIR / dataset_name

    dirs = {
        "base": base,
        "plots": base / "plots",
        "models": base / "models",
        "predictions": base / "predictions",
        "reports": base / "reports",
        "tuning": base / "tuning",
    }

    for p in dirs.values():
        p.mkdir(parents=True, exist_ok=True)

    return dirs


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 2 — Metrics
# ═════════════════════════════════════════════════════════════════════════════

def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Root mean squared error."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Mean absolute percentage error, ignoring zero targets."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mask = y_true != 0

    if mask.sum() == 0:
        return np.nan

    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def smape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Symmetric mean absolute percentage error."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    denom = np.abs(y_true) + np.abs(y_pred)
    mask = denom != 0

    if mask.sum() == 0:
        return np.nan

    return float(np.mean(2.0 * np.abs(y_pred[mask] - y_true[mask]) / denom[mask]) * 100)


def evaluate_predictions(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    dataset_name: str,
    horizon_name: str,
    model_name: str,
    split_name: str,
    fold_name: str = "",
) -> Dict[str, Any]:
    """Compute prediction metrics."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mask = np.isfinite(y_true) & np.isfinite(y_pred)

    y_true = y_true[mask]
    y_pred = y_pred[mask]

    if len(y_true) == 0:
        return {
            "dataset": dataset_name,
            "horizon": horizon_name,
            "horizon_label": short_horizon_label(horizon_name),
            "split": split_name,
            "fold": fold_name,
            "model": model_name,
            "n": 0,
            "MAE": np.nan,
            "RMSE": np.nan,
            "MAPE_percent": np.nan,
            "SMAPE_percent": np.nan,
            "R2": np.nan,
        }

    return {
        "dataset": dataset_name,
        "horizon": horizon_name,
        "horizon_label": short_horizon_label(horizon_name),
        "split": split_name,
        "fold": fold_name,
        "model": model_name,
        "n": int(len(y_true)),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": rmse(y_true, y_pred),
        "MAPE_percent": mape(y_true, y_pred),
        "SMAPE_percent": smape(y_true, y_pred),
        "R2": float(r2_score(y_true, y_pred)) if len(y_true) > 1 else np.nan,
    }


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 3 — Data loading and leakage-safe feature engineering
# ═════════════════════════════════════════════════════════════════════════════

def load_15min_dataset(conn: sqlite3.Connection) -> pd.DataFrame:
    """Load DATASET_15MIN from SQLite."""
    table_q = qident(TABLE_15MIN)

    df = pd.read_sql_query(
        f"""
        SELECT *
        FROM {table_q}
        ORDER BY {qident(TIMESTAMP_COL)};
        """,
        conn,
    )

    if df.empty:
        raise ValueError(f"Table {TABLE_15MIN} is empty. Run Step 2 first.")

    required = [TIMESTAMP_COL, VALUE_COL, "is_reliable"]

    for col in required:
        if col not in df.columns:
            raise ValueError(f"Column '{col}' not found in {TABLE_15MIN}.")

    df[TIMESTAMP_COL] = parse_timestamp_column(df[TIMESTAMP_COL])
    df[VALUE_COL] = pd.to_numeric(df[VALUE_COL], errors="coerce")
    df["is_reliable"] = pd.to_numeric(
        df["is_reliable"],
        errors="coerce"
    ).fillna(0).astype(int)

    df = df.dropna(subset=[TIMESTAMP_COL])
    df = df.sort_values(TIMESTAMP_COL).reset_index(drop=True)

    return df


def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add calendar and cyclical time features at target timestamp."""
    out = df.copy()

    ts = out[TIMESTAMP_COL]

    out["hour"] = ts.dt.hour.astype(int)
    out["minute"] = ts.dt.minute.astype(int)
    out["minute_of_day"] = (ts.dt.hour * 60 + ts.dt.minute).astype(int)
    out["day_of_week"] = ts.dt.dayofweek.astype(int)
    out["day_of_month"] = ts.dt.day.astype(int)
    out["month"] = ts.dt.month.astype(int)
    out["quarter"] = ts.dt.quarter.astype(int)
    out["day_of_year"] = ts.dt.dayofyear.astype(int)
    out["week_of_year"] = ts.dt.isocalendar().week.astype(int)
    out["is_weekend"] = (ts.dt.dayofweek >= 5).astype(int)

    out["minute_of_day_sin"] = np.sin(2 * np.pi * out["minute_of_day"] / 1440.0)
    out["minute_of_day_cos"] = np.cos(2 * np.pi * out["minute_of_day"] / 1440.0)

    out["day_of_week_sin"] = np.sin(2 * np.pi * out["day_of_week"] / 7.0)
    out["day_of_week_cos"] = np.cos(2 * np.pi * out["day_of_week"] / 7.0)

    out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12.0)
    out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12.0)

    out["day_of_year_sin"] = np.sin(2 * np.pi * out["day_of_year"] / 365.25)
    out["day_of_year_cos"] = np.cos(2 * np.pi * out["day_of_year"] / 365.25)

    return out


def build_supervised_dataset(
    df15: pd.DataFrame,
    dataset_name: str,
    horizon_name: str,
    horizon_steps: int,
) -> Tuple[pd.DataFrame, List[str]]:
    """
    Build leakage-safe supervised dataset.

    For target timestamp t:
      target = load at t

    Forecast origin:
      origin = t - horizon_steps

    Features use only values available at origin or earlier.
    Calendar features at target time are allowed because future calendar is known.
    """
    df = df15.copy()
    df = df.dropna(subset=[TIMESTAMP_COL]).sort_values(TIMESTAMP_COL)

    start = df[TIMESTAMP_COL].min().floor(FREQ)
    end = df[TIMESTAMP_COL].max().ceil(FREQ)

    full_index = pd.date_range(start=start, end=end, freq=FREQ)
    grid = pd.DataFrame({TIMESTAMP_COL: full_index})

    keep_cols = [TIMESTAMP_COL, VALUE_COL, "is_reliable"]

    optional_cols = [
        "sample_count",
        "coverage_ratio",
        "is_empty_interval",
        "is_partial_interval",
        "is_complete_interval",
        "is_gap",
        "max_gap_before_minutes",
    ]

    for c in optional_cols:
        if c in df.columns:
            keep_cols.append(c)

    df_grid = grid.merge(df[keep_cols], on=TIMESTAMP_COL, how="left")

    # Reliable value only.
    df_grid["value_model"] = np.where(
        (df_grid["is_reliable"] == 1) & df_grid[VALUE_COL].notna(),
        df_grid[VALUE_COL],
        np.nan,
    )

    df_grid = add_calendar_features(df_grid)

    # Forecast origin timestamp.
    df_grid["forecast_origin_timestamp"] = (
        df_grid[TIMESTAMP_COL]
        - pd.to_timedelta(horizon_steps * FREQ_MINUTES, unit="m")
    )

    # Baselines.
    df_grid["baseline_persistence"] = df_grid["value_model"].shift(horizon_steps)
    df_grid["baseline_daily"] = df_grid["value_model"].shift(96)
    df_grid["baseline_weekly"] = df_grid["value_model"].shift(672)

    # Lag features relative to forecast origin.
    for lag_name, lag_after_origin in LAGS_AFTER_ORIGIN.items():
        actual_shift = horizon_steps + lag_after_origin
        df_grid[f"value_{lag_name}"] = df_grid["value_model"].shift(actual_shift)

    # Rolling features ending at forecast origin.
    origin_series = df_grid["value_model"].shift(horizon_steps)

    for roll_name, window in ROLLING_WINDOWS.items():
        min_periods = max(1, int(window * ROLLING_MIN_RATIO))

        df_grid[f"{roll_name}_mean"] = origin_series.rolling(
            window=window,
            min_periods=min_periods,
        ).mean()

        df_grid[f"{roll_name}_std"] = origin_series.rolling(
            window=window,
            min_periods=min_periods,
        ).std()

        df_grid[f"{roll_name}_min"] = origin_series.rolling(
            window=window,
            min_periods=min_periods,
        ).min()

        df_grid[f"{roll_name}_max"] = origin_series.rolling(
            window=window,
            min_periods=min_periods,
        ).max()

    # Difference features.
    df_grid["diff_origin_0_4"] = (
        df_grid["value_origin_lag_0"] - df_grid["value_origin_lag_4"]
    )

    df_grid["diff_origin_0_96"] = (
        df_grid["value_origin_lag_0"] - df_grid["value_origin_lag_96"]
    )

    df_grid["diff_origin_96_672"] = (
        df_grid["value_origin_lag_96"] - df_grid["value_origin_lag_672"]
    )

    # Target.
    df_grid["target"] = df_grid["value_model"]

    calendar_features = [
        "hour",
        "minute",
        "minute_of_day",
        "day_of_week",
        "day_of_month",
        "month",
        "quarter",
        "day_of_year",
        "week_of_year",
        "is_weekend",
        "minute_of_day_sin",
        "minute_of_day_cos",
        "day_of_week_sin",
        "day_of_week_cos",
        "month_sin",
        "month_cos",
        "day_of_year_sin",
        "day_of_year_cos",
    ]

    lag_features = [f"value_{name}" for name in LAGS_AFTER_ORIGIN.keys()]

    rolling_features = []
    for roll_name in ROLLING_WINDOWS.keys():
        rolling_features.extend(
            [
                f"{roll_name}_mean",
                f"{roll_name}_std",
                f"{roll_name}_min",
                f"{roll_name}_max",
            ]
        )

    diff_features = [
        "diff_origin_0_4",
        "diff_origin_0_96",
        "diff_origin_96_672",
    ]

    feature_cols = calendar_features + lag_features + rolling_features + diff_features

    supervised = df_grid.dropna(subset=feature_cols + ["target"]).copy()
    supervised = supervised.sort_values(TIMESTAMP_COL).reset_index(drop=True)

    supervised["dataset"] = dataset_name
    supervised["horizon"] = horizon_name
    supervised["horizon_steps"] = horizon_steps
    supervised["horizon_minutes"] = horizon_steps * FREQ_MINUTES

    if supervised.empty:
        raise ValueError(f"No supervised rows for {dataset_name} — {horizon_name}")

    return supervised, feature_cols


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 4 — Walk-forward folds
# ═════════════════════════════════════════════════════════════════════════════

def make_walk_forward_folds(
    supervised: pd.DataFrame,
    dataset_name: str,
) -> List[Dict[str, Any]]:
    """Create walk-forward validation folds."""
    df = supervised.sort_values(TIMESTAMP_COL).reset_index(drop=True)

    folds: List[Dict[str, Any]] = []

    configured_years = WALK_FORWARD_TEST_YEARS_BY_DATASET.get(dataset_name)

    if configured_years:
        for year in configured_years:
            test_start = pd.Timestamp(f"{year}-01-01")
            test_end = pd.Timestamp(f"{year + 1}-01-01")

            train_df = df[df[TIMESTAMP_COL] < test_start].copy()
            test_df = df[
                (df[TIMESTAMP_COL] >= test_start)
                & (df[TIMESTAMP_COL] < test_end)
            ].copy()

            if len(train_df) >= MIN_TRAIN_ROWS and len(test_df) >= MIN_TEST_ROWS:
                folds.append(
                    {
                        "fold": f"train_until_{year - 1}_test_{year}",
                        "train": train_df,
                        "test": test_df,
                    }
                )

    if folds:
        return folds

    # Quarter-based fallback for datasets without predefined test years.
    min_ts = df[TIMESTAMP_COL].min()
    max_ts = df[TIMESTAMP_COL].max()

    quarter_starts = pd.date_range(
        start=min_ts.to_period("Q").start_time,
        end=max_ts,
        freq="QS",
    )

    for q_start in quarter_starts[1:]:
        q_end = q_start + pd.DateOffset(months=3)

        train_df = df[df[TIMESTAMP_COL] < q_start].copy()
        test_df = df[
            (df[TIMESTAMP_COL] >= q_start)
            & (df[TIMESTAMP_COL] < q_end)
        ].copy()

        if len(train_df) >= MIN_TRAIN_ROWS and len(test_df) >= MIN_TEST_ROWS:
            folds.append(
                {
                    "fold": f"train_until_{q_start.date()}_test_next_quarter",
                    "train": train_df,
                    "test": test_df,
                }
            )

    if folds:
        return folds

    # Final fallback: last 20%.
    split_idx = int(len(df) * 0.80)

    train_df = df.iloc[:split_idx].copy()
    test_df = df.iloc[split_idx:].copy()

    if len(train_df) < MIN_TRAIN_ROWS or len(test_df) < MIN_TEST_ROWS:
        raise ValueError("Not enough data for train/test split.")

    folds.append(
        {
            "fold": "fallback_last_20_percent",
            "train": train_df,
            "test": test_df,
        }
    )

    return folds


def chronological_train_validation_split(
    X: pd.DataFrame,
    y: pd.Series,
    valid_ratio: float,
) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]:
    """Create chronological train/validation split."""
    n = len(X)
    split_idx = int(n * (1.0 - valid_ratio))
    split_idx = max(1, min(split_idx, n - 1))

    X_train = X.iloc[:split_idx].copy()
    y_train = y.iloc[:split_idx].copy()

    X_val = X.iloc[split_idx:].copy()
    y_val = y.iloc[split_idx:].copy()

    return X_train, y_train, X_val, y_val


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 5 — Models and tuning
# ═════════════════════════════════════════════════════════════════════════════

def get_model_registry(
    tuned_params: Optional[Dict[str, Dict[str, Any]]] = None,
    include_tuned: bool = False,
) -> Tuple[Dict[str, object], List[str]]:
    """Create lightweight model registry."""
    tuned_params = tuned_params or {}

    models: Dict[str, object] = {}
    skipped: List[str] = []

    models["Ridge"] = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=5.0, random_state=RANDOM_STATE)),
        ]
    )

    models["ElasticNet"] = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "model",
                ElasticNet(
                    alpha=0.001,
                    l1_ratio=0.20,
                    max_iter=5000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

    models["LinearSVR"] = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "model",
                LinearSVR(
                    C=1.0,
                    epsilon=0.1,
                    max_iter=5000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

    models["HistGradientBoosting"] = HistGradientBoostingRegressor(
        max_iter=400,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=0.01,
        random_state=RANDOM_STATE,
    )

    if HAS_XGBOOST:
        xgb_default = dict(
            n_estimators=600,
            learning_rate=0.04,
            max_depth=6,
            min_child_weight=3,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
            tree_method="hist",
        )

        models["XGBoost"] = XGBRegressor(**xgb_default)

        if include_tuned and "XGBoost" in tuned_params:
            tuned = xgb_default.copy()
            tuned.update(tuned_params["XGBoost"])
            models["XGBoost_Tuned"] = XGBRegressor(**tuned)
    else:
        skipped.append("XGBoost skipped because xgboost is not installed.")

    if HAS_LIGHTGBM:
        lgb_default = dict(
            n_estimators=800,
            learning_rate=0.035,
            num_leaves=64,
            max_depth=-1,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.05,
            reg_lambda=0.10,
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
            verbose=-1,
        )

        models["LightGBM"] = LGBMRegressor(**lgb_default)

        if include_tuned and "LightGBM" in tuned_params:
            tuned = lgb_default.copy()
            tuned.update(tuned_params["LightGBM"])
            models["LightGBM_Tuned"] = LGBMRegressor(**tuned)
    else:
        skipped.append("LightGBM skipped because lightgbm is not installed.")

    return models, skipped


def limit_training_data(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    model_name: str,
) -> Tuple[pd.DataFrame, pd.Series]:
    """Limit training rows for expensive models."""
    limit = MODEL_TRAIN_LIMITS.get(model_name)

    if limit is None:
        return X_train, y_train

    if len(X_train) <= limit:
        return X_train, y_train

    return X_train.iloc[-limit:].copy(), y_train.iloc[-limit:].copy()


def get_tuning_space(model_name: str) -> Dict[str, List[Any]]:
    """Return lightweight tuning search space."""
    if model_name == "LightGBM":
        return {
            "n_estimators": [500, 700, 900],
            "learning_rate": [0.025, 0.035, 0.05],
            "num_leaves": [31, 63, 96],
            "max_depth": [-1, 8, 12],
            "min_child_samples": [20, 40, 80],
            "subsample": [0.80, 0.90, 1.00],
            "colsample_bytree": [0.80, 0.90, 1.00],
            "reg_alpha": [0.0, 0.05, 0.10],
            "reg_lambda": [0.05, 0.10, 0.50],
        }

    if model_name == "XGBoost":
        return {
            "n_estimators": [500, 700, 900],
            "learning_rate": [0.025, 0.04, 0.06],
            "max_depth": [4, 6, 8],
            "min_child_weight": [1, 3, 5],
            "subsample": [0.80, 0.90, 1.00],
            "colsample_bytree": [0.80, 0.90, 1.00],
            "reg_alpha": [0.0, 0.05, 0.10],
            "reg_lambda": [0.5, 1.0, 2.0],
        }

    return {}


def build_tunable_model(model_name: str, params: Dict[str, Any]) -> object:
    """Build LightGBM or XGBoost with params."""
    if model_name == "LightGBM":
        base = dict(
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
            verbose=-1,
        )
        base.update(params)
        return LGBMRegressor(**base)

    if model_name == "XGBoost":
        base = dict(
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
            tree_method="hist",
        )
        base.update(params)
        return XGBRegressor(**base)

    raise ValueError(f"Unsupported tunable model: {model_name}")


def tune_model_random_search(
    model_name: str,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    dataset_name: str,
    horizon_name: str,
    out_dir: Path,
) -> Tuple[Optional[Dict[str, Any]], pd.DataFrame]:
    """Lightweight randomized tuning using chronological validation."""
    if not ENABLE_TUNING:
        return None, pd.DataFrame()

    if model_name == "LightGBM" and not HAS_LIGHTGBM:
        return None, pd.DataFrame()

    if model_name == "XGBoost" and not HAS_XGBOOST:
        return None, pd.DataFrame()

    X_inner_train, y_inner_train, X_val, y_val = chronological_train_validation_split(
        X_train,
        y_train,
        valid_ratio=TUNING_VALID_RATIO,
    )

    if len(X_inner_train) > TUNING_TRAIN_LIMIT:
        X_inner_train = X_inner_train.iloc[-TUNING_TRAIN_LIMIT:].copy()
        y_inner_train = y_inner_train.iloc[-TUNING_TRAIN_LIMIT:].copy()

    space = get_tuning_space(model_name)

    sampler = list(
        ParameterSampler(
            space,
            n_iter=N_TUNING_TRIALS,
            random_state=RANDOM_STATE,
        )
    )

    rows = []

    print(f"      Tuning {model_name} with {len(sampler)} trials...")

    for i, params in enumerate(sampler, start=1):
        try:
            model = build_tunable_model(model_name, params)
            model.fit(X_inner_train, y_inner_train)

            pred = model.predict(X_val)

            metric = evaluate_predictions(
                y_true=y_val.values,
                y_pred=pred,
                dataset_name=dataset_name,
                horizon_name=horizon_name,
                model_name=model_name,
                split_name="inner_validation",
                fold_name=f"trial_{i}",
            )

            row = {
                "dataset": dataset_name,
                "horizon": horizon_name,
                "model": model_name,
                "trial": i,
                "MAE": metric["MAE"],
                "RMSE": metric["RMSE"],
                "SMAPE_percent": metric["SMAPE_percent"],
                "R2": metric["R2"],
                "params_json": json.dumps(params),
            }

            rows.append(row)

            print(
                f"        trial {i:02d}: "
                f"RMSE={metric['RMSE']:.4f}, MAE={metric['MAE']:.4f}"
            )

        except Exception as exc:
            rows.append(
                {
                    "dataset": dataset_name,
                    "horizon": horizon_name,
                    "model": model_name,
                    "trial": i,
                    "MAE": np.nan,
                    "RMSE": np.nan,
                    "SMAPE_percent": np.nan,
                    "R2": np.nan,
                    "error": str(exc),
                    "params_json": json.dumps(params),
                }
            )
            print(f"        trial {i:02d}: failed — {exc}")

    results = pd.DataFrame(rows)

    if results.empty or results["RMSE"].dropna().empty:
        return None, results

    results = results.sort_values("RMSE").reset_index(drop=True)

    best_params = json.loads(results.iloc[0]["params_json"])

    out_dir.mkdir(parents=True, exist_ok=True)
    results.to_csv(
        out_dir / f"tuning_{horizon_name}_{model_name}.csv",
        index=False,
    )

    return best_params, results


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 6 — Baselines, training, and error analysis
# ═════════════════════════════════════════════════════════════════════════════

def evaluate_baselines(
    test_df: pd.DataFrame,
    dataset_name: str,
    horizon_name: str,
    split_name: str,
    fold_name: str,
) -> Tuple[List[Dict[str, Any]], List[pd.DataFrame]]:
    """Evaluate baseline models."""
    metrics: List[Dict[str, Any]] = []
    preds: List[pd.DataFrame] = []

    baselines = {
        "Baseline_Persistence": "baseline_persistence",
        "Baseline_SeasonalNaive_Daily": "baseline_daily",
        "Baseline_SeasonalNaive_Weekly": "baseline_weekly",
    }

    y_true = test_df["target"].values

    for model_name, col in baselines.items():
        if col not in test_df.columns:
            continue

        y_pred = test_df[col].values

        metric = evaluate_predictions(
            y_true=y_true,
            y_pred=y_pred,
            dataset_name=dataset_name,
            horizon_name=horizon_name,
            model_name=model_name,
            split_name=split_name,
            fold_name=fold_name,
        )

        metrics.append(metric)

        pred_df = pd.DataFrame(
            {
                "dataset": dataset_name,
                "horizon": horizon_name,
                "horizon_label": short_horizon_label(horizon_name),
                "split": split_name,
                "fold": fold_name,
                "model": model_name,
                TIMESTAMP_COL: test_df[TIMESTAMP_COL].values,
                "forecast_origin_timestamp": test_df["forecast_origin_timestamp"].values,
                "y_true": y_true,
                "y_pred": y_pred,
            }
        )

        preds.append(pred_df)

    return metrics, preds


def train_and_evaluate_models(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: List[str],
    dataset_name: str,
    horizon_name: str,
    split_name: str,
    fold_name: str,
    models: Dict[str, object],
    save_models_dir: Optional[Path] = None,
) -> Tuple[List[Dict[str, Any]], List[pd.DataFrame], Dict[str, object]]:
    """Train and evaluate models."""
    X_train = train_df[feature_cols].copy()
    y_train = train_df["target"].copy()

    X_test = test_df[feature_cols].copy()
    y_test = test_df["target"].copy()

    metrics: List[Dict[str, Any]] = []
    preds: List[pd.DataFrame] = []
    fitted_models: Dict[str, object] = {}

    for model_name, model in models.items():
        print(f"      Training {model_name}...")

        try:
            X_fit, y_fit = limit_training_data(X_train, y_train, model_name)

            model.fit(X_fit, y_fit)
            y_pred = model.predict(X_test)

            metric = evaluate_predictions(
                y_true=y_test.values,
                y_pred=y_pred,
                dataset_name=dataset_name,
                horizon_name=horizon_name,
                model_name=model_name,
                split_name=split_name,
                fold_name=fold_name,
            )

            metrics.append(metric)

            pred_df = pd.DataFrame(
                {
                    "dataset": dataset_name,
                    "horizon": horizon_name,
                    "horizon_label": short_horizon_label(horizon_name),
                    "split": split_name,
                    "fold": fold_name,
                    "model": model_name,
                    TIMESTAMP_COL: test_df[TIMESTAMP_COL].values,
                    "forecast_origin_timestamp": test_df["forecast_origin_timestamp"].values,
                    "y_true": y_test.values,
                    "y_pred": y_pred,
                }
            )

            preds.append(pred_df)
            fitted_models[model_name] = model

            print(
                f"        MAE={metric['MAE']:.4f}, "
                f"RMSE={metric['RMSE']:.4f}, "
                f"SMAPE={metric['SMAPE_percent']:.2f}%, "
                f"R2={metric['R2']:.4f}"
            )

            if SAVE_MODELS and HAS_JOBLIB and save_models_dir is not None:
                save_models_dir.mkdir(parents=True, exist_ok=True)
                joblib.dump(
                    model,
                    save_models_dir / f"{dataset_name}_{horizon_name}_{model_name}.joblib",
                )

        except Exception as exc:
            print(f"        [ERROR] {model_name} failed: {exc}")

    return metrics, preds, fitted_models


def compute_segment_metrics(
    pred_df: pd.DataFrame,
    dataset_name: str,
    horizon_name: str,
    model_name: str,
    high_load_threshold: float,
) -> pd.DataFrame:
    """Compute metrics by operational segments."""
    df = pred_df.copy()

    df[TIMESTAMP_COL] = parse_timestamp_column(df[TIMESTAMP_COL])
    df["hour"] = df[TIMESTAMP_COL].dt.hour
    df["day_of_week"] = df[TIMESTAMP_COL].dt.dayofweek

    df["abs_error"] = np.abs(df["y_true"] - df["y_pred"])
    df["squared_error"] = (df["y_true"] - df["y_pred"]) ** 2

    segment_specs = [
        ("peak_hours", df["hour"].isin(PEAK_HOURS)),
        ("non_peak_hours", ~df["hour"].isin(PEAK_HOURS)),
        ("weekday", df["day_of_week"] < 5),
        ("weekend", df["day_of_week"] >= 5),
        ("high_load", df["y_true"] >= high_load_threshold),
        ("normal_load", df["y_true"] < high_load_threshold),
    ]

    rows = []

    for segment_name, mask in segment_specs:
        sub = df[mask & df["y_true"].notna() & df["y_pred"].notna()].copy()

        if sub.empty:
            rows.append(
                {
                    "dataset": dataset_name,
                    "horizon": horizon_name,
                    "horizon_label": short_horizon_label(horizon_name),
                    "model": model_name,
                    "segment": segment_name,
                    "n": 0,
                    "MAE": np.nan,
                    "RMSE": np.nan,
                    "SMAPE_percent": np.nan,
                    "mean_actual": np.nan,
                    "mean_predicted": np.nan,
                }
            )
            continue

        rows.append(
            {
                "dataset": dataset_name,
                "horizon": horizon_name,
                "horizon_label": short_horizon_label(horizon_name),
                "model": model_name,
                "segment": segment_name,
                "n": int(len(sub)),
                "MAE": float(sub["abs_error"].mean()),
                "RMSE": float(np.sqrt(sub["squared_error"].mean())),
                "SMAPE_percent": smape(sub["y_true"].values, sub["y_pred"].values),
                "mean_actual": float(sub["y_true"].mean()),
                "mean_predicted": float(sub["y_pred"].mean()),
            }
        )

    return pd.DataFrame(rows)


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 7 — Plots
# ═════════════════════════════════════════════════════════════════════════════

def plot_walk_forward_heatmap(
    wf_metrics: pd.DataFrame,
    dataset_name: str,
    horizon_name: str,
    out_dir: Path,
) -> None:
    """Plot walk-forward RMSE heatmap."""
    tmp = wf_metrics[
        (wf_metrics["dataset"] == dataset_name)
        & (wf_metrics["horizon"] == horizon_name)
    ].copy()

    if tmp.empty:
        return

    pivot = tmp.pivot_table(
        values="RMSE",
        index="model",
        columns="fold",
        aggfunc="mean",
    )

    if pivot.empty:
        return

    order = pivot.mean(axis=1).sort_values().index
    pivot = pivot.loc[order]

    fig, ax = plt.subplots(figsize=(14, max(6, 0.45 * len(pivot))))

    sns.heatmap(
        pivot,
        annot=True,
        fmt=".2f",
        cmap="YlGnBu_r",
        ax=ax,
        cbar_kws={"label": "RMSE"},
    )

    ax.set_title(
        f"{dataset_name} | Walk-forward RMSE — {short_horizon_label(horizon_name)}",
        fontweight="bold",
    )
    ax.set_xlabel("Fold")
    ax.set_ylabel("Model")

    save_fig(out_dir / f"walk_forward_rmse_heatmap_{horizon_name}.png")


def plot_model_comparison(
    metrics_df: pd.DataFrame,
    dataset_name: str,
    horizon_name: str,
    out_dir: Path,
) -> None:
    """Plot final model comparison."""
    tmp = metrics_df[
        (metrics_df["dataset"] == dataset_name)
        & (metrics_df["horizon"] == horizon_name)
    ].copy()

    if tmp.empty:
        return

    tmp = tmp.sort_values("RMSE")

    fig, ax = plt.subplots(figsize=(14, max(6, 0.45 * len(tmp))))

    sns.barplot(
        data=tmp,
        y="model",
        x="RMSE",
        ax=ax,
        color=PALETTE["bar"],
    )

    ax.set_title(
        f"{dataset_name} | Final RMSE comparison — {short_horizon_label(horizon_name)}",
        fontweight="bold",
    )
    ax.set_xlabel("RMSE")
    ax.set_ylabel("Model")

    save_fig(out_dir / f"final_model_comparison_rmse_{horizon_name}.png")

    tmp_mae = tmp.sort_values("MAE")

    fig, ax = plt.subplots(figsize=(14, max(6, 0.45 * len(tmp_mae))))

    sns.barplot(
        data=tmp_mae,
        y="model",
        x="MAE",
        ax=ax,
        color=PALETTE["best"],
    )

    ax.set_title(
        f"{dataset_name} | Final MAE comparison — {short_horizon_label(horizon_name)}",
        fontweight="bold",
    )
    ax.set_xlabel("MAE")
    ax.set_ylabel("Model")

    save_fig(out_dir / f"final_model_comparison_mae_{horizon_name}.png")


def plot_best_by_horizon(
    final_metrics: pd.DataFrame,
    dataset_name: str,
    out_dir: Path,
) -> None:
    """Plot best RMSE and MAE per horizon."""
    tmp = final_metrics[final_metrics["dataset"] == dataset_name].copy()

    if tmp.empty:
        return

    best = tmp.sort_values("RMSE").groupby("horizon", as_index=False).first()
    best["horizon_label"] = best["horizon"].map(HORIZON_LABELS)

    fig, ax = plt.subplots(figsize=(12, 5))

    sns.barplot(
        data=best,
        x="horizon_label",
        y="RMSE",
        hue="model",
        dodge=False,
        ax=ax,
    )

    ax.set_title(f"{dataset_name} | Best RMSE by horizon", fontweight="bold")
    ax.set_xlabel("Forecast horizon")
    ax.set_ylabel("Best RMSE")
    ax.legend(title="Best model", fontsize=9)

    save_fig(out_dir / "best_rmse_by_horizon.png")

    fig, ax = plt.subplots(figsize=(12, 5))

    sns.barplot(
        data=best,
        x="horizon_label",
        y="MAE",
        hue="model",
        dodge=False,
        ax=ax,
    )

    ax.set_title(f"{dataset_name} | Best MAE by horizon", fontweight="bold")
    ax.set_xlabel("Forecast horizon")
    ax.set_ylabel("Best MAE")
    ax.legend(title="Best model", fontsize=9)

    save_fig(out_dir / "best_mae_by_horizon.png")


def plot_actual_vs_prediction(
    pred_df: pd.DataFrame,
    dataset_name: str,
    horizon_name: str,
    model_name: str,
    out_dir: Path,
) -> None:
    """Plot actual vs predicted."""
    if pred_df.empty:
        return

    tmp = pred_df.copy()
    tmp[TIMESTAMP_COL] = parse_timestamp_column(tmp[TIMESTAMP_COL])

    fig, ax = plt.subplots(figsize=(16, 5))

    ax.plot(
        tmp[TIMESTAMP_COL],
        tmp["y_true"],
        linewidth=0.8,
        color=PALETTE["actual"],
        label="Actual",
    )

    ax.plot(
        tmp[TIMESTAMP_COL],
        tmp["y_pred"],
        linewidth=0.8,
        alpha=0.85,
        color=PALETTE["pred"],
        label=f"Predicted — {model_name}",
    )

    ax.set_title(
        f"{dataset_name} | Actual vs predicted — {model_name} — {short_horizon_label(horizon_name)}",
        fontweight="bold",
    )
    ax.set_xlabel("Time")
    ax.set_ylabel("Load")
    ax.legend()

    save_fig(out_dir / f"best_actual_vs_predicted_{horizon_name}.png")


def plot_zoomed_prediction(
    pred_df: pd.DataFrame,
    dataset_name: str,
    horizon_name: str,
    model_name: str,
    out_dir: Path,
    days: int = 14,
) -> None:
    """Plot zoomed prediction."""
    if pred_df.empty:
        return

    tmp = pred_df.copy()
    tmp[TIMESTAMP_COL] = parse_timestamp_column(tmp[TIMESTAMP_COL])

    start = tmp[TIMESTAMP_COL].min()
    end = start + pd.Timedelta(days=days)

    zoom = tmp[
        (tmp[TIMESTAMP_COL] >= start)
        & (tmp[TIMESTAMP_COL] < end)
    ].copy()

    if zoom.empty:
        return

    fig, ax = plt.subplots(figsize=(16, 5))

    ax.plot(
        zoom[TIMESTAMP_COL],
        zoom["y_true"],
        linewidth=1.0,
        color=PALETTE["actual"],
        label="Actual",
    )

    ax.plot(
        zoom[TIMESTAMP_COL],
        zoom["y_pred"],
        linewidth=1.0,
        color=PALETTE["pred"],
        alpha=0.85,
        label=f"Predicted — {model_name}",
    )

    ax.set_title(
        f"{dataset_name} | First {days} test days — {model_name} — {short_horizon_label(horizon_name)}",
        fontweight="bold",
    )
    ax.set_xlabel("Time")
    ax.set_ylabel("Load")
    ax.legend()

    save_fig(out_dir / f"best_zoom_first_{days}_days_{horizon_name}.png")


def plot_predicted_vs_actual_scatter(
    pred_df: pd.DataFrame,
    dataset_name: str,
    horizon_name: str,
    model_name: str,
    out_dir: Path,
) -> None:
    """Scatter plot predicted vs actual."""
    if pred_df.empty:
        return

    sample = pred_df.copy()

    if len(sample) > 80_000:
        sample = sample.sample(80_000, random_state=RANDOM_STATE)

    fig, ax = plt.subplots(figsize=(7, 7))

    ax.scatter(
        sample["y_true"],
        sample["y_pred"],
        s=8,
        alpha=0.35,
        color=PALETTE["main"],
        edgecolors="none",
    )

    min_v = min(sample["y_true"].min(), sample["y_pred"].min())
    max_v = max(sample["y_true"].max(), sample["y_pred"].max())

    ax.plot([min_v, max_v], [min_v, max_v], "--", color="black", linewidth=1)

    ax.set_title(
        f"{dataset_name} | Predicted vs actual — {model_name} — {short_horizon_label(horizon_name)}",
        fontweight="bold",
    )
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")

    save_fig(out_dir / f"best_predicted_vs_actual_scatter_{horizon_name}.png")


def plot_residuals(
    pred_df: pd.DataFrame,
    dataset_name: str,
    horizon_name: str,
    model_name: str,
    out_dir: Path,
) -> None:
    """Plot residual distribution and residual over time."""
    if pred_df.empty:
        return

    tmp = pred_df.copy()
    tmp[TIMESTAMP_COL] = parse_timestamp_column(tmp[TIMESTAMP_COL])
    tmp["residual"] = tmp["y_true"] - tmp["y_pred"]

    fig, ax = plt.subplots(figsize=(12, 5))

    sns.histplot(
        tmp["residual"].dropna(),
        bins=100,
        kde=True,
        ax=ax,
        color=PALETTE["bar"],
    )

    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_title(
        f"{dataset_name} | Residual distribution — {model_name} — {short_horizon_label(horizon_name)}",
        fontweight="bold",
    )
    ax.set_xlabel("Residual = actual - predicted")
    ax.set_ylabel("Count")

    save_fig(out_dir / f"best_residual_distribution_{horizon_name}.png")

    sample = tmp.copy()

    if len(sample) > 100_000:
        sample = sample.sample(100_000, random_state=RANDOM_STATE).sort_values(TIMESTAMP_COL)

    fig, ax = plt.subplots(figsize=(16, 5))

    ax.scatter(
        sample[TIMESTAMP_COL],
        sample["residual"],
        s=4,
        alpha=0.35,
        color=PALETTE["neutral"],
        edgecolors="none",
    )

    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set_title(
        f"{dataset_name} | Residuals over time — {model_name} — {short_horizon_label(horizon_name)}",
        fontweight="bold",
    )
    ax.set_xlabel("Time")
    ax.set_ylabel("Residual")

    save_fig(out_dir / f"best_residuals_over_time_{horizon_name}.png")


def plot_mae_by_hour(
    pred_df: pd.DataFrame,
    dataset_name: str,
    horizon_name: str,
    model_name: str,
    out_dir: Path,
) -> None:
    """Plot MAE by hour."""
    if pred_df.empty:
        return

    tmp = pred_df.copy()
    tmp[TIMESTAMP_COL] = parse_timestamp_column(tmp[TIMESTAMP_COL])
    tmp["hour"] = tmp[TIMESTAMP_COL].dt.hour
    tmp["abs_error"] = np.abs(tmp["y_true"] - tmp["y_pred"])

    hourly = tmp.groupby("hour")["abs_error"].mean().reset_index()

    fig, ax = plt.subplots(figsize=(12, 5))

    ax.bar(
        hourly["hour"],
        hourly["abs_error"],
        color=PALETTE["bar"],
        alpha=0.85,
    )

    ax.set_xticks(range(24))
    ax.set_title(
        f"{dataset_name} | MAE by hour — {model_name} — {short_horizon_label(horizon_name)}",
        fontweight="bold",
    )
    ax.set_xlabel("Hour of day")
    ax.set_ylabel("MAE")

    save_fig(out_dir / f"best_mae_by_hour_{horizon_name}.png")


def plot_segment_metrics(
    segment_df: pd.DataFrame,
    dataset_name: str,
    horizon_name: str,
    model_name: str,
    out_dir: Path,
) -> None:
    """Plot segment MAE."""
    tmp = segment_df[
        (segment_df["dataset"] == dataset_name)
        & (segment_df["horizon"] == horizon_name)
        & (segment_df["model"] == model_name)
    ].copy()

    if tmp.empty:
        return

    fig, ax = plt.subplots(figsize=(12, 5))

    sns.barplot(
        data=tmp,
        x="segment",
        y="MAE",
        ax=ax,
        color=PALETTE["warn"],
    )

    ax.set_title(
        f"{dataset_name} | Segment MAE — {model_name} — {short_horizon_label(horizon_name)}",
        fontweight="bold",
    )
    ax.set_xlabel("Segment")
    ax.set_ylabel("MAE")
    plt.xticks(rotation=20, ha="right")

    save_fig(out_dir / f"best_segment_mae_{horizon_name}.png")


def plot_feature_importance(
    fitted_model: object,
    feature_cols: List[str],
    dataset_name: str,
    horizon_name: str,
    model_name: str,
    out_dir: Path,
) -> None:
    """Plot feature importance or coefficients if available."""
    model = fitted_model

    if isinstance(model, Pipeline):
        model = model.named_steps.get("model", model)

    importance = None

    if hasattr(model, "feature_importances_"):
        importance = np.asarray(model.feature_importances_, dtype=float)

    elif hasattr(model, "coef_"):
        coef = np.asarray(model.coef_).ravel()
        importance = np.abs(coef)

    if importance is None:
        return

    if len(importance) != len(feature_cols):
        return

    imp_df = pd.DataFrame(
        {
            "feature": feature_cols,
            "importance": importance,
        }
    ).sort_values("importance", ascending=False).head(25)

    fig, ax = plt.subplots(figsize=(12, 8))

    sns.barplot(
        data=imp_df,
        y="feature",
        x="importance",
        ax=ax,
        color=PALETTE["best"],
    )

    ax.set_title(
        f"{dataset_name} | Feature importance — {model_name} — {short_horizon_label(horizon_name)}",
        fontweight="bold",
    )
    ax.set_xlabel("Importance")
    ax.set_ylabel("Feature")

    save_fig(out_dir / f"best_feature_importance_{horizon_name}.png")


def plot_tuning_results(
    tuning_df: pd.DataFrame,
    dataset_name: str,
    out_dir: Path,
) -> None:
    """Plot tuning results."""
    if tuning_df.empty:
        return

    for (horizon_name, model_name), sub in tuning_df.groupby(["horizon", "model"]):
        sub = sub.dropna(subset=["RMSE"]).copy()

        if sub.empty:
            continue

        sub = sub.sort_values("trial")

        fig, ax = plt.subplots(figsize=(12, 5))

        ax.plot(
            sub["trial"],
            sub["RMSE"],
            marker="o",
            linewidth=1.5,
            color=PALETTE["bar"],
        )

        best_idx = sub["RMSE"].idxmin()
        best_row = sub.loc[best_idx]

        ax.scatter(
            [best_row["trial"]],
            [best_row["RMSE"]],
            s=120,
            color=PALETTE["best"],
            zorder=5,
            label=f"Best RMSE = {best_row['RMSE']:.4f}",
        )

        ax.set_title(
            f"{dataset_name} | Tuning RMSE — {model_name} — {short_horizon_label(horizon_name)}",
            fontweight="bold",
        )
        ax.set_xlabel("Trial")
        ax.set_ylabel("Validation RMSE")
        ax.legend()

        save_fig(out_dir / f"tuning_rmse_{horizon_name}_{model_name}.png")


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 8 — Core routines
# ═════════════════════════════════════════════════════════════════════════════

def run_walk_forward_for_horizon(
    supervised: pd.DataFrame,
    feature_cols: List[str],
    dataset_name: str,
    horizon_name: str,
    dirs: Dict[str, Path],
) -> pd.DataFrame:
    """Run lightweight walk-forward validation for one horizon."""
    print(f"\n    Walk-forward validation — {short_horizon_label(horizon_name)}")

    folds = make_walk_forward_folds(supervised, dataset_name)

    rows = []

    models, skipped = get_model_registry(include_tuned=False)

    for s in skipped:
        print(f"      [SKIP] {s}")

    for fold_obj in folds:
        fold_name = fold_obj["fold"]
        train_df = fold_obj["train"]
        test_df = fold_obj["test"]

        print(
            f"      Fold {fold_name}: "
            f"train={len(train_df):,}, test={len(test_df):,}"
        )

        baseline_metrics, _ = evaluate_baselines(
            test_df=test_df,
            dataset_name=dataset_name,
            horizon_name=horizon_name,
            split_name="walk_forward",
            fold_name=fold_name,
        )

        rows.extend(baseline_metrics)

        model_metrics, _, _ = train_and_evaluate_models(
            train_df=train_df,
            test_df=test_df,
            feature_cols=feature_cols,
            dataset_name=dataset_name,
            horizon_name=horizon_name,
            split_name="walk_forward",
            fold_name=fold_name,
            models=models,
            save_models_dir=None,
        )

        rows.extend(model_metrics)

    wf_metrics = pd.DataFrame(rows)

    if not wf_metrics.empty:
        wf_metrics.to_csv(
            dirs["reports"] / f"walk_forward_metrics_{horizon_name}.csv",
            index=False,
        )

        plot_walk_forward_heatmap(
            wf_metrics=wf_metrics,
            dataset_name=dataset_name,
            horizon_name=horizon_name,
            out_dir=dirs["plots"],
        )

    return wf_metrics


def run_final_evaluation_for_horizon(
    supervised: pd.DataFrame,
    feature_cols: List[str],
    dataset_name: str,
    horizon_name: str,
    dirs: Dict[str, Path],
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Run final evaluation, tuning, plots, and segment analysis."""
    print(f"\n    Final evaluation — {short_horizon_label(horizon_name)}")

    folds = make_walk_forward_folds(supervised, dataset_name)
    final_fold = folds[-1]

    train_df = final_fold["train"]
    test_df = final_fold["test"]
    fold_name = final_fold["fold"]

    print(
        f"      Final fold: {fold_name}, "
        f"train={len(train_df):,}, test={len(test_df):,}"
    )

    X_train = train_df[feature_cols].copy()
    y_train = train_df["target"].copy()

    tuned_params: Dict[str, Dict[str, Any]] = {}
    tuning_frames: List[pd.DataFrame] = []

    if ENABLE_TUNING:
        if HAS_LIGHTGBM:
            best_lgb_params, lgb_tuning = tune_model_random_search(
                model_name="LightGBM",
                X_train=X_train,
                y_train=y_train,
                dataset_name=dataset_name,
                horizon_name=horizon_name,
                out_dir=dirs["tuning"],
            )

            if best_lgb_params:
                tuned_params["LightGBM"] = best_lgb_params

            if not lgb_tuning.empty:
                tuning_frames.append(lgb_tuning)

        if HAS_XGBOOST:
            best_xgb_params, xgb_tuning = tune_model_random_search(
                model_name="XGBoost",
                X_train=X_train,
                y_train=y_train,
                dataset_name=dataset_name,
                horizon_name=horizon_name,
                out_dir=dirs["tuning"],
            )

            if best_xgb_params:
                tuned_params["XGBoost"] = best_xgb_params

            if not xgb_tuning.empty:
                tuning_frames.append(xgb_tuning)

    models, skipped = get_model_registry(
        tuned_params=tuned_params,
        include_tuned=True,
    )

    for s in skipped:
        print(f"      [SKIP] {s}")

    all_metrics: List[Dict[str, Any]] = []
    prediction_frames: List[pd.DataFrame] = []

    baseline_metrics, baseline_preds = evaluate_baselines(
        test_df=test_df,
        dataset_name=dataset_name,
        horizon_name=horizon_name,
        split_name="final_test",
        fold_name=fold_name,
    )

    all_metrics.extend(baseline_metrics)
    prediction_frames.extend(baseline_preds)

    model_metrics, model_preds, fitted_models = train_and_evaluate_models(
        train_df=train_df,
        test_df=test_df,
        feature_cols=feature_cols,
        dataset_name=dataset_name,
        horizon_name=horizon_name,
        split_name="final_test",
        fold_name=fold_name,
        models=models,
        save_models_dir=dirs["models"],
    )

    all_metrics.extend(model_metrics)
    prediction_frames.extend(model_preds)

    metrics_df = pd.DataFrame(all_metrics)

    if metrics_df.empty:
        raise ValueError(f"No final metrics for {dataset_name} — {horizon_name}")

    metrics_df = metrics_df.sort_values("RMSE").reset_index(drop=True)

    all_predictions = pd.concat(prediction_frames, ignore_index=True)

    best_row = metrics_df.iloc[0]
    best_model_name = str(best_row["model"])

    best_pred_df = all_predictions[
        all_predictions["model"] == best_model_name
    ].copy()

    best_pred_df[TIMESTAMP_COL] = parse_timestamp_column(best_pred_df[TIMESTAMP_COL])
    best_pred_df["error"] = best_pred_df["y_true"] - best_pred_df["y_pred"]
    best_pred_df["abs_error"] = best_pred_df["error"].abs()

    high_load_threshold = float(train_df["target"].quantile(HIGH_LOAD_PERCENTILE))

    segment_frames = []

    for model_name, pred_df in all_predictions.groupby("model"):
        segment_frames.append(
            compute_segment_metrics(
                pred_df=pred_df,
                dataset_name=dataset_name,
                horizon_name=horizon_name,
                model_name=model_name,
                high_load_threshold=high_load_threshold,
            )
        )

    segment_df = pd.concat(segment_frames, ignore_index=True)

    tuning_df = (
        pd.concat(tuning_frames, ignore_index=True)
        if tuning_frames
        else pd.DataFrame()
    )

    # Save reports.
    metrics_df.to_csv(
        dirs["reports"] / f"final_metrics_{horizon_name}.csv",
        index=False,
    )

    all_predictions.to_csv(
        dirs["predictions"] / f"final_all_predictions_{horizon_name}.csv",
        index=False,
    )

    best_pred_df.to_csv(
        dirs["predictions"] / f"final_best_predictions_{horizon_name}.csv",
        index=False,
    )

    segment_df.to_csv(
        dirs["reports"] / f"segment_metrics_{horizon_name}.csv",
        index=False,
    )

    if not tuning_df.empty:
        tuning_df.to_csv(
            dirs["tuning"] / f"tuning_results_{horizon_name}.csv",
            index=False,
        )

    # Plots.
    plot_model_comparison(
        metrics_df=metrics_df,
        dataset_name=dataset_name,
        horizon_name=horizon_name,
        out_dir=dirs["plots"],
    )

    plot_actual_vs_prediction(
        pred_df=best_pred_df,
        dataset_name=dataset_name,
        horizon_name=horizon_name,
        model_name=best_model_name,
        out_dir=dirs["plots"],
    )

    plot_zoomed_prediction(
        pred_df=best_pred_df,
        dataset_name=dataset_name,
        horizon_name=horizon_name,
        model_name=best_model_name,
        out_dir=dirs["plots"],
        days=14,
    )

    plot_predicted_vs_actual_scatter(
        pred_df=best_pred_df,
        dataset_name=dataset_name,
        horizon_name=horizon_name,
        model_name=best_model_name,
        out_dir=dirs["plots"],
    )

    plot_residuals(
        pred_df=best_pred_df,
        dataset_name=dataset_name,
        horizon_name=horizon_name,
        model_name=best_model_name,
        out_dir=dirs["plots"],
    )

    plot_mae_by_hour(
        pred_df=best_pred_df,
        dataset_name=dataset_name,
        horizon_name=horizon_name,
        model_name=best_model_name,
        out_dir=dirs["plots"],
    )

    plot_segment_metrics(
        segment_df=segment_df,
        dataset_name=dataset_name,
        horizon_name=horizon_name,
        model_name=best_model_name,
        out_dir=dirs["plots"],
    )

    best_model_obj = fitted_models.get(best_model_name)

    if best_model_obj is not None:
        plot_feature_importance(
            fitted_model=best_model_obj,
            feature_cols=feature_cols,
            dataset_name=dataset_name,
            horizon_name=horizon_name,
            model_name=best_model_name,
            out_dir=dirs["plots"],
        )

    print(
        f"      Best model for {short_horizon_label(horizon_name)}: "
        f"{best_model_name} | "
        f"RMSE={best_row['RMSE']:.4f}, "
        f"MAE={best_row['MAE']:.4f}, "
        f"SMAPE={best_row['SMAPE_percent']:.2f}%"
    )

    return metrics_df, best_pred_df, segment_df, tuning_df


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 9 — Main pipeline per database
# ═════════════════════════════════════════════════════════════════════════════

def run_step4_lite_for_db(db_path: str) -> None:
    """Run Step 4 Lite for one database."""
    dataset_name = tag(db_path)
    dirs = make_output_dirs(dataset_name)

    print(f"\n{'=' * 90}")
    print(f"  STEP 4 LITE — {db_path}")
    print(f"{'=' * 90}")

    conn = sqlite3.connect(db_path)

    try:
        print("\n[1/6] Loading DATASET_15MIN ...")
        df15 = load_15min_dataset(conn)

        print(f"    Rows loaded        : {len(df15):,}")
        print(f"    Reliable intervals : {int((df15['is_reliable'] == 1).sum()):,}")
        print(f"    Time range         : {df15[TIMESTAMP_COL].min()} → {df15[TIMESTAMP_COL].max()}")

        all_wf_metrics: List[pd.DataFrame] = []
        all_final_metrics: List[pd.DataFrame] = []
        all_best_predictions: List[pd.DataFrame] = []
        all_segment_metrics: List[pd.DataFrame] = []
        all_tuning_results: List[pd.DataFrame] = []

        print("\n[2/6] Running horizons ...")

        for horizon_name, horizon_steps in HORIZONS.items():
            print(f"\n  {'-' * 80}")
            print(f"  Horizon: {short_horizon_label(horizon_name)} ({horizon_steps} steps)")
            print(f"  {'-' * 80}")

            print("    Building supervised dataset ...")

            supervised, feature_cols = build_supervised_dataset(
                df15=df15,
                dataset_name=dataset_name,
                horizon_name=horizon_name,
                horizon_steps=horizon_steps,
            )

            print(f"    Supervised rows    : {len(supervised):,}")
            print(f"    Number of features : {len(feature_cols):,}")
            print(f"    Target range       : {supervised[TIMESTAMP_COL].min()} → {supervised[TIMESTAMP_COL].max()}")

            supervised.head(1000).to_csv(
                dirs["reports"] / f"supervised_sample_{horizon_name}.csv",
                index=False,
            )

            pd.Series(feature_cols, name="feature").to_csv(
                dirs["reports"] / f"feature_columns_{horizon_name}.csv",
                index=False,
            )

            print("\n[3/6] Walk-forward validation ...")

            wf_metrics = run_walk_forward_for_horizon(
                supervised=supervised,
                feature_cols=feature_cols,
                dataset_name=dataset_name,
                horizon_name=horizon_name,
                dirs=dirs,
            )

            if not wf_metrics.empty:
                all_wf_metrics.append(wf_metrics)

            print("\n[4/6] Final evaluation and tuning ...")

            final_metrics, best_pred, segment_metrics, tuning_results = run_final_evaluation_for_horizon(
                supervised=supervised,
                feature_cols=feature_cols,
                dataset_name=dataset_name,
                horizon_name=horizon_name,
                dirs=dirs,
            )

            all_final_metrics.append(final_metrics)
            all_best_predictions.append(best_pred)
            all_segment_metrics.append(segment_metrics)

            if not tuning_results.empty:
                all_tuning_results.append(tuning_results)

        print("\n[5/6] Saving combined reports and SQLite tables ...")

        combined_wf = (
            pd.concat(all_wf_metrics, ignore_index=True)
            if all_wf_metrics
            else pd.DataFrame()
        )

        combined_final = (
            pd.concat(all_final_metrics, ignore_index=True)
            if all_final_metrics
            else pd.DataFrame()
        )

        combined_best_pred = (
            pd.concat(all_best_predictions, ignore_index=True)
            if all_best_predictions
            else pd.DataFrame()
        )

        combined_segments = (
            pd.concat(all_segment_metrics, ignore_index=True)
            if all_segment_metrics
            else pd.DataFrame()
        )

        combined_tuning = (
            pd.concat(all_tuning_results, ignore_index=True)
            if all_tuning_results
            else pd.DataFrame()
        )

        if not combined_wf.empty:
            combined_wf = combined_wf.sort_values(
                ["horizon", "fold", "RMSE"]
            ).reset_index(drop=True)

            combined_wf.to_csv(
                dirs["reports"] / "STEP4_LITE_walk_forward_metrics_all.csv",
                index=False,
            )

            safe_to_sql(
                combined_wf,
                conn,
                "STEP4_LITE_WALK_FORWARD_METRICS",
            )

        if not combined_final.empty:
            combined_final = combined_final.sort_values(
                ["horizon", "RMSE"]
            ).reset_index(drop=True)

            combined_final.to_csv(
                dirs["reports"] / "STEP4_LITE_final_metrics_all.csv",
                index=False,
            )

            safe_to_sql(
                combined_final,
                conn,
                "STEP4_LITE_FINAL_METRICS",
            )

        if not combined_best_pred.empty:
            combined_best_pred.to_csv(
                dirs["predictions"] / "STEP4_LITE_best_predictions_all_horizons.csv",
                index=False,
            )

            safe_to_sql(
                combined_best_pred,
                conn,
                "STEP4_LITE_BEST_PREDICTIONS",
            )

        if not combined_segments.empty:
            combined_segments.to_csv(
                dirs["reports"] / "STEP4_LITE_segment_metrics_all.csv",
                index=False,
            )

            safe_to_sql(
                combined_segments,
                conn,
                "STEP4_LITE_SEGMENT_METRICS",
            )

        if not combined_tuning.empty:
            combined_tuning.to_csv(
                dirs["tuning"] / "STEP4_LITE_tuning_results_all.csv",
                index=False,
            )

            safe_to_sql(
                combined_tuning,
                conn,
                "STEP4_LITE_TUNING_RESULTS",
            )

        print("\n[6/6] Generating combined plots and summary ...")

        if not combined_final.empty:
            plot_best_by_horizon(
                final_metrics=combined_final,
                dataset_name=dataset_name,
                out_dir=dirs["plots"],
            )

        if not combined_tuning.empty:
            plot_tuning_results(
                tuning_df=combined_tuning,
                dataset_name=dataset_name,
                out_dir=dirs["plots"],
            )

        summary_path = dirs["reports"] / "STEP4_LITE_summary.txt"

        with open(summary_path, "w", encoding="utf-8") as f:
            f.write(f"STEP 4 LITE Summary — {dataset_name}\n")
            f.write("=" * 80 + "\n\n")

            f.write(f"Source table: {TABLE_15MIN}\n")
            f.write(f"Rows loaded: {len(df15):,}\n")
            f.write(f"Reliable intervals: {int((df15['is_reliable'] == 1).sum()):,}\n")
            f.write(f"Time range: {df15[TIMESTAMP_COL].min()} -> {df15[TIMESTAMP_COL].max()}\n\n")

            f.write("Horizons:\n")
            for h, steps in HORIZONS.items():
                f.write(f"  - {h}: {short_horizon_label(h)} ({steps} steps)\n")

            f.write("\nPackage status:\n")
            f.write(f"  joblib: {HAS_JOBLIB}\n")
            f.write(f"  xgboost: {HAS_XGBOOST}\n")
            f.write(f"  lightgbm: {HAS_LIGHTGBM}\n")

            if not combined_final.empty:
                f.write("\n\nBest final model per horizon by RMSE:\n")
                best_by_h = (
                    combined_final
                    .sort_values("RMSE")
                    .groupby("horizon", as_index=False)
                    .first()
                )

                f.write(
                    best_by_h[
                        ["horizon", "horizon_label", "model", "MAE", "RMSE", "SMAPE_percent", "R2"]
                    ].to_string(index=False)
                )

                f.write("\n\nAll final metrics:\n")
                f.write(
                    combined_final[
                        ["horizon", "model", "n", "MAE", "RMSE", "SMAPE_percent", "R2"]
                    ].to_string(index=False)
                )

            if not combined_wf.empty:
                f.write("\n\nWalk-forward average metrics:\n")

                wf_avg = (
                    combined_wf
                    .groupby(["horizon", "horizon_label", "model"], as_index=False)
                    .agg(
                        mean_RMSE=("RMSE", "mean"),
                        std_RMSE=("RMSE", "std"),
                        mean_MAE=("MAE", "mean"),
                        folds=("fold", "nunique"),
                    )
                    .sort_values(["horizon", "mean_RMSE"])
                )

                f.write(wf_avg.to_string(index=False))

        print(f"    Summary saved → {summary_path}")

        print(f"\n  ── STEP 4 LITE summary for {dataset_name} ──")

        if not combined_final.empty:
            best_by_h = (
                combined_final
                .sort_values("RMSE")
                .groupby("horizon", as_index=False)
                .first()
            )

            for _, row in best_by_h.iterrows():
                print(
                    f"  {row['horizon_label']:16s} | "
                    f"Best={row['model']:22s} | "
                    f"RMSE={row['RMSE']:.4f} | "
                    f"MAE={row['MAE']:.4f} | "
                    f"SMAPE={row['SMAPE_percent']:.2f}%"
                )

        print(f"  Outputs saved → {dirs['base']}")
        print()

    finally:
        conn.close()


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 10 — Entry point
# ═════════════════════════════════════════════════════════════════════════════

def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 90)
    print("  STEP 4 LITE — Multi-Horizon Walk-Forward Forecasting")
    print("=" * 90)

    print("\nPackage status:")
    print(f"  joblib   : {HAS_JOBLIB}")
    print(f"  xgboost : {HAS_XGBOOST}")
    print(f"  lightgbm: {HAS_LIGHTGBM}")

    for db_path in DB_PATHS:
        if not os.path.isfile(db_path):
            print(f"\n[SKIP] File not found: {db_path}")
            continue

        run_step4_lite_for_db(db_path)

    print("\n" + "=" * 90)
    print("  STEP 4 LITE complete for all databases.")
    print("=" * 90)


if __name__ == "__main__":
    main()

In [ ]:
r"""
STEP 5 — Separated Final Reports for Each Dataset
==================================================

This script reads STEP 4 LITE outputs and creates a separate final report package
for each dataset.

Important:
  - It does NOT combine campus_load into one report.
  - Each dataset gets its own folder.
  - Each dataset gets its own tables, plots, Excel file, Markdown report, and HTML report.
  - No model training is performed here.
  - This script only summarizes and visualizes already-produced STEP 4 LITE outputs.

Input folder:
  PROJECT_ROOT\STEP4_LITE_OUTPUTS

Output folder:
  PROJECT_ROOT\STEP5_FINAL_REPORT_OUTPUTS_SEPARATED

All comments are in English.
"""

from __future__ import annotations

import html
import warnings
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)


# Portable project paths for GitHub
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_BASE_DIR = PROJECT_ROOT / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)

# ═════════════════════════════════════════════════════════════════════════════
# USER CONFIGURATION
# ═════════════════════════════════════════════════════════════════════════════

STEP4_OUTPUT_DIR = OUTPUT_BASE_DIR / "STEP4_LITE_OUTPUTS"

FINAL_OUTPUT_DIR = OUTPUT_BASE_DIR / "STEP5_FINAL_REPORT_OUTPUTS_SEPARATED"

DATASETS = [
    "campus_load",
]

HORIZON_ORDER = [
    "h15min",
    "h1hour",
    "h24hour",
]

HORIZON_LABELS = {
    "h15min": "15-minute ahead",
    "h1hour": "1-hour ahead",
    "h24hour": "24-hour ahead",
}

BASELINE_PREFIX = "Baseline"

MAX_POINTS_LINE_PLOT = 60_000
MAX_POINTS_SCATTER = 80_000

PLOT_DPI = 200

# ═════════════════════════════════════════════════════════════════════════════


# ── Plot aesthetics ─────────────────────────────────────────────────────────

sns.set_theme(style="whitegrid", context="talk", font_scale=0.95)

plt.rcParams.update(
    {
        "figure.dpi": 130,
        "savefig.dpi": PLOT_DPI,
        "axes.spines.right": False,
        "axes.spines.top": False,
        "figure.constrained_layout.use": True,
    }
)

PALETTE = {
    "main": "#2196F3",
    "best": "#43A047",
    "warn": "#FFC107",
    "bad": "#E53935",
    "neutral": "#78909C",
    "actual": "#212121",
    "pred": "#D81B60",
    "bar": "#5C6BC0",
}


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 1 — General helpers
# ═════════════════════════════════════════════════════════════════════════════

def ensure_dir(path: Path) -> None:
    """Create a directory if it does not exist."""
    path.mkdir(parents=True, exist_ok=True)


def save_fig(path: Path) -> None:
    """Save current figure and close all figures."""
    ensure_dir(path.parent)
    plt.savefig(path, bbox_inches="tight")
    plt.close("all")


def horizon_label(horizon: str) -> str:
    """Return a readable horizon label."""
    return HORIZON_LABELS.get(str(horizon), str(horizon))


def safe_filename(text: str) -> str:
    """Convert text to safe filename."""
    text = str(text)
    bad_chars = [" ", "/", "\\", ":", "|", "(", ")", "%", ",", ";"]
    for ch in bad_chars:
        text = text.replace(ch, "_")
    return text


def parse_timestamp_column(s: pd.Series) -> pd.Series:
    """Parse timestamp column robustly."""
    try:
        return pd.to_datetime(s, errors="coerce", format="mixed")
    except TypeError:
        return pd.to_datetime(s, errors="coerce")


def read_csv_if_exists(path: Path) -> pd.DataFrame:
    """Read CSV if it exists, otherwise return an empty DataFrame."""
    if not path.is_file():
        return pd.DataFrame()
    return pd.read_csv(path)


def add_horizon_info(df: pd.DataFrame) -> pd.DataFrame:
    """Add horizon label and horizon order."""
    out = df.copy()

    if "horizon" in out.columns:
        out["horizon_label"] = out["horizon"].map(HORIZON_LABELS).fillna(out["horizon"])
        order_map = {h: i for i, h in enumerate(HORIZON_ORDER)}
        out["horizon_order"] = out["horizon"].map(order_map).fillna(999).astype(int)

    return out


def is_baseline_model(model_name: str) -> bool:
    """Check whether a model is a baseline model."""
    return str(model_name).startswith(BASELINE_PREFIX)


def model_family(model_name: str) -> str:
    """Classify model into a broad model family."""
    name = str(model_name).lower()

    if "baseline" in name:
        return "Baseline"
    if "lightgbm" in name:
        return "LightGBM"
    if "xgboost" in name:
        return "XGBoost"
    if "histgradient" in name or "gradient" in name:
        return "Gradient Boosting"
    if "ridge" in name or "elastic" in name or "linear" in name:
        return "Linear / SVR"
    if "svr" in name:
        return "SVR"
    if "forest" in name or "trees" in name:
        return "Tree Ensemble"
    if "neural" in name or "mlp" in name or "lstm" in name or "gru" in name:
        return "Neural Network"

    return "Other"


def downsample_time_df(
    df: pd.DataFrame,
    time_col: str,
    max_points: int,
    random_state: int = 42,
) -> pd.DataFrame:
    """Downsample a time-series DataFrame for plotting."""
    if df.empty:
        return df

    if len(df) <= max_points:
        return df.sort_values(time_col).reset_index(drop=True)

    return (
        df.sample(max_points, random_state=random_state)
        .sort_values(time_col)
        .reset_index(drop=True)
    )


def format_float(x, decimals: int = 4) -> str:
    """Safely format a float value."""
    if pd.isna(x):
        return ""
    try:
        return f"{float(x):.{decimals}f}"
    except Exception:
        return str(x)


def dataframe_to_markdown_safe(df: pd.DataFrame, max_rows: int = 50) -> str:
    """Convert a DataFrame to markdown safely."""
    if df.empty:
        return "_No data available._"

    show = df.head(max_rows).copy()

    try:
        return show.to_markdown(index=False)
    except Exception:
        return show.to_string(index=False)


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 2 — Load one dataset outputs
# ═════════════════════════════════════════════════════════════════════════════

def load_step4_outputs_for_dataset(dataset_name: str) -> Dict[str, pd.DataFrame]:
    """
    Load STEP 4 LITE outputs for one dataset only.

    No cross-dataset concatenation happens here.
    """
    base = STEP4_OUTPUT_DIR / dataset_name

    paths = {
        "final_metrics": base / "reports" / "STEP4_LITE_final_metrics_all.csv",
        "walk_forward": base / "reports" / "STEP4_LITE_walk_forward_metrics_all.csv",
        "segments": base / "reports" / "STEP4_LITE_segment_metrics_all.csv",
        "best_predictions": base / "predictions" / "STEP4_LITE_best_predictions_all_horizons.csv",
        "tuning": base / "tuning" / "STEP4_LITE_tuning_results_all.csv",
    }

    outputs = {}

    for key, path in paths.items():
        df = read_csv_if_exists(path)

        if not df.empty:
            df["dataset"] = dataset_name
            df = add_horizon_info(df)

        outputs[key] = df

    return outputs


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 3 — Build tables for one dataset
# ═════════════════════════════════════════════════════════════════════════════

def build_best_model_table(final_metrics: pd.DataFrame) -> pd.DataFrame:
    """Create best model table for one dataset."""
    if final_metrics.empty:
        return pd.DataFrame()

    df = add_horizon_info(final_metrics.copy())

    best = (
        df.sort_values(["horizon_order", "RMSE"])
        .groupby("horizon", as_index=False)
        .first()
    )

    best["model_family"] = best["model"].apply(model_family)
    best["is_baseline"] = best["model"].apply(is_baseline_model)

    keep_cols = [
        "dataset",
        "horizon",
        "horizon_label",
        "model",
        "model_family",
        "is_baseline",
        "n",
        "MAE",
        "RMSE",
        "MAPE_percent",
        "SMAPE_percent",
        "R2",
    ]

    keep_cols = [c for c in keep_cols if c in best.columns]

    best = best[keep_cols]
    best = add_horizon_info(best)
    best = best.sort_values("horizon_order").reset_index(drop=True)

    return best


def build_baseline_improvement_table(final_metrics: pd.DataFrame) -> pd.DataFrame:
    """
    Compare selected best model with best baseline for each horizon.

    Positive improvement means selected best model is better than baseline.
    """
    if final_metrics.empty:
        return pd.DataFrame()

    df = add_horizon_info(final_metrics.copy())
    rows = []

    for horizon, sub in df.groupby("horizon"):
        sub = sub.copy()

        if sub.empty:
            continue

        best_overall = sub.sort_values("RMSE").iloc[0]

        baseline_sub = sub[sub["model"].astype(str).str.startswith(BASELINE_PREFIX)].copy()

        if baseline_sub.empty:
            continue

        best_baseline = baseline_sub.sort_values("RMSE").iloc[0]

        rmse_improvement = np.nan
        mae_improvement = np.nan

        if best_baseline["RMSE"] != 0:
            rmse_improvement = (
                (best_baseline["RMSE"] - best_overall["RMSE"])
                / best_baseline["RMSE"]
                * 100
            )

        if best_baseline["MAE"] != 0:
            mae_improvement = (
                (best_baseline["MAE"] - best_overall["MAE"])
                / best_baseline["MAE"]
                * 100
            )

        rows.append(
            {
                "dataset": best_overall.get("dataset", ""),
                "horizon": horizon,
                "horizon_label": horizon_label(horizon),
                "best_model": best_overall["model"],
                "best_model_MAE": best_overall["MAE"],
                "best_model_RMSE": best_overall["RMSE"],
                "best_model_SMAPE_percent": best_overall["SMAPE_percent"],
                "best_baseline": best_baseline["model"],
                "best_baseline_MAE": best_baseline["MAE"],
                "best_baseline_RMSE": best_baseline["RMSE"],
                "best_baseline_SMAPE_percent": best_baseline["SMAPE_percent"],
                "MAE_improvement_percent": mae_improvement,
                "RMSE_improvement_percent": rmse_improvement,
                "best_is_baseline": is_baseline_model(best_overall["model"]),
            }
        )

    out = pd.DataFrame(rows)
    out = add_horizon_info(out)

    if not out.empty:
        out = out.sort_values("horizon_order").reset_index(drop=True)

    return out


def build_walk_forward_average_table(wf_metrics: pd.DataFrame) -> pd.DataFrame:
    """Aggregate walk-forward metrics for one dataset."""
    if wf_metrics.empty:
        return pd.DataFrame()

    df = add_horizon_info(wf_metrics.copy())

    avg = (
        df.groupby(["dataset", "horizon", "horizon_label", "model"], as_index=False)
        .agg(
            folds=("fold", "nunique"),
            mean_MAE=("MAE", "mean"),
            std_MAE=("MAE", "std"),
            mean_RMSE=("RMSE", "mean"),
            std_RMSE=("RMSE", "std"),
            mean_SMAPE_percent=("SMAPE_percent", "mean"),
            mean_R2=("R2", "mean"),
        )
    )

    avg["model_family"] = avg["model"].apply(model_family)
    avg["is_baseline"] = avg["model"].apply(is_baseline_model)

    avg = add_horizon_info(avg)
    avg = avg.sort_values(["horizon_order", "mean_RMSE"]).reset_index(drop=True)

    return avg


def build_best_walk_forward_table(wf_avg: pd.DataFrame) -> pd.DataFrame:
    """Select best average walk-forward model per horizon."""
    if wf_avg.empty:
        return pd.DataFrame()

    best = (
        wf_avg.sort_values(["horizon_order", "mean_RMSE"])
        .groupby("horizon", as_index=False)
        .first()
    )

    best = add_horizon_info(best)
    best = best.sort_values("horizon_order").reset_index(drop=True)

    return best


def build_best_segment_table(
    segment_metrics: pd.DataFrame,
    best_models: pd.DataFrame,
) -> pd.DataFrame:
    """Keep only segment metrics for the selected best model per horizon."""
    if segment_metrics.empty or best_models.empty:
        return pd.DataFrame()

    seg = add_horizon_info(segment_metrics.copy())

    best = best_models[["horizon", "model"]].copy()
    best = best.rename(columns={"model": "best_model"})

    merged = seg.merge(best, on="horizon", how="left")
    merged = merged[merged["model"] == merged["best_model"]].copy()

    merged = add_horizon_info(merged)

    if not merged.empty:
        merged = merged.sort_values(["horizon_order", "segment"]).reset_index(drop=True)

    return merged


def build_tuning_summary(tuning_df: pd.DataFrame) -> pd.DataFrame:
    """Select best tuning trial for each horizon/model."""
    if tuning_df.empty or "trial" not in tuning_df.columns:
        return pd.DataFrame()

    df = add_horizon_info(tuning_df.copy())

    if "RMSE" not in df.columns:
        return pd.DataFrame()

    best_trials = (
        df.dropna(subset=["RMSE"])
        .sort_values(["horizon_order", "model", "RMSE"])
        .groupby(["horizon", "model"], as_index=False)
        .first()
    )

    best_trials = add_horizon_info(best_trials)
    best_trials = best_trials.sort_values(["horizon_order", "model"]).reset_index(drop=True)

    keep_cols = [
        "dataset",
        "horizon",
        "horizon_label",
        "model",
        "trial",
        "MAE",
        "RMSE",
        "SMAPE_percent",
        "R2",
        "params_json",
    ]

    keep_cols = [c for c in keep_cols if c in best_trials.columns]

    return best_trials[keep_cols]


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 4 — Save tables for one dataset
# ═════════════════════════════════════════════════════════════════════════════

def save_tables_for_dataset(
    dataset_name: str,
    final_metrics: pd.DataFrame,
    wf_metrics: pd.DataFrame,
    segment_metrics: pd.DataFrame,
    predictions: pd.DataFrame,
    tuning_df: pd.DataFrame,
    best_models: pd.DataFrame,
    baseline_improvement: pd.DataFrame,
    wf_avg: pd.DataFrame,
    wf_best: pd.DataFrame,
    best_segments: pd.DataFrame,
    tuning_summary: pd.DataFrame,
    tables_dir: Path,
) -> None:
    """Save CSV and Excel tables for one dataset."""
    ensure_dir(tables_dir)

    table_map = {
        "01_best_models_by_horizon": best_models,
        "02_all_final_metrics": final_metrics,
        "03_baseline_improvement": baseline_improvement,
        "04_walk_forward_metrics_all": wf_metrics,
        "05_walk_forward_average": wf_avg,
        "06_walk_forward_best": wf_best,
        "07_segment_metrics_all": segment_metrics,
        "08_best_model_segment_metrics": best_segments,
        "09_tuning_results_all": tuning_df,
        "10_tuning_summary_best_trials": tuning_summary,
        "11_best_predictions_all_horizons": predictions,
    }

    for name, df in table_map.items():
        if not df.empty:
            df.to_csv(tables_dir / f"{name}_{dataset_name}.csv", index=False)

    excel_path = tables_dir / f"FINAL_PROJECT_SUMMARY_{dataset_name}.xlsx"

    try:
        with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
            for name, df in table_map.items():
                if df.empty:
                    continue

                sheet_name = name[:31]
                df.to_excel(writer, sheet_name=sheet_name, index=False)

        print(f"    Excel summary saved → {excel_path}")

    except Exception as exc:
        print(f"    [WARN] Excel file was not created for {dataset_name}: {exc}")
        print("    CSV files were still saved.")


def save_table_as_image(
    df: pd.DataFrame,
    title: str,
    path: Path,
    max_rows: int = 20,
    float_cols: Optional[List[str]] = None,
) -> None:
    """Save DataFrame as a PNG table."""
    if df.empty:
        return

    float_cols = float_cols or []

    show = df.head(max_rows).copy()

    for col in float_cols:
        if col in show.columns:
            show[col] = show[col].apply(lambda x: format_float(x, 4))

    show = show.astype(str)

    n_rows, n_cols = show.shape

    fig_width = max(12, n_cols * 1.7)
    fig_height = max(4, n_rows * 0.45 + 1.6)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis("off")

    table = ax.table(
        cellText=show.values,
        colLabels=show.columns,
        loc="center",
        cellLoc="center",
    )

    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.0, 1.3)

    ax.set_title(title, fontsize=16, fontweight="bold", pad=20)

    save_fig(path)


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 5 — Plot tables and metric summaries for one dataset
# ═════════════════════════════════════════════════════════════════════════════

def plot_best_metric_by_horizon(
    best_models: pd.DataFrame,
    dataset_name: str,
    metric: str,
    plots_dir: Path,
) -> None:
    """Plot selected metric for best model per horizon."""
    if best_models.empty or metric not in best_models.columns:
        return

    df = add_horizon_info(best_models.copy())
    df = df.sort_values("horizon_order")

    fig, ax = plt.subplots(figsize=(12, 6))

    bars = ax.bar(
        df["horizon_label"],
        df[metric],
        color=PALETTE["bar"],
        alpha=0.88,
        edgecolor="none",
    )

    for bar, (_, row) in zip(bars, df.iterrows()):
        value = row[metric]
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{row['model']}\n{value:.2f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

    ax.set_title(
        f"{dataset_name} | Best {metric} by forecast horizon",
        fontweight="bold",
    )
    ax.set_xlabel("Forecast horizon")
    ax.set_ylabel(metric)

    save_fig(plots_dir / f"best_{metric.lower()}_by_horizon_{dataset_name}.png")


def plot_baseline_improvement(
    baseline_improvement: pd.DataFrame,
    dataset_name: str,
    plots_dir: Path,
) -> None:
    """Plot RMSE improvement over best baseline."""
    if baseline_improvement.empty:
        return

    df = add_horizon_info(baseline_improvement.copy())
    df = df.sort_values("horizon_order")

    colors = [
        PALETTE["best"] if val >= 0 else PALETTE["bad"]
        for val in df["RMSE_improvement_percent"]
    ]

    fig, ax = plt.subplots(figsize=(12, 6))

    bars = ax.bar(
        df["horizon_label"],
        df["RMSE_improvement_percent"],
        color=colors,
        alpha=0.88,
        edgecolor="none",
    )

    ax.axhline(0, color="black", linewidth=1)

    for bar, (_, row) in zip(bars, df.iterrows()):
        value = row["RMSE_improvement_percent"]
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value,
            f"{value:.1f}%",
            ha="center",
            va="bottom" if value >= 0 else "top",
            fontsize=10,
        )

    ax.set_title(
        f"{dataset_name} | RMSE improvement over best baseline",
        fontweight="bold",
    )
    ax.set_xlabel("Forecast horizon")
    ax.set_ylabel("RMSE improvement over baseline (%)")

    save_fig(plots_dir / f"baseline_improvement_rmse_{dataset_name}.png")


def plot_model_ranking_per_horizon(
    final_metrics: pd.DataFrame,
    dataset_name: str,
    plots_dir: Path,
) -> None:
    """Plot model ranking by RMSE for each horizon."""
    if final_metrics.empty:
        return

    df = add_horizon_info(final_metrics.copy())

    for horizon, sub in df.groupby("horizon"):
        sub = sub.sort_values("RMSE").copy()

        fig_height = max(6, 0.45 * len(sub))
        fig, ax = plt.subplots(figsize=(14, fig_height))

        sns.barplot(
            data=sub,
            y="model",
            x="RMSE",
            ax=ax,
            color=PALETTE["bar"],
        )

        for i, (_, row) in enumerate(sub.iterrows()):
            ax.text(
                row["RMSE"],
                i,
                f"  {row['RMSE']:.2f}",
                va="center",
                fontsize=9,
            )

        ax.set_title(
            f"{dataset_name} | Model ranking by RMSE — {horizon_label(horizon)}",
            fontweight="bold",
        )
        ax.set_xlabel("RMSE")
        ax.set_ylabel("Model")

        save_fig(
            plots_dir / f"model_ranking_rmse_{dataset_name}_{safe_filename(horizon)}.png"
        )


def plot_walk_forward_heatmaps(
    wf_metrics: pd.DataFrame,
    dataset_name: str,
    plots_dir: Path,
) -> None:
    """Plot walk-forward RMSE heatmap for each horizon."""
    if wf_metrics.empty:
        return

    df = add_horizon_info(wf_metrics.copy())

    for horizon, sub in df.groupby("horizon"):
        pivot = sub.pivot_table(
            values="RMSE",
            index="model",
            columns="fold",
            aggfunc="mean",
        )

        if pivot.empty:
            continue

        order = pivot.mean(axis=1).sort_values().index
        pivot = pivot.loc[order]

        fig, ax = plt.subplots(figsize=(15, max(6, 0.45 * len(pivot))))

        sns.heatmap(
            pivot,
            annot=True,
            fmt=".2f",
            cmap="YlGnBu_r",
            cbar_kws={"label": "RMSE"},
            ax=ax,
        )

        ax.set_title(
            f"{dataset_name} | Walk-forward RMSE heatmap — {horizon_label(horizon)}",
            fontweight="bold",
        )
        ax.set_xlabel("Walk-forward fold")
        ax.set_ylabel("Model")

        save_fig(
            plots_dir / f"walk_forward_heatmap_{dataset_name}_{safe_filename(horizon)}.png"
        )


def plot_walk_forward_stability(
    wf_avg: pd.DataFrame,
    dataset_name: str,
    plots_dir: Path,
) -> None:
    """Plot walk-forward mean RMSE vs RMSE standard deviation."""
    if wf_avg.empty:
        return

    df = add_horizon_info(wf_avg.copy())

    for horizon, sub in df.groupby("horizon"):
        sub = sub.copy()
        sub["std_RMSE"] = sub["std_RMSE"].fillna(0)

        fig, ax = plt.subplots(figsize=(10, 7))

        ax.scatter(
            sub["mean_RMSE"],
            sub["std_RMSE"],
            s=120,
            alpha=0.8,
            color=PALETTE["main"],
            edgecolor="white",
            linewidth=0.8,
        )

        for _, row in sub.iterrows():
            ax.text(
                row["mean_RMSE"],
                row["std_RMSE"],
                f" {row['model']}",
                fontsize=9,
                va="center",
            )

        ax.set_title(
            f"{dataset_name} | Walk-forward stability — {horizon_label(horizon)}",
            fontweight="bold",
        )
        ax.set_xlabel("Mean RMSE across folds")
        ax.set_ylabel("RMSE standard deviation across folds")

        save_fig(
            plots_dir / f"walk_forward_stability_{dataset_name}_{safe_filename(horizon)}.png"
        )


def plot_best_segment_metrics(
    best_segments: pd.DataFrame,
    dataset_name: str,
    plots_dir: Path,
) -> None:
    """Plot segment MAE for best model per horizon."""
    if best_segments.empty:
        return

    df = add_horizon_info(best_segments.copy())

    for horizon, sub in df.groupby("horizon"):
        sub = sub.sort_values("MAE").copy()

        model_name = sub["model"].iloc[0] if "model" in sub.columns else "Best model"

        fig, ax = plt.subplots(figsize=(12, 6))

        sns.barplot(
            data=sub,
            x="segment",
            y="MAE",
            ax=ax,
            color=PALETTE["warn"],
        )

        ax.set_title(
            f"{dataset_name} | Segment MAE — {model_name} — {horizon_label(horizon)}",
            fontweight="bold",
        )
        ax.set_xlabel("Segment")
        ax.set_ylabel("MAE")
        plt.xticks(rotation=20, ha="right")

        save_fig(
            plots_dir / f"segment_mae_{dataset_name}_{safe_filename(horizon)}.png"
        )


def plot_tuning_results(
    tuning_df: pd.DataFrame,
    dataset_name: str,
    plots_dir: Path,
) -> None:
    """Plot tuning RMSE by trial."""
    if tuning_df.empty or "trial" not in tuning_df.columns:
        return

    df = add_horizon_info(tuning_df.copy())

    for (horizon, model), sub in df.groupby(["horizon", "model"]):
        sub = sub.dropna(subset=["RMSE"]).sort_values("trial").copy()

        if sub.empty:
            continue

        fig, ax = plt.subplots(figsize=(12, 5))

        ax.plot(
            sub["trial"],
            sub["RMSE"],
            marker="o",
            linewidth=1.7,
            color=PALETTE["bar"],
        )

        best_idx = sub["RMSE"].idxmin()
        best_row = sub.loc[best_idx]

        ax.scatter(
            [best_row["trial"]],
            [best_row["RMSE"]],
            s=140,
            color=PALETTE["best"],
            zorder=5,
            label=f"Best trial RMSE = {best_row['RMSE']:.4f}",
        )

        ax.set_title(
            f"{dataset_name} | Tuning results — {model} — {horizon_label(horizon)}",
            fontweight="bold",
        )
        ax.set_xlabel("Trial")
        ax.set_ylabel("Validation RMSE")
        ax.legend()

        save_fig(
            plots_dir
            / f"tuning_{dataset_name}_{safe_filename(horizon)}_{safe_filename(model)}.png"
        )


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 6 — Prediction plots for one dataset
# ═════════════════════════════════════════════════════════════════════════════

def prepare_predictions(predictions: pd.DataFrame) -> pd.DataFrame:
    """Prepare best predictions table for plotting."""
    if predictions.empty:
        return predictions

    df = add_horizon_info(predictions.copy())

    if "timestamp_15min" in df.columns:
        df["timestamp_15min"] = parse_timestamp_column(df["timestamp_15min"])

    if "y_true" in df.columns:
        df["y_true"] = pd.to_numeric(df["y_true"], errors="coerce")

    if "y_pred" in df.columns:
        df["y_pred"] = pd.to_numeric(df["y_pred"], errors="coerce")

    df["error"] = df["y_true"] - df["y_pred"]
    df["abs_error"] = df["error"].abs()

    return df


def plot_actual_vs_predicted(
    predictions: pd.DataFrame,
    dataset_name: str,
    plots_dir: Path,
) -> None:
    """Plot actual vs predicted for each horizon."""
    if predictions.empty:
        return

    df = prepare_predictions(predictions)

    for horizon, sub in df.groupby("horizon"):
        sub = sub.dropna(subset=["timestamp_15min", "y_true", "y_pred"]).copy()

        if sub.empty:
            continue

        sub_plot = downsample_time_df(
            sub,
            time_col="timestamp_15min",
            max_points=MAX_POINTS_LINE_PLOT,
        )

        model_name = sub_plot["model"].iloc[0]

        fig, ax = plt.subplots(figsize=(16, 5))

        ax.plot(
            sub_plot["timestamp_15min"],
            sub_plot["y_true"],
            linewidth=0.8,
            color=PALETTE["actual"],
            label="Actual",
        )

        ax.plot(
            sub_plot["timestamp_15min"],
            sub_plot["y_pred"],
            linewidth=0.8,
            alpha=0.85,
            color=PALETTE["pred"],
            label=f"Predicted — {model_name}",
        )

        ax.set_title(
            f"{dataset_name} | Actual vs predicted — {model_name} — {horizon_label(horizon)}",
            fontweight="bold",
        )
        ax.set_xlabel("Time")
        ax.set_ylabel("Load")
        ax.legend()

        save_fig(
            plots_dir / f"actual_vs_predicted_{dataset_name}_{safe_filename(horizon)}.png"
        )


def plot_zoomed_actual_vs_predicted(
    predictions: pd.DataFrame,
    dataset_name: str,
    plots_dir: Path,
    days: int = 14,
) -> None:
    """Plot first N days of actual vs predicted."""
    if predictions.empty:
        return

    df = prepare_predictions(predictions)

    for horizon, sub in df.groupby("horizon"):
        sub = sub.dropna(subset=["timestamp_15min", "y_true", "y_pred"]).copy()

        if sub.empty:
            continue

        start = sub["timestamp_15min"].min()
        end = start + pd.Timedelta(days=days)

        zoom = sub[
            (sub["timestamp_15min"] >= start)
            & (sub["timestamp_15min"] < end)
        ].copy()

        if zoom.empty:
            continue

        model_name = zoom["model"].iloc[0]

        fig, ax = plt.subplots(figsize=(16, 5))

        ax.plot(
            zoom["timestamp_15min"],
            zoom["y_true"],
            linewidth=1.0,
            color=PALETTE["actual"],
            label="Actual",
        )

        ax.plot(
            zoom["timestamp_15min"],
            zoom["y_pred"],
            linewidth=1.0,
            color=PALETTE["pred"],
            alpha=0.85,
            label=f"Predicted — {model_name}",
        )

        ax.set_title(
            f"{dataset_name} | First {days} test days — {model_name} — {horizon_label(horizon)}",
            fontweight="bold",
        )
        ax.set_xlabel("Time")
        ax.set_ylabel("Load")
        ax.legend()

        save_fig(
            plots_dir / f"zoom_{days}_days_{dataset_name}_{safe_filename(horizon)}.png"
        )


def plot_predicted_vs_actual_scatter(
    predictions: pd.DataFrame,
    dataset_name: str,
    plots_dir: Path,
) -> None:
    """Plot predicted vs actual scatter for each horizon."""
    if predictions.empty:
        return

    df = prepare_predictions(predictions)

    for horizon, sub in df.groupby("horizon"):
        sub = sub.dropna(subset=["y_true", "y_pred"]).copy()

        if sub.empty:
            continue

        if len(sub) > MAX_POINTS_SCATTER:
            sub = sub.sample(MAX_POINTS_SCATTER, random_state=42)

        model_name = sub["model"].iloc[0]

        fig, ax = plt.subplots(figsize=(7, 7))

        ax.scatter(
            sub["y_true"],
            sub["y_pred"],
            s=8,
            alpha=0.35,
            color=PALETTE["main"],
            edgecolors="none",
        )

        min_v = min(sub["y_true"].min(), sub["y_pred"].min())
        max_v = max(sub["y_true"].max(), sub["y_pred"].max())

        ax.plot([min_v, max_v], [min_v, max_v], "--", color="black", linewidth=1)

        ax.set_title(
            f"{dataset_name} | Predicted vs actual — {model_name} — {horizon_label(horizon)}",
            fontweight="bold",
        )
        ax.set_xlabel("Actual")
        ax.set_ylabel("Predicted")

        save_fig(
            plots_dir / f"scatter_pred_vs_actual_{dataset_name}_{safe_filename(horizon)}.png"
        )


def plot_residual_distribution(
    predictions: pd.DataFrame,
    dataset_name: str,
    plots_dir: Path,
) -> None:
    """Plot residual distribution for each horizon."""
    if predictions.empty:
        return

    df = prepare_predictions(predictions)

    for horizon, sub in df.groupby("horizon"):
        sub = sub.dropna(subset=["error"]).copy()

        if sub.empty:
            continue

        model_name = sub["model"].iloc[0]

        fig, ax = plt.subplots(figsize=(12, 5))

        sns.histplot(
            sub["error"],
            bins=100,
            kde=True,
            color=PALETTE["bar"],
            ax=ax,
        )

        ax.axvline(0, color="black", linestyle="--", linewidth=1)

        ax.set_title(
            f"{dataset_name} | Residual distribution — {model_name} — {horizon_label(horizon)}",
            fontweight="bold",
        )
        ax.set_xlabel("Residual = actual - predicted")
        ax.set_ylabel("Count")

        save_fig(
            plots_dir / f"residual_distribution_{dataset_name}_{safe_filename(horizon)}.png"
        )


def plot_residuals_over_time(
    predictions: pd.DataFrame,
    dataset_name: str,
    plots_dir: Path,
) -> None:
    """Plot residuals over time for each horizon."""
    if predictions.empty:
        return

    df = prepare_predictions(predictions)

    for horizon, sub in df.groupby("horizon"):
        sub = sub.dropna(subset=["timestamp_15min", "error"]).copy()

        if sub.empty:
            continue

        sub_plot = downsample_time_df(
            sub,
            time_col="timestamp_15min",
            max_points=MAX_POINTS_SCATTER,
        )

        model_name = sub_plot["model"].iloc[0]

        fig, ax = plt.subplots(figsize=(16, 5))

        ax.scatter(
            sub_plot["timestamp_15min"],
            sub_plot["error"],
            s=4,
            alpha=0.35,
            color=PALETTE["neutral"],
            edgecolors="none",
        )

        ax.axhline(0, color="black", linestyle="--", linewidth=1)

        ax.set_title(
            f"{dataset_name} | Residuals over time — {model_name} — {horizon_label(horizon)}",
            fontweight="bold",
        )
        ax.set_xlabel("Time")
        ax.set_ylabel("Residual")

        save_fig(
            plots_dir / f"residuals_over_time_{dataset_name}_{safe_filename(horizon)}.png"
        )


def plot_mae_by_hour(
    predictions: pd.DataFrame,
    dataset_name: str,
    plots_dir: Path,
) -> None:
    """Plot MAE by hour for each horizon."""
    if predictions.empty:
        return

    df = prepare_predictions(predictions)

    if "timestamp_15min" not in df.columns:
        return

    df["hour"] = df["timestamp_15min"].dt.hour

    for horizon, sub in df.groupby("horizon"):
        sub = sub.dropna(subset=["hour", "abs_error"]).copy()

        if sub.empty:
            continue

        hourly = sub.groupby("hour")["abs_error"].mean().reset_index()
        model_name = sub["model"].iloc[0]

        fig, ax = plt.subplots(figsize=(12, 5))

        ax.bar(
            hourly["hour"],
            hourly["abs_error"],
            color=PALETTE["bar"],
            alpha=0.88,
        )

        ax.set_xticks(range(24))
        ax.set_title(
            f"{dataset_name} | MAE by hour — {model_name} — {horizon_label(horizon)}",
            fontweight="bold",
        )
        ax.set_xlabel("Hour of day")
        ax.set_ylabel("MAE")

        save_fig(
            plots_dir / f"mae_by_hour_{dataset_name}_{safe_filename(horizon)}.png"
        )


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 7 — Reports for one dataset
# ═════════════════════════════════════════════════════════════════════════════

def build_interpretation_text(
    dataset_name: str,
    best_models: pd.DataFrame,
    baseline_improvement: pd.DataFrame,
    wf_best: pd.DataFrame,
) -> str:
    """Create automatic interpretation text for one dataset."""
    lines = []

    lines.append(f"# Final Interpretation — {dataset_name}")
    lines.append("")

    lines.append("## Selected best models")
    lines.append("")

    if best_models.empty:
        lines.append("No best model table was available.")
        return "\n".join(lines)

    for _, row in best_models.iterrows():
        if bool(row.get("is_baseline", False)):
            meaning = (
                "The selected method is a baseline. This means that a simple "
                "time-based rule was more reliable than the trained ML models "
                "for this horizon."
            )
        else:
            meaning = (
                "The selected method is a trained machine learning model. This "
                "means that the engineered features added predictive value beyond "
                "the baseline methods."
            )

        lines.append(
            f"- **{row['horizon_label']}**: best model = **{row['model']}**, "
            f"RMSE = **{row['RMSE']:.4f}**, MAE = **{row['MAE']:.4f}**, "
            f"SMAPE = **{row['SMAPE_percent']:.2f}%**. {meaning}"
        )

    lines.append("")
    lines.append("## Baseline comparison")
    lines.append("")

    if baseline_improvement.empty:
        lines.append("No baseline comparison table was available.")
    else:
        for _, row in baseline_improvement.iterrows():
            if row["RMSE_improvement_percent"] > 0:
                lines.append(
                    f"- **{row['horizon_label']}**: the selected model improved "
                    f"RMSE by **{row['RMSE_improvement_percent']:.2f}%** compared "
                    f"with the best baseline `{row['best_baseline']}`."
                )
            elif abs(row["RMSE_improvement_percent"]) < 1e-9:
                lines.append(
                    f"- **{row['horizon_label']}**: the selected model and the best "
                    f"baseline are effectively tied. The best method is "
                    f"`{row['best_model']}`."
                )
            else:
                lines.append(
                    f"- **{row['horizon_label']}**: the best baseline "
                    f"`{row['best_baseline']}` remained stronger than the trained "
                    f"ML models. This is an important and honest result."
                )

    lines.append("")
    lines.append("## Walk-forward validation")
    lines.append("")

    if wf_best.empty:
        lines.append("No walk-forward best table was available.")
    else:
        for _, row in wf_best.iterrows():
            lines.append(
                f"- **{row['horizon_label']}**: best average walk-forward model = "
                f"**{row['model']}**, mean RMSE = **{row['mean_RMSE']:.4f}**, "
                f"number of folds = **{int(row['folds'])}**."
            )

    lines.append("")
    lines.append("## Recommended report statement")
    lines.append("")
    lines.append(
        f"For `{dataset_name}`, the final conclusions should be based on the "
        "multi-horizon results, the walk-forward validation results, and the "
        "comparison against baseline models. This makes the model selection more "
        "defensible than using only a single train/test split."
    )

    return "\n".join(lines)


def create_markdown_report(
    dataset_name: str,
    best_models: pd.DataFrame,
    baseline_improvement: pd.DataFrame,
    wf_avg: pd.DataFrame,
    wf_best: pd.DataFrame,
    best_segments: pd.DataFrame,
    tuning_summary: pd.DataFrame,
    reports_dir: Path,
) -> Path:
    """Create Markdown report for one dataset."""
    ensure_dir(reports_dir)

    report_path = reports_dir / f"FINAL_REPORT_{dataset_name}.md"

    interpretation = build_interpretation_text(
        dataset_name=dataset_name,
        best_models=best_models,
        baseline_improvement=baseline_improvement,
        wf_best=wf_best,
    )

    lines = []

    lines.append(f"# Final Energy Load Forecasting Report — {dataset_name}")
    lines.append("")

    lines.append("## 1. Executive summary")
    lines.append("")
    lines.append(dataframe_to_markdown_safe(best_models))
    lines.append("")

    lines.append("## 2. Improvement over baselines")
    lines.append("")
    lines.append(dataframe_to_markdown_safe(baseline_improvement))
    lines.append("")

    lines.append("## 3. Walk-forward average metrics")
    lines.append("")
    lines.append(dataframe_to_markdown_safe(wf_avg, max_rows=80))
    lines.append("")

    lines.append("## 4. Best walk-forward model per horizon")
    lines.append("")
    lines.append(dataframe_to_markdown_safe(wf_best))
    lines.append("")

    lines.append("## 5. Segment-level error analysis for selected models")
    lines.append("")
    lines.append(dataframe_to_markdown_safe(best_segments, max_rows=80))
    lines.append("")

    lines.append("## 6. Tuning summary")
    lines.append("")
    lines.append(dataframe_to_markdown_safe(tuning_summary, max_rows=80))
    lines.append("")

    lines.append(interpretation)

    report_path.write_text("\n".join(lines), encoding="utf-8")

    return report_path


def create_html_report(
    dataset_name: str,
    best_models: pd.DataFrame,
    baseline_improvement: pd.DataFrame,
    wf_avg: pd.DataFrame,
    wf_best: pd.DataFrame,
    best_segments: pd.DataFrame,
    tuning_summary: pd.DataFrame,
    plots_dir: Path,
    reports_dir: Path,
    dataset_output_dir: Path,
) -> Path:
    """Create HTML report for one dataset."""
    ensure_dir(reports_dir)

    report_path = reports_dir / f"FINAL_REPORT_{dataset_name}.html"

    def table_html(df: pd.DataFrame, max_rows: int = 80) -> str:
        if df.empty:
            return "<p><em>No data available.</em></p>"

        show = df.head(max_rows).copy()

        return show.to_html(index=False, border=0, classes="data-table")

    plot_files = sorted(plots_dir.glob("*.png"))

    plot_html_parts = []

    for p in plot_files:
        rel = p.relative_to(dataset_output_dir).as_posix()

        plot_html_parts.append(
            f"""
            <div class="plot-block">
              <h3>{html.escape(p.stem)}</h3>
              <img src="{html.escape(rel)}" alt="{html.escape(p.stem)}">
            </div>
            """
        )

    interpretation = build_interpretation_text(
        dataset_name=dataset_name,
        best_models=best_models,
        baseline_improvement=baseline_improvement,
        wf_best=wf_best,
    )

    interpretation_html = (
        "<pre class='interpretation'>"
        + html.escape(interpretation)
        + "</pre>"
    )

    html_text = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>Final Report — {html.escape(dataset_name)}</title>
<style>
body {{
    font-family: Arial, sans-serif;
    margin: 30px;
    background: #fafafa;
    color: #222;
}}
h1, h2, h3 {{
    color: #1f3a5f;
}}
.data-table {{
    border-collapse: collapse;
    font-size: 13px;
    margin-bottom: 28px;
    width: 100%;
}}
.data-table th {{
    background: #1f3a5f;
    color: white;
    padding: 8px;
    border: 1px solid #ddd;
}}
.data-table td {{
    padding: 7px;
    border: 1px solid #ddd;
}}
.data-table tr:nth-child(even) {{
    background: #f2f2f2;
}}
.plot-block {{
    margin: 28px 0;
    padding: 18px;
    background: white;
    border: 1px solid #ddd;
    border-radius: 8px;
}}
.plot-block img {{
    max-width: 100%;
    height: auto;
}}
.interpretation {{
    white-space: pre-wrap;
    background: #ffffff;
    border: 1px solid #ddd;
    padding: 18px;
    border-radius: 8px;
    line-height: 1.5;
}}
</style>
</head>
<body>

<h1>Final Energy Load Forecasting Report — {html.escape(dataset_name)}</h1>

<h2>1. Executive Summary: Best Models</h2>
{table_html(best_models)}

<h2>2. Improvement over Baselines</h2>
{table_html(baseline_improvement)}

<h2>3. Walk-forward Average Metrics</h2>
{table_html(wf_avg, max_rows=120)}

<h2>4. Best Walk-forward Model per Horizon</h2>
{table_html(wf_best)}

<h2>5. Segment-level Error Analysis</h2>
{table_html(best_segments, max_rows=120)}

<h2>6. Tuning Summary</h2>
{table_html(tuning_summary, max_rows=120)}

<h2>7. Automatic Interpretation</h2>
{interpretation_html}

<h2>8. Final Figures</h2>
{''.join(plot_html_parts)}

</body>
</html>
"""

    report_path.write_text(html_text, encoding="utf-8")

    return report_path


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 8 — Run one dataset report
# ═════════════════════════════════════════════════════════════════════════════

def run_report_for_dataset(dataset_name: str) -> None:
    """Run STEP 5 separated final report for one dataset."""
    print("\n" + "=" * 90)
    print(f"  STEP 5 — Final separated report for {dataset_name}")
    print("=" * 90)

    dataset_output_dir = FINAL_OUTPUT_DIR / dataset_name
    tables_dir = dataset_output_dir / "tables"
    plots_dir = dataset_output_dir / "plots"
    reports_dir = dataset_output_dir / "reports"

    ensure_dir(dataset_output_dir)
    ensure_dir(tables_dir)
    ensure_dir(plots_dir)
    ensure_dir(reports_dir)

    print("\n[1/6] Loading STEP 4 LITE outputs for this dataset only ...")

    outputs = load_step4_outputs_for_dataset(dataset_name)

    final_metrics = outputs["final_metrics"]
    wf_metrics = outputs["walk_forward"]
    segment_metrics = outputs["segments"]
    predictions = outputs["best_predictions"]
    tuning_df = outputs["tuning"]

    print(f"    Final metrics rows       : {len(final_metrics):,}")
    print(f"    Walk-forward rows        : {len(wf_metrics):,}")
    print(f"    Segment metrics rows     : {len(segment_metrics):,}")
    print(f"    Best prediction rows     : {len(predictions):,}")
    print(f"    Tuning rows              : {len(tuning_df):,}")

    if final_metrics.empty:
        print(f"    [SKIP] No final metrics found for {dataset_name}.")
        return

    print("\n[2/6] Building dataset-specific tables ...")

    best_models = build_best_model_table(final_metrics)
    baseline_improvement = build_baseline_improvement_table(final_metrics)
    wf_avg = build_walk_forward_average_table(wf_metrics)
    wf_best = build_best_walk_forward_table(wf_avg)
    best_segments = build_best_segment_table(segment_metrics, best_models)
    tuning_summary = build_tuning_summary(tuning_df)

    print(f"    Best model rows          : {len(best_models):,}")
    print(f"    Baseline improvement rows: {len(baseline_improvement):,}")
    print(f"    WF average rows          : {len(wf_avg):,}")
    print(f"    WF best rows             : {len(wf_best):,}")
    print(f"    Best segment rows        : {len(best_segments):,}")
    print(f"    Tuning summary rows      : {len(tuning_summary):,}")

    print("\n[3/6] Saving dataset-specific CSV and Excel tables ...")

    save_tables_for_dataset(
        dataset_name=dataset_name,
        final_metrics=final_metrics,
        wf_metrics=wf_metrics,
        segment_metrics=segment_metrics,
        predictions=predictions,
        tuning_df=tuning_df,
        best_models=best_models,
        baseline_improvement=baseline_improvement,
        wf_avg=wf_avg,
        wf_best=wf_best,
        best_segments=best_segments,
        tuning_summary=tuning_summary,
        tables_dir=tables_dir,
    )

    print("\n[4/6] Creating dataset-specific plots ...")

    # Table images.
    save_table_as_image(
        best_models,
        title=f"{dataset_name} — Best Models by Horizon",
        path=plots_dir / f"table_best_models_{dataset_name}.png",
        float_cols=["MAE", "RMSE", "SMAPE_percent", "R2"],
    )

    save_table_as_image(
        baseline_improvement,
        title=f"{dataset_name} — Improvement over Best Baseline",
        path=plots_dir / f"table_baseline_improvement_{dataset_name}.png",
        float_cols=[
            "best_model_RMSE",
            "best_baseline_RMSE",
            "RMSE_improvement_percent",
            "MAE_improvement_percent",
        ],
    )

    # Metric summary plots.
    plot_best_metric_by_horizon(best_models, dataset_name, "RMSE", plots_dir)
    plot_best_metric_by_horizon(best_models, dataset_name, "MAE", plots_dir)
    plot_best_metric_by_horizon(best_models, dataset_name, "SMAPE_percent", plots_dir)

    plot_baseline_improvement(baseline_improvement, dataset_name, plots_dir)
    plot_model_ranking_per_horizon(final_metrics, dataset_name, plots_dir)

    # Walk-forward plots.
    plot_walk_forward_heatmaps(wf_metrics, dataset_name, plots_dir)
    plot_walk_forward_stability(wf_avg, dataset_name, plots_dir)

    # Segment and tuning plots.
    plot_best_segment_metrics(best_segments, dataset_name, plots_dir)
    plot_tuning_results(tuning_df, dataset_name, plots_dir)

    # Prediction plots.
    plot_actual_vs_predicted(predictions, dataset_name, plots_dir)
    plot_zoomed_actual_vs_predicted(predictions, dataset_name, plots_dir, days=14)
    plot_predicted_vs_actual_scatter(predictions, dataset_name, plots_dir)
    plot_residual_distribution(predictions, dataset_name, plots_dir)
    plot_residuals_over_time(predictions, dataset_name, plots_dir)
    plot_mae_by_hour(predictions, dataset_name, plots_dir)

    print("    Dataset-specific plots generated.")

    print("\n[5/6] Creating dataset-specific Markdown and HTML reports ...")

    md_path = create_markdown_report(
        dataset_name=dataset_name,
        best_models=best_models,
        baseline_improvement=baseline_improvement,
        wf_avg=wf_avg,
        wf_best=wf_best,
        best_segments=best_segments,
        tuning_summary=tuning_summary,
        reports_dir=reports_dir,
    )

    html_path = create_html_report(
        dataset_name=dataset_name,
        best_models=best_models,
        baseline_improvement=baseline_improvement,
        wf_avg=wf_avg,
        wf_best=wf_best,
        best_segments=best_segments,
        tuning_summary=tuning_summary,
        plots_dir=plots_dir,
        reports_dir=reports_dir,
        dataset_output_dir=dataset_output_dir,
    )

    print(f"    Markdown report saved → {md_path}")
    print(f"    HTML report saved     → {html_path}")

    print("\n[6/6] Console summary for this dataset ...")

    print("\n  ── Best models ──")

    if not best_models.empty:
        show_cols = [
            "horizon_label",
            "model",
            "MAE",
            "RMSE",
            "SMAPE_percent",
            "R2",
        ]
        show_cols = [c for c in show_cols if c in best_models.columns]
        print(best_models[show_cols].to_string(index=False))

    print("\n  ── Baseline improvement ──")

    if not baseline_improvement.empty:
        show_cols = [
            "horizon_label",
            "best_model",
            "best_baseline",
            "RMSE_improvement_percent",
        ]
        show_cols = [c for c in show_cols if c in baseline_improvement.columns]
        print(baseline_improvement[show_cols].to_string(index=False))

    print(f"\n  Outputs saved separately → {dataset_output_dir}")


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 9 — Entry point
# ═════════════════════════════════════════════════════════════════════════════

def main() -> None:
    """Run STEP 5 separated final reports for all datasets."""
    ensure_dir(FINAL_OUTPUT_DIR)

    print("\n" + "=" * 90)
    print("  STEP 5 — Separated Final Reports")
    print("=" * 90)

    print(f"\nInput STEP 4 folder : {STEP4_OUTPUT_DIR}")
    print(f"Output STEP 5 folder: {FINAL_OUTPUT_DIR}")

    for dataset_name in DATASETS:
        run_report_for_dataset(dataset_name)

    print("\n" + "=" * 90)
    print("  STEP 5 separated reports complete.")
    print("=" * 90)

    print("\nFinal output folders:")

    for dataset_name in DATASETS:
        print(f"  - {FINAL_OUTPUT_DIR / dataset_name}")


if __name__ == "__main__":
    main()

In [ ]:
r"""
STEP 6 — Ensemble Forecasting, Prediction Intervals, and Reliability Analysis
============================================================================

This script reads STEP 4 LITE outputs and creates a separated reliability package
for each dataset.

It performs:

  1. Ensemble forecasting:
       - Top-3 all models weighted ensemble
       - Top-3 ML-only weighted ensemble
       - Top-3 all models mean ensemble
       - Top-3 all models median ensemble

  2. Champion model selection:
       - Compare best single model vs ensemble models
       - Select final champion model per horizon

  3. Prediction intervals:
       - Residual-based 80% interval
       - Residual-based 95% interval

  4. Error diagnostics:
       - Error by hour
       - Error by day of week
       - Error by month
       - Error by weekday/weekend
       - Error by load quantile
       - Worst prediction days
       - Best prediction days

  5. Final outputs:
       - CSV tables
       - Excel summary
       - PNG plots
       - Markdown report
       - HTML report

Important:
  - This script does NOT retrain original models.
  - It uses prediction files generated by STEP 4 LITE.
  - Outputs are separated for campus_load.

Input folder:
  PROJECT_ROOT\STEP4_LITE_OUTPUTS

Output folder:
  PROJECT_ROOT\STEP6_RELIABILITY_OUTPUTS

All comments are in English.
"""

from __future__ import annotations

import html
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)


# Portable project paths for GitHub
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_BASE_DIR = PROJECT_ROOT / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)

# ═════════════════════════════════════════════════════════════════════════════
# USER CONFIGURATION
# ═════════════════════════════════════════════════════════════════════════════

STEP4_OUTPUT_DIR = OUTPUT_BASE_DIR / "STEP4_LITE_OUTPUTS"

STEP6_OUTPUT_DIR = OUTPUT_BASE_DIR / "STEP6_RELIABILITY_OUTPUTS"

DATASETS = [
    "campus_load",
]

HORIZON_ORDER = [
    "h15min",
    "h1hour",
    "h24hour",
]

HORIZON_LABELS = {
    "h15min": "15-minute ahead",
    "h1hour": "1-hour ahead",
    "h24hour": "24-hour ahead",
}

TIMESTAMP_COL = "timestamp_15min"

BASELINE_PREFIX = "Baseline"

TOP_N_ENSEMBLE = 3

MIN_IMPROVEMENT_FOR_ENSEMBLE_PERCENT = 0.50

INTERVAL_CALIBRATION_RATIO = 0.50

PI_LEVELS = {
    "PI80": (0.10, 0.90),
    "PI95": (0.025, 0.975),
}

LOAD_QUANTILE_BINS = 5

MAX_POINTS_LINE_PLOT = 60_000
MAX_POINTS_SCATTER = 80_000

PLOT_DPI = 200
RANDOM_STATE = 42

# ═════════════════════════════════════════════════════════════════════════════


# ── Plot aesthetics ─────────────────────────────────────────────────────────

sns.set_theme(style="whitegrid", context="talk", font_scale=0.95)

plt.rcParams.update(
    {
        "figure.dpi": 130,
        "savefig.dpi": PLOT_DPI,
        "axes.spines.right": False,
        "axes.spines.top": False,
        "figure.constrained_layout.use": True,
    }
)

PALETTE = {
    "actual": "#212121",
    "pred": "#D81B60",
    "pi80": "#90CAF9",
    "pi95": "#BBDEFB",
    "main": "#2196F3",
    "best": "#43A047",
    "warn": "#FFC107",
    "bad": "#E53935",
    "neutral": "#78909C",
    "bar": "#5C6BC0",
}


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 1 — General helpers
# ═════════════════════════════════════════════════════════════════════════════

def ensure_dir(path: Path) -> None:
    """Create directory if it does not exist."""
    path.mkdir(parents=True, exist_ok=True)


def save_fig(path: Path) -> None:
    """Save current figure and close all figures."""
    ensure_dir(path.parent)
    plt.savefig(path, bbox_inches="tight")
    plt.close("all")


def safe_filename(text: str) -> str:
    """Convert text to a safe filename."""
    text = str(text)
    for ch in [" ", "/", "\\", ":", "|", "(", ")", "%", ",", ";", "[", "]"]:
        text = text.replace(ch, "_")
    return text


def horizon_label(horizon: str) -> str:
    """Return readable horizon label."""
    return HORIZON_LABELS.get(str(horizon), str(horizon))


def add_horizon_info(df: pd.DataFrame) -> pd.DataFrame:
    """Add readable horizon label and horizon order."""
    if df.empty:
        return df

    out = df.copy()

    if "horizon" in out.columns:
        out["horizon_label"] = out["horizon"].map(HORIZON_LABELS).fillna(out["horizon"])
        order_map = {h: i for i, h in enumerate(HORIZON_ORDER)}
        out["horizon_order"] = out["horizon"].map(order_map).fillna(999).astype(int)

    return out


def parse_timestamp_column(s: pd.Series) -> pd.Series:
    """Parse timestamp column robustly."""
    try:
        return pd.to_datetime(s, errors="coerce", format="mixed")
    except TypeError:
        return pd.to_datetime(s, errors="coerce")


def read_csv_if_exists(path: Path) -> pd.DataFrame:
    """Read CSV if it exists."""
    if not path.is_file():
        return pd.DataFrame()
    return pd.read_csv(path)


def is_baseline_model(model_name: str) -> bool:
    """Return True if model is a baseline."""
    return str(model_name).startswith(BASELINE_PREFIX)


def model_family(model_name: str) -> str:
    """Classify model into a broad family."""
    name = str(model_name).lower()

    if "baseline" in name:
        return "Baseline"
    if "lightgbm" in name:
        return "LightGBM"
    if "xgboost" in name:
        return "XGBoost"
    if "histgradient" in name or "gradient" in name:
        return "Gradient Boosting"
    if "ridge" in name or "elastic" in name or "linear" in name:
        return "Linear / SVR"
    if "svr" in name:
        return "SVR"
    if "forest" in name or "trees" in name:
        return "Tree Ensemble"
    if "neural" in name or "mlp" in name or "lstm" in name or "gru" in name:
        return "Neural Network"

    return "Other"


def downsample_time_df(
    df: pd.DataFrame,
    time_col: str,
    max_points: int,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    """Downsample a time series DataFrame for plotting."""
    if df.empty:
        return df

    if len(df) <= max_points:
        return df.sort_values(time_col).reset_index(drop=True)

    return (
        df.sample(max_points, random_state=random_state)
        .sort_values(time_col)
        .reset_index(drop=True)
    )


def dataframe_to_markdown_safe(df: pd.DataFrame, max_rows: int = 80) -> str:
    """Convert DataFrame to markdown safely."""
    if df.empty:
        return "_No data available._"

    show = df.head(max_rows).copy()

    try:
        return show.to_markdown(index=False)
    except Exception:
        return show.to_string(index=False)


def save_table_as_image(
    df: pd.DataFrame,
    title: str,
    path: Path,
    max_rows: int = 25,
    float_cols: Optional[List[str]] = None,
) -> None:
    """Save DataFrame as a PNG table."""
    if df.empty:
        return

    float_cols = float_cols or []
    show = df.head(max_rows).copy()

    for col in float_cols:
        if col in show.columns:
            show[col] = show[col].apply(
                lambda x: "" if pd.isna(x) else f"{float(x):.4f}"
            )

    show = show.astype(str)

    n_rows, n_cols = show.shape

    fig_width = max(12, n_cols * 1.7)
    fig_height = max(4, n_rows * 0.45 + 1.6)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis("off")

    table = ax.table(
        cellText=show.values,
        colLabels=show.columns,
        loc="center",
        cellLoc="center",
    )

    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.0, 1.3)

    ax.set_title(title, fontsize=16, fontweight="bold", pad=20)

    save_fig(path)


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 2 — Metrics
# ═════════════════════════════════════════════════════════════════════════════

def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Root mean squared error."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Mean absolute error."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))


def smape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Symmetric mean absolute percentage error."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    denom = np.abs(y_true) + np.abs(y_pred)
    mask = denom != 0

    if mask.sum() == 0:
        return np.nan

    return float(np.mean(2.0 * np.abs(y_pred[mask] - y_true[mask]) / denom[mask]) * 100)


def mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Mean absolute percentage error, ignoring zero targets."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mask = y_true != 0

    if mask.sum() == 0:
        return np.nan

    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def r2_score_manual(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Manual R2 score."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)

    if ss_tot == 0:
        return np.nan

    return float(1.0 - ss_res / ss_tot)


def evaluate_prediction_frame(
    pred_df: pd.DataFrame,
    dataset_name: str,
    horizon: str,
    model_name: str,
    split_name: str = "final_test",
) -> Dict:
    """Evaluate a prediction DataFrame."""
    df = pred_df.copy()

    df["y_true"] = pd.to_numeric(df["y_true"], errors="coerce")
    df["y_pred"] = pd.to_numeric(df["y_pred"], errors="coerce")

    mask = np.isfinite(df["y_true"]) & np.isfinite(df["y_pred"])
    df = df[mask].copy()

    if df.empty:
        return {
            "dataset": dataset_name,
            "horizon": horizon,
            "horizon_label": horizon_label(horizon),
            "split": split_name,
            "model": model_name,
            "model_family": model_family(model_name),
            "is_baseline": is_baseline_model(model_name),
            "n": 0,
            "MAE": np.nan,
            "RMSE": np.nan,
            "MAPE_percent": np.nan,
            "SMAPE_percent": np.nan,
            "R2": np.nan,
        }

    y_true = df["y_true"].values
    y_pred = df["y_pred"].values

    return {
        "dataset": dataset_name,
        "horizon": horizon,
        "horizon_label": horizon_label(horizon),
        "split": split_name,
        "model": model_name,
        "model_family": model_family(model_name),
        "is_baseline": is_baseline_model(model_name),
        "n": int(len(df)),
        "MAE": mae(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "MAPE_percent": mape(y_true, y_pred),
        "SMAPE_percent": smape(y_true, y_pred),
        "R2": r2_score_manual(y_true, y_pred),
    }


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 3 — Load STEP 4 outputs
# ═════════════════════════════════════════════════════════════════════════════

def load_step4_final_metrics(dataset_name: str) -> pd.DataFrame:
    """Load STEP 4 final metrics for one dataset."""
    path = (
        STEP4_OUTPUT_DIR
        / dataset_name
        / "reports"
        / "STEP4_LITE_final_metrics_all.csv"
    )

    df = read_csv_if_exists(path)

    if df.empty:
        return df

    df["dataset"] = dataset_name
    df = add_horizon_info(df)

    return df


def load_step4_all_predictions_for_horizon(
    dataset_name: str,
    horizon: str,
) -> pd.DataFrame:
    """Load all-model predictions for one dataset/horizon."""
    path = (
        STEP4_OUTPUT_DIR
        / dataset_name
        / "predictions"
        / f"final_all_predictions_{horizon}.csv"
    )

    df = read_csv_if_exists(path)

    if df.empty:
        return df

    df["dataset"] = dataset_name
    df["horizon"] = horizon
    df["horizon_label"] = horizon_label(horizon)

    if TIMESTAMP_COL in df.columns:
        df[TIMESTAMP_COL] = parse_timestamp_column(df[TIMESTAMP_COL])

    if "forecast_origin_timestamp" in df.columns:
        df["forecast_origin_timestamp"] = parse_timestamp_column(
            df["forecast_origin_timestamp"]
        )

    df["y_true"] = pd.to_numeric(df["y_true"], errors="coerce")
    df["y_pred"] = pd.to_numeric(df["y_pred"], errors="coerce")

    df = df.dropna(subset=[TIMESTAMP_COL, "y_true", "y_pred"]).copy()

    return df


def load_all_predictions_for_dataset(dataset_name: str) -> pd.DataFrame:
    """Load all final prediction files for one dataset."""
    frames = []

    for horizon in HORIZON_ORDER:
        df = load_step4_all_predictions_for_horizon(dataset_name, horizon)

        if not df.empty:
            frames.append(df)

    if not frames:
        return pd.DataFrame()

    out = pd.concat(frames, ignore_index=True)
    out = add_horizon_info(out)

    return out


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 4 — Ensemble construction
# ═════════════════════════════════════════════════════════════════════════════

def select_top_models(
    metrics_df: pd.DataFrame,
    horizon: str,
    top_n: int = TOP_N_ENSEMBLE,
    ml_only: bool = False,
) -> pd.DataFrame:
    """Select top models by RMSE for one horizon."""
    sub = metrics_df[metrics_df["horizon"] == horizon].copy()

    if sub.empty:
        return pd.DataFrame()

    if ml_only:
        sub = sub[~sub["model"].astype(str).str.startswith(BASELINE_PREFIX)].copy()

    sub = sub.dropna(subset=["RMSE"]).sort_values("RMSE").reset_index(drop=True)

    return sub.head(top_n).copy()


def make_inverse_rmse_weights(top_models: pd.DataFrame) -> Dict[str, float]:
    """Create inverse-RMSE weights."""
    if top_models.empty:
        return {}

    df = top_models.copy()
    df["safe_RMSE"] = pd.to_numeric(df["RMSE"], errors="coerce")
    df = df.dropna(subset=["safe_RMSE"])
    df = df[df["safe_RMSE"] > 0].copy()

    if df.empty:
        return {}

    inv = 1.0 / df["safe_RMSE"]
    weights = inv / inv.sum()

    return dict(zip(df["model"], weights))


def pivot_predictions_for_models(
    pred_df: pd.DataFrame,
    selected_models: List[str],
) -> pd.DataFrame:
    """Pivot predictions to wide format for selected models."""
    sub = pred_df[pred_df["model"].isin(selected_models)].copy()

    if sub.empty:
        return pd.DataFrame()

    wide_pred = sub.pivot_table(
        index=TIMESTAMP_COL,
        columns="model",
        values="y_pred",
        aggfunc="mean",
    )

    y_true = (
        sub.groupby(TIMESTAMP_COL, as_index=True)["y_true"]
        .mean()
        .rename("y_true")
    )

    out = wide_pred.join(y_true, how="inner")
    out = out.reset_index()

    return out


def build_weighted_ensemble_predictions(
    pred_df: pd.DataFrame,
    top_models: pd.DataFrame,
    ensemble_name: str,
    dataset_name: str,
    horizon: str,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Build weighted ensemble using inverse RMSE."""
    selected_models = top_models["model"].tolist()
    weights = make_inverse_rmse_weights(top_models)

    if not selected_models or not weights:
        return pd.DataFrame(), pd.DataFrame()

    wide = pivot_predictions_for_models(pred_df, selected_models)

    if wide.empty:
        return pd.DataFrame(), pd.DataFrame()

    available_models = [m for m in selected_models if m in wide.columns]

    if not available_models:
        return pd.DataFrame(), pd.DataFrame()

    weight_sum = sum(weights.get(m, 0.0) for m in available_models)

    if weight_sum <= 0:
        return pd.DataFrame(), pd.DataFrame()

    normalized_weights = {m: weights[m] / weight_sum for m in available_models}

    y_pred = np.zeros(len(wide), dtype=float)

    for model_name, weight in normalized_weights.items():
        y_pred += wide[model_name].values * weight

    out = pd.DataFrame(
        {
            "dataset": dataset_name,
            "horizon": horizon,
            "horizon_label": horizon_label(horizon),
            "model": ensemble_name,
            TIMESTAMP_COL: wide[TIMESTAMP_COL].values,
            "y_true": wide["y_true"].values,
            "y_pred": y_pred,
        }
    )

    weight_rows = []

    for model_name, weight in normalized_weights.items():
        weight_rows.append(
            {
                "dataset": dataset_name,
                "horizon": horizon,
                "horizon_label": horizon_label(horizon),
                "ensemble": ensemble_name,
                "base_model": model_name,
                "weight": weight,
            }
        )

    weights_df = pd.DataFrame(weight_rows)

    return out, weights_df


def build_simple_ensemble_predictions(
    pred_df: pd.DataFrame,
    top_models: pd.DataFrame,
    ensemble_name: str,
    method: str,
    dataset_name: str,
    horizon: str,
) -> pd.DataFrame:
    """Build mean or median ensemble."""
    selected_models = top_models["model"].tolist()

    if not selected_models:
        return pd.DataFrame()

    wide = pivot_predictions_for_models(pred_df, selected_models)

    if wide.empty:
        return pd.DataFrame()

    model_cols = [m for m in selected_models if m in wide.columns]

    if not model_cols:
        return pd.DataFrame()

    if method == "mean":
        y_pred = wide[model_cols].mean(axis=1).values
    elif method == "median":
        y_pred = wide[model_cols].median(axis=1).values
    else:
        raise ValueError(f"Unsupported ensemble method: {method}")

    out = pd.DataFrame(
        {
            "dataset": dataset_name,
            "horizon": horizon,
            "horizon_label": horizon_label(horizon),
            "model": ensemble_name,
            TIMESTAMP_COL: wide[TIMESTAMP_COL].values,
            "y_true": wide["y_true"].values,
            "y_pred": y_pred,
        }
    )

    return out


def build_ensembles_for_horizon(
    dataset_name: str,
    horizon: str,
    final_metrics: pd.DataFrame,
    all_pred_df: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Build ensemble predictions and metrics for one horizon."""
    pred_h = all_pred_df[all_pred_df["horizon"] == horizon].copy()

    if pred_h.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    metrics_rows = []
    pred_frames = []
    weight_frames = []

    top_all = select_top_models(
        metrics_df=final_metrics,
        horizon=horizon,
        top_n=TOP_N_ENSEMBLE,
        ml_only=False,
    )

    top_ml = select_top_models(
        metrics_df=final_metrics,
        horizon=horizon,
        top_n=TOP_N_ENSEMBLE,
        ml_only=True,
    )

    ensemble_specs = []

    if not top_all.empty:
        ensemble_specs.append(("Ensemble_Top3_All_Weighted", "weighted", top_all))
        ensemble_specs.append(("Ensemble_Top3_All_Mean", "mean", top_all))
        ensemble_specs.append(("Ensemble_Top3_All_Median", "median", top_all))

    if len(top_ml) >= 2:
        ensemble_specs.append(("Ensemble_Top3_ML_Weighted", "weighted", top_ml))

    for ensemble_name, method, top_models in ensemble_specs:
        if method == "weighted":
            pred_df, weights_df = build_weighted_ensemble_predictions(
                pred_df=pred_h,
                top_models=top_models,
                ensemble_name=ensemble_name,
                dataset_name=dataset_name,
                horizon=horizon,
            )

            if not weights_df.empty:
                weight_frames.append(weights_df)

        else:
            pred_df = build_simple_ensemble_predictions(
                pred_df=pred_h,
                top_models=top_models,
                ensemble_name=ensemble_name,
                method=method,
                dataset_name=dataset_name,
                horizon=horizon,
            )

        if pred_df.empty:
            continue

        metric = evaluate_prediction_frame(
            pred_df=pred_df,
            dataset_name=dataset_name,
            horizon=horizon,
            model_name=ensemble_name,
            split_name="final_test",
        )

        metrics_rows.append(metric)
        pred_frames.append(pred_df)

    metrics_df = pd.DataFrame(metrics_rows)
    predictions_df = pd.concat(pred_frames, ignore_index=True) if pred_frames else pd.DataFrame()
    weights_df = pd.concat(weight_frames, ignore_index=True) if weight_frames else pd.DataFrame()

    metrics_df = add_horizon_info(metrics_df)
    predictions_df = add_horizon_info(predictions_df)
    weights_df = add_horizon_info(weights_df)

    return metrics_df, predictions_df, weights_df


def build_all_ensembles(
    dataset_name: str,
    final_metrics: pd.DataFrame,
    all_predictions: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Build ensembles for all horizons."""
    metric_frames = []
    pred_frames = []
    weight_frames = []

    for horizon in HORIZON_ORDER:
        ensemble_metrics, ensemble_preds, ensemble_weights = build_ensembles_for_horizon(
            dataset_name=dataset_name,
            horizon=horizon,
            final_metrics=final_metrics,
            all_pred_df=all_predictions,
        )

        if not ensemble_metrics.empty:
            metric_frames.append(ensemble_metrics)

        if not ensemble_preds.empty:
            pred_frames.append(ensemble_preds)

        if not ensemble_weights.empty:
            weight_frames.append(ensemble_weights)

    all_metrics = pd.concat(metric_frames, ignore_index=True) if metric_frames else pd.DataFrame()
    all_preds = pd.concat(pred_frames, ignore_index=True) if pred_frames else pd.DataFrame()
    all_weights = pd.concat(weight_frames, ignore_index=True) if weight_frames else pd.DataFrame()

    all_metrics = add_horizon_info(all_metrics)
    all_preds = add_horizon_info(all_preds)
    all_weights = add_horizon_info(all_weights)

    return all_metrics, all_preds, all_weights


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 5 — Champion selection
# ═════════════════════════════════════════════════════════════════════════════

def build_champion_table(
    dataset_name: str,
    final_metrics: pd.DataFrame,
    ensemble_metrics: pd.DataFrame,
) -> pd.DataFrame:
    """Build final champion selection table."""
    rows = []

    final_metrics = add_horizon_info(final_metrics.copy())
    ensemble_metrics = add_horizon_info(ensemble_metrics.copy())

    for horizon in HORIZON_ORDER:
        single_sub = final_metrics[final_metrics["horizon"] == horizon].copy()
        ens_sub = ensemble_metrics[ensemble_metrics["horizon"] == horizon].copy()

        if single_sub.empty:
            continue

        best_single = single_sub.sort_values("RMSE").iloc[0]

        if ens_sub.empty:
            rows.append(
                {
                    "dataset": dataset_name,
                    "horizon": horizon,
                    "horizon_label": horizon_label(horizon),
                    "best_single_model": best_single["model"],
                    "best_single_RMSE": best_single["RMSE"],
                    "best_single_MAE": best_single["MAE"],
                    "best_single_SMAPE_percent": best_single["SMAPE_percent"],
                    "best_ensemble_model": "",
                    "best_ensemble_RMSE": np.nan,
                    "best_ensemble_MAE": np.nan,
                    "best_ensemble_SMAPE_percent": np.nan,
                    "RMSE_improvement_of_ensemble_percent": np.nan,
                    "champion_model": best_single["model"],
                    "champion_type": "Single",
                    "selection_reason": "No valid ensemble was available.",
                }
            )
            continue

        best_ensemble = ens_sub.sort_values("RMSE").iloc[0]

        improvement = (
            (best_single["RMSE"] - best_ensemble["RMSE"])
            / best_single["RMSE"]
            * 100
            if best_single["RMSE"] != 0
            else np.nan
        )

        if improvement >= MIN_IMPROVEMENT_FOR_ENSEMBLE_PERCENT:
            champion_model = best_ensemble["model"]
            champion_type = "Ensemble"
            reason = (
                f"Ensemble improved RMSE by {improvement:.2f}%, "
                f"which is >= {MIN_IMPROVEMENT_FOR_ENSEMBLE_PERCENT:.2f}% threshold."
            )
        else:
            champion_model = best_single["model"]
            champion_type = "Single"
            reason = (
                f"Ensemble improvement was {improvement:.2f}%, "
                f"which is below {MIN_IMPROVEMENT_FOR_ENSEMBLE_PERCENT:.2f}% threshold. "
                f"The simpler best single model is selected."
            )

        rows.append(
            {
                "dataset": dataset_name,
                "horizon": horizon,
                "horizon_label": horizon_label(horizon),
                "best_single_model": best_single["model"],
                "best_single_RMSE": best_single["RMSE"],
                "best_single_MAE": best_single["MAE"],
                "best_single_SMAPE_percent": best_single["SMAPE_percent"],
                "best_ensemble_model": best_ensemble["model"],
                "best_ensemble_RMSE": best_ensemble["RMSE"],
                "best_ensemble_MAE": best_ensemble["MAE"],
                "best_ensemble_SMAPE_percent": best_ensemble["SMAPE_percent"],
                "RMSE_improvement_of_ensemble_percent": improvement,
                "champion_model": champion_model,
                "champion_type": champion_type,
                "selection_reason": reason,
            }
        )

    out = pd.DataFrame(rows)
    out = add_horizon_info(out)

    if not out.empty:
        out = out.sort_values("horizon_order").reset_index(drop=True)

    return out


def extract_champion_predictions(
    dataset_name: str,
    horizon: str,
    champion_model: str,
    all_predictions: pd.DataFrame,
    ensemble_predictions: pd.DataFrame,
) -> pd.DataFrame:
    """Extract predictions for selected champion model."""
    if str(champion_model).startswith("Ensemble"):
        source = ensemble_predictions
    else:
        source = all_predictions

    sub = source[
        (source["horizon"] == horizon)
        & (source["model"] == champion_model)
    ].copy()

    if sub.empty:
        return pd.DataFrame()

    sub["dataset"] = dataset_name
    sub["horizon"] = horizon
    sub["horizon_label"] = horizon_label(horizon)
    sub["champion_model"] = champion_model

    sub[TIMESTAMP_COL] = parse_timestamp_column(sub[TIMESTAMP_COL])
    sub["y_true"] = pd.to_numeric(sub["y_true"], errors="coerce")
    sub["y_pred"] = pd.to_numeric(sub["y_pred"], errors="coerce")

    sub["error"] = sub["y_true"] - sub["y_pred"]
    sub["abs_error"] = sub["error"].abs()

    sub = sub.dropna(subset=[TIMESTAMP_COL, "y_true", "y_pred"])
    sub = sub.sort_values(TIMESTAMP_COL).reset_index(drop=True)

    return sub


def build_all_champion_predictions(
    dataset_name: str,
    champion_table: pd.DataFrame,
    all_predictions: pd.DataFrame,
    ensemble_predictions: pd.DataFrame,
) -> pd.DataFrame:
    """Build champion predictions for all horizons."""
    frames = []

    for _, row in champion_table.iterrows():
        pred = extract_champion_predictions(
            dataset_name=dataset_name,
            horizon=row["horizon"],
            champion_model=row["champion_model"],
            all_predictions=all_predictions,
            ensemble_predictions=ensemble_predictions,
        )

        if not pred.empty:
            frames.append(pred)

    if not frames:
        return pd.DataFrame()

    out = pd.concat(frames, ignore_index=True)
    out = add_horizon_info(out)

    return out


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 6 — Prediction intervals
# ═════════════════════════════════════════════════════════════════════════════

def add_prediction_intervals_for_group(
    group: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Add residual-based prediction intervals for one group."""
    df = group.copy().sort_values(TIMESTAMP_COL).reset_index(drop=True)

    n = len(df)

    if n < 20:
        return df, pd.DataFrame()

    split_idx = int(n * INTERVAL_CALIBRATION_RATIO)
    split_idx = max(5, min(split_idx, n - 5))

    df["interval_split"] = "evaluation"
    df.loc[: split_idx - 1, "interval_split"] = "calibration"

    calibration = df.iloc[:split_idx].copy()
    evaluation = df.iloc[split_idx:].copy()

    residuals = calibration["error"].dropna()

    if residuals.empty:
        return df, pd.DataFrame()

    interval_rows = []

    for level_name, (q_low, q_high) in PI_LEVELS.items():
        low_resid = float(residuals.quantile(q_low))
        high_resid = float(residuals.quantile(q_high))

        lower_col = f"{level_name}_lower"
        upper_col = f"{level_name}_upper"
        covered_col = f"{level_name}_covered"

        df[lower_col] = df["y_pred"] + low_resid
        df[upper_col] = df["y_pred"] + high_resid

        df[covered_col] = (
            (df["y_true"] >= df[lower_col])
            & (df["y_true"] <= df[upper_col])
        ).astype(int)

        eval_df = df[df["interval_split"] == "evaluation"].copy()

        coverage_eval = float(eval_df[covered_col].mean() * 100) if not eval_df.empty else np.nan
        avg_width_eval = float((eval_df[upper_col] - eval_df[lower_col]).mean()) if not eval_df.empty else np.nan

        coverage_all = float(df[covered_col].mean() * 100)
        avg_width_all = float((df[upper_col] - df[lower_col]).mean())

        interval_rows.append(
            {
                "dataset": df["dataset"].iloc[0],
                "horizon": df["horizon"].iloc[0],
                "horizon_label": df["horizon_label"].iloc[0],
                "model": df["champion_model"].iloc[0],
                "interval": level_name,
                "calibration_rows": int(len(calibration)),
                "evaluation_rows": int(len(evaluation)),
                "residual_quantile_low": q_low,
                "residual_quantile_high": q_high,
                "residual_low": low_resid,
                "residual_high": high_resid,
                "coverage_eval_percent": coverage_eval,
                "avg_width_eval": avg_width_eval,
                "coverage_all_percent": coverage_all,
                "avg_width_all": avg_width_all,
            }
        )

    return df, pd.DataFrame(interval_rows)


def add_prediction_intervals(
    champion_predictions: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Add prediction intervals for all champion groups."""
    if champion_predictions.empty:
        return pd.DataFrame(), pd.DataFrame()

    pred_frames = []
    metric_frames = []

    for _, group in champion_predictions.groupby(["dataset", "horizon", "champion_model"]):
        pred_with_pi, pi_metrics = add_prediction_intervals_for_group(group)

        if not pred_with_pi.empty:
            pred_frames.append(pred_with_pi)

        if not pi_metrics.empty:
            metric_frames.append(pi_metrics)

    preds_out = pd.concat(pred_frames, ignore_index=True) if pred_frames else pd.DataFrame()
    metrics_out = pd.concat(metric_frames, ignore_index=True) if metric_frames else pd.DataFrame()

    preds_out = add_horizon_info(preds_out)
    metrics_out = add_horizon_info(metrics_out)

    return preds_out, metrics_out


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 7 — Error diagnostics
# ═════════════════════════════════════════════════════════════════════════════

def metric_summary_for_subset(
    df: pd.DataFrame,
    dataset_name: str,
    horizon: str,
    model_name: str,
    diagnostic_type: str,
    diagnostic_value: str,
) -> Dict:
    """Compute metrics for a diagnostic subset."""
    sub = df.dropna(subset=["y_true", "y_pred"]).copy()

    if sub.empty:
        return {
            "dataset": dataset_name,
            "horizon": horizon,
            "horizon_label": horizon_label(horizon),
            "model": model_name,
            "diagnostic_type": diagnostic_type,
            "diagnostic_value": diagnostic_value,
            "n": 0,
            "MAE": np.nan,
            "RMSE": np.nan,
            "SMAPE_percent": np.nan,
            "bias_mean_error": np.nan,
            "mean_actual": np.nan,
            "mean_predicted": np.nan,
        }

    y_true = sub["y_true"].values
    y_pred = sub["y_pred"].values
    error = y_true - y_pred

    return {
        "dataset": dataset_name,
        "horizon": horizon,
        "horizon_label": horizon_label(horizon),
        "model": model_name,
        "diagnostic_type": diagnostic_type,
        "diagnostic_value": diagnostic_value,
        "n": int(len(sub)),
        "MAE": mae(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "SMAPE_percent": smape(y_true, y_pred),
        "bias_mean_error": float(np.mean(error)),
        "mean_actual": float(np.mean(y_true)),
        "mean_predicted": float(np.mean(y_pred)),
    }


def build_error_diagnostics(champion_predictions: pd.DataFrame) -> pd.DataFrame:
    """Build detailed error diagnostics."""
    if champion_predictions.empty:
        return pd.DataFrame()

    df = champion_predictions.copy()
    df[TIMESTAMP_COL] = parse_timestamp_column(df[TIMESTAMP_COL])

    df["hour"] = df[TIMESTAMP_COL].dt.hour
    df["day_of_week"] = df[TIMESTAMP_COL].dt.dayofweek
    df["day_name"] = df[TIMESTAMP_COL].dt.day_name()
    df["month"] = df[TIMESTAMP_COL].dt.month
    df["month_name"] = df[TIMESTAMP_COL].dt.month_name()
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

    rows = []

    for (dataset_name, horizon, model_name), group in df.groupby(
        ["dataset", "horizon", "champion_model"]
    ):
        group = group.copy()

        for hour, sub in group.groupby("hour"):
            rows.append(
                metric_summary_for_subset(
                    sub,
                    dataset_name,
                    horizon,
                    model_name,
                    "hour",
                    str(int(hour)),
                )
            )

        for day_name, sub in group.groupby("day_name"):
            rows.append(
                metric_summary_for_subset(
                    sub,
                    dataset_name,
                    horizon,
                    model_name,
                    "day_of_week",
                    str(day_name),
                )
            )

        for month_name, sub in group.groupby("month_name"):
            rows.append(
                metric_summary_for_subset(
                    sub,
                    dataset_name,
                    horizon,
                    model_name,
                    "month",
                    str(month_name),
                )
            )

        for is_weekend, sub in group.groupby("is_weekend"):
            label = "weekend" if int(is_weekend) == 1 else "weekday"
            rows.append(
                metric_summary_for_subset(
                    sub,
                    dataset_name,
                    horizon,
                    model_name,
                    "weekday_weekend",
                    label,
                )
            )

        try:
            group["load_quantile"] = pd.qcut(
                group["y_true"],
                q=LOAD_QUANTILE_BINS,
                labels=[f"Q{i}" for i in range(1, LOAD_QUANTILE_BINS + 1)],
                duplicates="drop",
            )

            for quantile_label, sub in group.groupby("load_quantile"):
                rows.append(
                    metric_summary_for_subset(
                        sub,
                        dataset_name,
                        horizon,
                        model_name,
                        "load_quantile",
                        str(quantile_label),
                    )
                )

        except Exception:
            pass

    out = pd.DataFrame(rows)
    out = add_horizon_info(out)

    return out


def build_daily_error_tables(
    champion_predictions: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Build worst and best prediction day tables."""
    if champion_predictions.empty:
        return pd.DataFrame(), pd.DataFrame()

    df = champion_predictions.copy()
    df[TIMESTAMP_COL] = parse_timestamp_column(df[TIMESTAMP_COL])

    df["date"] = df[TIMESTAMP_COL].dt.date
    df["abs_error"] = np.abs(df["y_true"] - df["y_pred"])
    df["squared_error"] = (df["y_true"] - df["y_pred"]) ** 2

    daily = (
        df.groupby(
            ["dataset", "horizon", "horizon_label", "champion_model", "date"],
            as_index=False,
        )
        .agg(
            n=("abs_error", "size"),
            MAE=("abs_error", "mean"),
            RMSE=("squared_error", lambda x: float(np.sqrt(np.mean(x)))),
            mean_actual=("y_true", "mean"),
            mean_predicted=("y_pred", "mean"),
            max_abs_error=("abs_error", "max"),
        )
    )

    daily = add_horizon_info(daily)

    worst_frames = []
    best_frames = []

    for _, sub in daily.groupby(["dataset", "horizon"]):
        worst_frames.append(sub.sort_values("MAE", ascending=False).head(10))
        best_frames.append(sub.sort_values("MAE", ascending=True).head(10))

    worst = pd.concat(worst_frames, ignore_index=True) if worst_frames else pd.DataFrame()
    best = pd.concat(best_frames, ignore_index=True) if best_frames else pd.DataFrame()

    return worst, best


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 8 — Plots
# ═════════════════════════════════════════════════════════════════════════════

def plot_single_vs_ensemble_metrics(
    comparison_metrics: pd.DataFrame,
    dataset_name: str,
    plots_dir: Path,
) -> None:
    """Plot single models vs ensembles by RMSE."""
    if comparison_metrics.empty:
        return

    df = add_horizon_info(comparison_metrics.copy())

    for horizon, sub in df.groupby("horizon"):
        sub = sub.sort_values("RMSE").copy()

        fig, ax = plt.subplots(figsize=(14, max(6, 0.45 * len(sub))))

        sns.barplot(
            data=sub,
            y="model",
            x="RMSE",
            ax=ax,
            color=PALETTE["bar"],
        )

        for i, (_, row) in enumerate(sub.iterrows()):
            ax.text(row["RMSE"], i, f"  {row['RMSE']:.3f}", va="center", fontsize=9)

        ax.set_title(
            f"{dataset_name} | Single models vs ensembles — {horizon_label(horizon)}",
            fontweight="bold",
        )
        ax.set_xlabel("RMSE")
        ax.set_ylabel("Model")

        save_fig(
            plots_dir / f"single_vs_ensemble_rmse_{dataset_name}_{safe_filename(horizon)}.png"
        )


def plot_champion_summary(
    champion_table: pd.DataFrame,
    dataset_name: str,
    plots_dir: Path,
) -> None:
    """Plot champion selection summary."""
    if champion_table.empty:
        return

    df = add_horizon_info(champion_table.copy())
    df = df.sort_values("horizon_order")

    fig, ax = plt.subplots(figsize=(12, 6))

    ax.bar(
        df["horizon_label"],
        df["best_single_RMSE"],
        color=PALETTE["neutral"],
        alpha=0.50,
        label="Best single RMSE",
    )

    ax.plot(
        df["horizon_label"],
        df["best_ensemble_RMSE"],
        marker="o",
        linewidth=2,
        color=PALETTE["main"],
        label="Best ensemble RMSE",
    )

    champion_rmse = df.apply(
        lambda r: r["best_ensemble_RMSE"]
        if r["champion_type"] == "Ensemble"
        else r["best_single_RMSE"],
        axis=1,
    )

    ax.scatter(
        df["horizon_label"],
        champion_rmse,
        s=160,
        color=PALETTE["best"],
        zorder=5,
        label="Selected champion",
    )

    for _, row in df.iterrows():
        cr = row["best_ensemble_RMSE"] if row["champion_type"] == "Ensemble" else row["best_single_RMSE"]
        ax.text(
            row["horizon_label"],
            cr,
            f"{row['champion_model']}\n{cr:.2f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

    ax.set_title(f"{dataset_name} | Champion model selection", fontweight="bold")
    ax.set_xlabel("Forecast horizon")
    ax.set_ylabel("RMSE")
    ax.legend()

    save_fig(plots_dir / f"champion_selection_{dataset_name}.png")


def plot_prediction_with_intervals(
    champion_predictions_pi: pd.DataFrame,
    dataset_name: str,
    plots_dir: Path,
    zoom_days: Optional[int] = None,
) -> None:
    """Plot actual vs predicted with prediction intervals."""
    if champion_predictions_pi.empty:
        return

    df = champion_predictions_pi.copy()
    df[TIMESTAMP_COL] = parse_timestamp_column(df[TIMESTAMP_COL])

    for horizon, sub in df.groupby("horizon"):
        sub = sub.dropna(subset=[TIMESTAMP_COL, "y_true", "y_pred"]).copy()

        if sub.empty:
            continue

        if zoom_days is not None:
            start = sub[TIMESTAMP_COL].min()
            end = start + pd.Timedelta(days=zoom_days)
            sub = sub[(sub[TIMESTAMP_COL] >= start) & (sub[TIMESTAMP_COL] < end)].copy()
        else:
            sub = downsample_time_df(sub, TIMESTAMP_COL, MAX_POINTS_LINE_PLOT)

        if sub.empty:
            continue

        model_name = sub["champion_model"].iloc[0]

        fig, ax = plt.subplots(figsize=(16, 6))

        if "PI95_lower" in sub.columns and "PI95_upper" in sub.columns:
            ax.fill_between(
                sub[TIMESTAMP_COL],
                sub["PI95_lower"],
                sub["PI95_upper"],
                alpha=0.25,
                color=PALETTE["pi95"],
                label="95% prediction interval",
            )

        if "PI80_lower" in sub.columns and "PI80_upper" in sub.columns:
            ax.fill_between(
                sub[TIMESTAMP_COL],
                sub["PI80_lower"],
                sub["PI80_upper"],
                alpha=0.45,
                color=PALETTE["pi80"],
                label="80% prediction interval",
            )

        ax.plot(
            sub[TIMESTAMP_COL],
            sub["y_true"],
            linewidth=0.9,
            color=PALETTE["actual"],
            label="Actual",
        )

        ax.plot(
            sub[TIMESTAMP_COL],
            sub["y_pred"],
            linewidth=0.9,
            color=PALETTE["pred"],
            label=f"Predicted — {model_name}",
        )

        if zoom_days is None:
            title_extra = "full test period"
            filename_extra = "full"
        else:
            title_extra = f"first {zoom_days} test days"
            filename_extra = f"zoom_{zoom_days}_days"

        ax.set_title(
            f"{dataset_name} | Champion forecast with intervals — "
            f"{horizon_label(horizon)} — {title_extra}",
            fontweight="bold",
        )
        ax.set_xlabel("Time")
        ax.set_ylabel("Load")
        ax.legend(loc="upper right", fontsize=9)

        save_fig(
            plots_dir
            / f"prediction_intervals_{dataset_name}_{safe_filename(horizon)}_{filename_extra}.png"
        )


def plot_interval_coverage(
    pi_metrics: pd.DataFrame,
    dataset_name: str,
    plots_dir: Path,
) -> None:
    """Plot interval coverage and width."""
    if pi_metrics.empty:
        return

    df = add_horizon_info(pi_metrics.copy())
    df = df.sort_values(["horizon_order", "interval"])

    fig, ax = plt.subplots(figsize=(13, 6))

    sns.barplot(
        data=df,
        x="horizon_label",
        y="coverage_eval_percent",
        hue="interval",
        ax=ax,
    )

    ax.axhline(80, color=PALETTE["warn"], linestyle="--", linewidth=1, label="80% target")
    ax.axhline(95, color=PALETTE["bad"], linestyle="--", linewidth=1, label="95% target")

    ax.set_title(
        f"{dataset_name} | Prediction interval coverage on evaluation split",
        fontweight="bold",
    )
    ax.set_xlabel("Forecast horizon")
    ax.set_ylabel("Coverage (%)")
    ax.legend()

    save_fig(plots_dir / f"prediction_interval_coverage_{dataset_name}.png")

    fig, ax = plt.subplots(figsize=(13, 6))

    sns.barplot(
        data=df,
        x="horizon_label",
        y="avg_width_eval",
        hue="interval",
        ax=ax,
    )

    ax.set_title(f"{dataset_name} | Prediction interval average width", fontweight="bold")
    ax.set_xlabel("Forecast horizon")
    ax.set_ylabel("Average interval width")
    ax.legend()

    save_fig(plots_dir / f"prediction_interval_width_{dataset_name}.png")


def plot_error_diagnostics(
    diagnostics: pd.DataFrame,
    dataset_name: str,
    plots_dir: Path,
) -> None:
    """Plot error diagnostics by segment."""
    if diagnostics.empty:
        return

    df = add_horizon_info(diagnostics.copy())

    diagnostic_order = [
        "hour",
        "day_of_week",
        "month",
        "weekday_weekend",
        "load_quantile",
    ]

    for diag_type in diagnostic_order:
        sub_type = df[df["diagnostic_type"] == diag_type].copy()

        if sub_type.empty:
            continue

        for horizon, sub in sub_type.groupby("horizon"):
            sub = sub.copy()

            if diag_type == "hour":
                sub["sort_key"] = pd.to_numeric(sub["diagnostic_value"], errors="coerce")
            elif diag_type == "load_quantile":
                sub["sort_key"] = sub["diagnostic_value"].str.extract(r"Q(\d+)")[0].astype(float)
            else:
                sub["sort_key"] = range(len(sub))

            sub = sub.sort_values("sort_key")

            fig, ax = plt.subplots(figsize=(14, 6))

            sns.barplot(
                data=sub,
                x="diagnostic_value",
                y="MAE",
                ax=ax,
                color=PALETTE["bar"],
            )

            model_name = sub["model"].iloc[0]

            ax.set_title(
                f"{dataset_name} | MAE by {diag_type} — {model_name} — "
                f"{horizon_label(horizon)}",
                fontweight="bold",
            )
            ax.set_xlabel(diag_type)
            ax.set_ylabel("MAE")
            plt.xticks(rotation=30, ha="right")

            save_fig(
                plots_dir
                / f"diagnostic_mae_{diag_type}_{dataset_name}_{safe_filename(horizon)}.png"
            )


def plot_daily_worst_best(
    worst_days: pd.DataFrame,
    best_days: pd.DataFrame,
    dataset_name: str,
    plots_dir: Path,
) -> None:
    """Save worst and best days as table images."""
    if not worst_days.empty:
        for horizon, sub in worst_days.groupby("horizon"):
            cols = [
                "horizon_label",
                "champion_model",
                "date",
                "n",
                "MAE",
                "RMSE",
                "mean_actual",
                "mean_predicted",
                "max_abs_error",
            ]
            cols = [c for c in cols if c in sub.columns]

            save_table_as_image(
                sub[cols],
                title=f"{dataset_name} | Worst prediction days — {horizon_label(horizon)}",
                path=plots_dir / f"worst_days_{dataset_name}_{safe_filename(horizon)}.png",
                max_rows=10,
                float_cols=["MAE", "RMSE", "mean_actual", "mean_predicted", "max_abs_error"],
            )

    if not best_days.empty:
        for horizon, sub in best_days.groupby("horizon"):
            cols = [
                "horizon_label",
                "champion_model",
                "date",
                "n",
                "MAE",
                "RMSE",
                "mean_actual",
                "mean_predicted",
                "max_abs_error",
            ]
            cols = [c for c in cols if c in sub.columns]

            save_table_as_image(
                sub[cols],
                title=f"{dataset_name} | Best prediction days — {horizon_label(horizon)}",
                path=plots_dir / f"best_days_{dataset_name}_{safe_filename(horizon)}.png",
                max_rows=10,
                float_cols=["MAE", "RMSE", "mean_actual", "mean_predicted", "max_abs_error"],
            )


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 9 — Reports
# ═════════════════════════════════════════════════════════════════════════════

def build_interpretation_text(
    dataset_name: str,
    champion_table: pd.DataFrame,
    pi_metrics: pd.DataFrame,
) -> str:
    """Build automatic interpretation text."""
    lines = []

    lines.append(f"# STEP 6 Interpretation — {dataset_name}")
    lines.append("")
    lines.append("## Champion model selection")
    lines.append("")

    if champion_table.empty:
        lines.append("No champion table was available.")
    else:
        for _, row in champion_table.iterrows():
            if row["champion_type"] == "Ensemble":
                meaning = (
                    "The ensemble was selected because it improved RMSE enough "
                    "to justify using a combined model."
                )
            else:
                meaning = (
                    "The best single model was selected because the ensemble did "
                    "not improve RMSE enough to justify additional complexity."
                )

            best_ensemble_rmse = row["best_ensemble_RMSE"]
            if pd.isna(best_ensemble_rmse):
                best_ensemble_text = "N/A"
            else:
                best_ensemble_text = f"{best_ensemble_rmse:.4f}"

            lines.append(
                f"- **{row['horizon_label']}**: champion = **{row['champion_model']}** "
                f"({row['champion_type']}). Best single RMSE = "
                f"**{row['best_single_RMSE']:.4f}**, best ensemble RMSE = "
                f"**{best_ensemble_text}**. {meaning}"
            )

    lines.append("")
    lines.append("## Prediction intervals")
    lines.append("")

    if pi_metrics.empty:
        lines.append("Prediction interval metrics were not available.")
    else:
        for _, row in pi_metrics.iterrows():
            lines.append(
                f"- **{row['horizon_label']} — {row['interval']}**: "
                f"evaluation coverage = **{row['coverage_eval_percent']:.2f}%**, "
                f"average interval width = **{row['avg_width_eval']:.4f}**."
            )

    lines.append("")
    lines.append("## Final recommendation")
    lines.append("")
    lines.append(
        "Use the Step 6 champion table as the final model selection table. "
        "Use prediction intervals to discuss uncertainty. Use diagnostics to explain "
        "where the model performs well or poorly."
    )

    return "\n".join(lines)


def create_markdown_report(
    dataset_name: str,
    champion_table: pd.DataFrame,
    comparison_metrics: pd.DataFrame,
    ensemble_weights: pd.DataFrame,
    pi_metrics: pd.DataFrame,
    diagnostics: pd.DataFrame,
    worst_days: pd.DataFrame,
    best_days: pd.DataFrame,
    reports_dir: Path,
) -> Path:
    """Create Markdown report."""
    ensure_dir(reports_dir)

    report_path = reports_dir / f"STEP6_RELIABILITY_REPORT_{dataset_name}.md"

    lines = []

    lines.append(f"# STEP 6 Reliability Report — {dataset_name}")
    lines.append("")

    lines.append("## 1. Champion model selection")
    lines.append("")
    lines.append(dataframe_to_markdown_safe(champion_table, max_rows=20))
    lines.append("")

    lines.append("## 2. Single models vs ensembles")
    lines.append("")
    lines.append(dataframe_to_markdown_safe(comparison_metrics, max_rows=120))
    lines.append("")

    lines.append("## 3. Ensemble weights")
    lines.append("")
    lines.append(dataframe_to_markdown_safe(ensemble_weights, max_rows=120))
    lines.append("")

    lines.append("## 4. Prediction interval metrics")
    lines.append("")
    lines.append(dataframe_to_markdown_safe(pi_metrics, max_rows=80))
    lines.append("")

    lines.append("## 5. Error diagnostics")
    lines.append("")
    lines.append(dataframe_to_markdown_safe(diagnostics, max_rows=120))
    lines.append("")

    lines.append("## 6. Worst prediction days")
    lines.append("")
    lines.append(dataframe_to_markdown_safe(worst_days, max_rows=80))
    lines.append("")

    lines.append("## 7. Best prediction days")
    lines.append("")
    lines.append(dataframe_to_markdown_safe(best_days, max_rows=80))
    lines.append("")

    lines.append(build_interpretation_text(dataset_name, champion_table, pi_metrics))

    report_path.write_text("\n".join(lines), encoding="utf-8")

    return report_path


def create_html_report(
    dataset_name: str,
    champion_table: pd.DataFrame,
    comparison_metrics: pd.DataFrame,
    ensemble_weights: pd.DataFrame,
    pi_metrics: pd.DataFrame,
    diagnostics: pd.DataFrame,
    worst_days: pd.DataFrame,
    best_days: pd.DataFrame,
    plots_dir: Path,
    reports_dir: Path,
    dataset_output_dir: Path,
) -> Path:
    """Create HTML report."""
    ensure_dir(reports_dir)

    report_path = reports_dir / f"STEP6_RELIABILITY_REPORT_{dataset_name}.html"

    def table_html(df: pd.DataFrame, max_rows: int = 120) -> str:
        if df.empty:
            return "<p><em>No data available.</em></p>"
        return df.head(max_rows).to_html(index=False, border=0, classes="data-table")

    plot_files = sorted(plots_dir.glob("*.png"))
    plot_html_parts = []

    for p in plot_files:
        rel = p.relative_to(dataset_output_dir).as_posix()
        plot_html_parts.append(
            f"""
            <div class="plot-block">
              <h3>{html.escape(p.stem)}</h3>
              <img src="{html.escape(rel)}" alt="{html.escape(p.stem)}">
            </div>
            """
        )

    interpretation = build_interpretation_text(dataset_name, champion_table, pi_metrics)

    html_text = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>STEP 6 Reliability Report — {html.escape(dataset_name)}</title>
<style>
body {{
    font-family: Arial, sans-serif;
    margin: 30px;
    background: #fafafa;
    color: #222;
}}
h1, h2, h3 {{
    color: #1f3a5f;
}}
.data-table {{
    border-collapse: collapse;
    font-size: 13px;
    margin-bottom: 28px;
    width: 100%;
}}
.data-table th {{
    background: #1f3a5f;
    color: white;
    padding: 8px;
    border: 1px solid #ddd;
}}
.data-table td {{
    padding: 7px;
    border: 1px solid #ddd;
}}
.data-table tr:nth-child(even) {{
    background: #f2f2f2;
}}
.plot-block {{
    margin: 28px 0;
    padding: 18px;
    background: white;
    border: 1px solid #ddd;
    border-radius: 8px;
}}
.plot-block img {{
    max-width: 100%;
    height: auto;
}}
.interpretation {{
    white-space: pre-wrap;
    background: #ffffff;
    border: 1px solid #ddd;
    padding: 18px;
    border-radius: 8px;
    line-height: 1.5;
}}
</style>
</head>
<body>

<h1>STEP 6 Reliability Report — {html.escape(dataset_name)}</h1>

<h2>1. Champion model selection</h2>
{table_html(champion_table)}

<h2>2. Single models vs ensembles</h2>
{table_html(comparison_metrics)}

<h2>3. Ensemble weights</h2>
{table_html(ensemble_weights)}

<h2>4. Prediction interval metrics</h2>
{table_html(pi_metrics)}

<h2>5. Error diagnostics</h2>
{table_html(diagnostics)}

<h2>6. Worst prediction days</h2>
{table_html(worst_days)}

<h2>7. Best prediction days</h2>
{table_html(best_days)}

<h2>8. Automatic interpretation</h2>
<pre class="interpretation">{html.escape(interpretation)}</pre>

<h2>9. Figures</h2>
{''.join(plot_html_parts)}

</body>
</html>
"""

    report_path.write_text(html_text, encoding="utf-8")

    return report_path


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 10 — Save outputs
# ═════════════════════════════════════════════════════════════════════════════

def save_outputs_for_dataset(
    dataset_name: str,
    final_metrics: pd.DataFrame,
    ensemble_metrics: pd.DataFrame,
    comparison_metrics: pd.DataFrame,
    ensemble_predictions: pd.DataFrame,
    ensemble_weights: pd.DataFrame,
    champion_table: pd.DataFrame,
    champion_predictions: pd.DataFrame,
    champion_predictions_pi: pd.DataFrame,
    pi_metrics: pd.DataFrame,
    diagnostics: pd.DataFrame,
    worst_days: pd.DataFrame,
    best_days: pd.DataFrame,
    tables_dir: Path,
    predictions_dir: Path,
) -> None:
    """Save all tables and predictions."""
    ensure_dir(tables_dir)
    ensure_dir(predictions_dir)

    table_map = {
        "01_original_final_metrics": final_metrics,
        "02_ensemble_metrics": ensemble_metrics,
        "03_single_vs_ensemble_comparison": comparison_metrics,
        "04_ensemble_weights": ensemble_weights,
        "05_champion_selection": champion_table,
        "06_prediction_interval_metrics": pi_metrics,
        "07_error_diagnostics": diagnostics,
        "08_worst_prediction_days": worst_days,
        "09_best_prediction_days": best_days,
    }

    prediction_map = {
        "01_ensemble_predictions": ensemble_predictions,
        "02_champion_predictions": champion_predictions,
        "03_champion_predictions_with_intervals": champion_predictions_pi,
    }

    for name, df in table_map.items():
        if not df.empty:
            df.to_csv(tables_dir / f"{name}_{dataset_name}.csv", index=False)

    for name, df in prediction_map.items():
        if not df.empty:
            df.to_csv(predictions_dir / f"{name}_{dataset_name}.csv", index=False)

    excel_path = tables_dir / f"STEP6_RELIABILITY_SUMMARY_{dataset_name}.xlsx"

    try:
        with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
            for name, df in table_map.items():
                if df.empty:
                    continue

                sheet_name = name[:31]
                df.to_excel(writer, sheet_name=sheet_name, index=False)

        print(f"    Excel summary saved → {excel_path}")

    except Exception as exc:
        print(f"    [WARN] Excel file was not created: {exc}")
        print("    CSV files were still saved.")


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 11 — Main pipeline per dataset
# ═════════════════════════════════════════════════════════════════════════════

def run_step6_for_dataset(dataset_name: str) -> None:
    """Run Step 6 for one dataset."""
    print("\n" + "=" * 90)
    print(f"  STEP 6 — Reliability analysis for {dataset_name}")
    print("=" * 90)

    dataset_output_dir = STEP6_OUTPUT_DIR / dataset_name
    tables_dir = dataset_output_dir / "tables"
    plots_dir = dataset_output_dir / "plots"
    predictions_dir = dataset_output_dir / "predictions"
    reports_dir = dataset_output_dir / "reports"

    for p in [dataset_output_dir, tables_dir, plots_dir, predictions_dir, reports_dir]:
        ensure_dir(p)

    print("\n[1/7] Loading STEP 4 final metrics and all-model predictions ...")

    final_metrics = load_step4_final_metrics(dataset_name)
    all_predictions = load_all_predictions_for_dataset(dataset_name)

    print(f"    Final metrics rows       : {len(final_metrics):,}")
    print(f"    All-model prediction rows: {len(all_predictions):,}")

    if final_metrics.empty:
        print(f"    [SKIP] No final metrics found for {dataset_name}.")
        return

    if all_predictions.empty:
        print(f"    [SKIP] No all-model predictions found for {dataset_name}.")
        print("    Make sure STEP 4 LITE saved final_all_predictions_h*.csv files.")
        return

    print("\n[2/7] Building ensemble predictions ...")

    ensemble_metrics, ensemble_predictions, ensemble_weights = build_all_ensembles(
        dataset_name=dataset_name,
        final_metrics=final_metrics,
        all_predictions=all_predictions,
    )

    print(f"    Ensemble metrics rows    : {len(ensemble_metrics):,}")
    print(f"    Ensemble prediction rows : {len(ensemble_predictions):,}")
    print(f"    Ensemble weight rows     : {len(ensemble_weights):,}")

    print("\n[3/7] Selecting champion models ...")

    comparison_metrics = pd.concat(
        [final_metrics, ensemble_metrics],
        ignore_index=True,
    )

    comparison_metrics = add_horizon_info(comparison_metrics)
    comparison_metrics = comparison_metrics.sort_values(
        ["horizon_order", "RMSE"]
    ).reset_index(drop=True)

    champion_table = build_champion_table(
        dataset_name=dataset_name,
        final_metrics=final_metrics,
        ensemble_metrics=ensemble_metrics,
    )

    champion_predictions = build_all_champion_predictions(
        dataset_name=dataset_name,
        champion_table=champion_table,
        all_predictions=all_predictions,
        ensemble_predictions=ensemble_predictions,
    )

    print(f"    Champion rows            : {len(champion_table):,}")
    print(f"    Champion prediction rows : {len(champion_predictions):,}")

    print("\n    Champion models:")
    if not champion_table.empty:
        show_cols = [
            "horizon_label",
            "champion_model",
            "champion_type",
            "best_single_RMSE",
            "best_ensemble_RMSE",
            "RMSE_improvement_of_ensemble_percent",
        ]
        print(champion_table[show_cols].to_string(index=False))

    print("\n[4/7] Building prediction intervals ...")

    champion_predictions_pi, pi_metrics = add_prediction_intervals(champion_predictions)

    print(f"    Prediction interval rows : {len(pi_metrics):,}")

    print("\n[5/7] Building error diagnostics ...")

    diagnostics = build_error_diagnostics(champion_predictions_pi)
    worst_days, best_days = build_daily_error_tables(champion_predictions_pi)

    print(f"    Diagnostic rows          : {len(diagnostics):,}")
    print(f"    Worst-day rows           : {len(worst_days):,}")
    print(f"    Best-day rows            : {len(best_days):,}")

    print("\n[6/7] Saving tables, predictions, and plots ...")

    save_outputs_for_dataset(
        dataset_name=dataset_name,
        final_metrics=final_metrics,
        ensemble_metrics=ensemble_metrics,
        comparison_metrics=comparison_metrics,
        ensemble_predictions=ensemble_predictions,
        ensemble_weights=ensemble_weights,
        champion_table=champion_table,
        champion_predictions=champion_predictions,
        champion_predictions_pi=champion_predictions_pi,
        pi_metrics=pi_metrics,
        diagnostics=diagnostics,
        worst_days=worst_days,
        best_days=best_days,
        tables_dir=tables_dir,
        predictions_dir=predictions_dir,
    )

    save_table_as_image(
        champion_table,
        title=f"{dataset_name} — Champion Model Selection",
        path=plots_dir / f"table_champion_selection_{dataset_name}.png",
        max_rows=10,
        float_cols=[
            "best_single_RMSE",
            "best_ensemble_RMSE",
            "RMSE_improvement_of_ensemble_percent",
        ],
    )

    save_table_as_image(
        pi_metrics,
        title=f"{dataset_name} — Prediction Interval Metrics",
        path=plots_dir / f"table_prediction_intervals_{dataset_name}.png",
        max_rows=20,
        float_cols=[
            "coverage_eval_percent",
            "avg_width_eval",
            "coverage_all_percent",
            "avg_width_all",
        ],
    )

    plot_single_vs_ensemble_metrics(comparison_metrics, dataset_name, plots_dir)
    plot_champion_summary(champion_table, dataset_name, plots_dir)

    plot_prediction_with_intervals(
        champion_predictions_pi,
        dataset_name,
        plots_dir,
        zoom_days=None,
    )

    plot_prediction_with_intervals(
        champion_predictions_pi,
        dataset_name,
        plots_dir,
        zoom_days=14,
    )

    plot_interval_coverage(pi_metrics, dataset_name, plots_dir)
    plot_error_diagnostics(diagnostics, dataset_name, plots_dir)
    plot_daily_worst_best(worst_days, best_days, dataset_name, plots_dir)

    print("    Plots generated.")

    print("\n[7/7] Creating Markdown and HTML reports ...")

    md_path = create_markdown_report(
        dataset_name=dataset_name,
        champion_table=champion_table,
        comparison_metrics=comparison_metrics,
        ensemble_weights=ensemble_weights,
        pi_metrics=pi_metrics,
        diagnostics=diagnostics,
        worst_days=worst_days,
        best_days=best_days,
        reports_dir=reports_dir,
    )

    html_path = create_html_report(
        dataset_name=dataset_name,
        champion_table=champion_table,
        comparison_metrics=comparison_metrics,
        ensemble_weights=ensemble_weights,
        pi_metrics=pi_metrics,
        diagnostics=diagnostics,
        worst_days=worst_days,
        best_days=best_days,
        plots_dir=plots_dir,
        reports_dir=reports_dir,
        dataset_output_dir=dataset_output_dir,
    )

    print(f"    Markdown report saved → {md_path}")
    print(f"    HTML report saved     → {html_path}")

    print(f"\n  Outputs saved → {dataset_output_dir}")


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 12 — Entry point
# ═════════════════════════════════════════════════════════════════════════════

def main() -> None:
    """Run Step 6 for all datasets."""
    ensure_dir(STEP6_OUTPUT_DIR)

    print("\n" + "=" * 90)
    print("  STEP 6 — Ensemble Forecasting, Prediction Intervals, and Reliability Analysis")
    print("=" * 90)

    print(f"\nInput STEP 4 folder : {STEP4_OUTPUT_DIR}")
    print(f"Output STEP 6 folder: {STEP6_OUTPUT_DIR}")

    for dataset_name in DATASETS:
        run_step6_for_dataset(dataset_name)

    print("\n" + "=" * 90)
    print("  STEP 6 complete for all datasets.")
    print("=" * 90)

    print("\nFinal output folders:")

    for dataset_name in DATASETS:
        print(f"  - {STEP6_OUTPUT_DIR / dataset_name}")


if __name__ == "__main__":
    main()